# PKG Contagion Phase 1 — inflow shock event study (v3.4)

**PNC Treasury Management · DSI · Payment Knowledge Graph (PKG)**

What happens to a TM organisation after its regular incoming payments **fall** (a payer stops,
skips or pays much less) or **rise** (an existing payer pays much more, or a new regular payer
starts)? Each event is compared with matched customers that had no event, 6 months before to
6 months after, on payment behaviour, deposits, revenue and credit.

**Changes from v3.3** (2026-11-15)

- **Own accounts at other banks are not payers.** Streams from the customer's own name at another bank (pair table
  `o_self`) are left out of event triggers, like related companies; their breaks are counted as "own money moving".
- **§6 second hop by event group**, to see why the rate fell when PNC entities became hub payers.
- **New §7c, exposure score and watchlist (prototype):** for every operating TM organisation, the share of its
  income from payers that are under stress now, its own warning signals, how much it would pass on (from §5e), and
  what is at stake (deposits, payments to other PNC customers, DHE). Backtested on past months before it is used.
- Report wording fixes (PD decimals, timing in plain words, trait labels per direction) and a score section.

**Changes from v3.2** (2026-11-15)

- **PNC's own entities are hub payers** (e.g. PNC BANK NA ACQUIRING BIN GLOBAL, merchant settlement): their events
  leave the headline like other processors.
- **New §5e, what makes a shock spread:** one regression per direction and outcome instead of more event types.
  Each customer's response at +3 (minus its matched controls) on the event's size, and how that response changes
  with the event's traits (main payer's share of income, payers gone vs paying less, regular share of income,
  number of payers, balance buffer, self-payment, payer-wide, PNC-customer payer). Intervals clustered by payer.
- **New §5f, self-payment:** money moving between a customer's own accounts at PNC and at other banks. How common,
  how it reacts to a shock, and whether customers with it absorb differently.
- **New §9, HTML report:** a standalone report built from this run's numbers and figures, for review and sharing.

**Changes from v3.1** (2026-11-14)

- **Outflow split in two:** dollars (and payments) to regular payees (paid in at least 4 of the 6 months before)
  and to everyone else. Shows whether customers cut one-off spending first and regular obligations later.
  One extra pass over the pair table, outgoing side (about as long as the incoming stream step).
- **Events are not independent.** Each event gets its main payer (largest lost or gained stream). §5d repeats
  the key effects with every payer-month counting once (weight 1 / events sharing that payer and month) and
  intervals clustered by payer, shows how concentrated events are on a few payers, the effect by event month
  and the range when each month is left out.
- **A second event definition on all inflow:** total incoming dollars at least 30% below (or above) their own
  6-month average over months e and e+1, with month e-1 still near normal. Compared with the regular-payer
  events, and split into those with and without a regular-payer event nearby.

**Changes from v3** (after the third real run, 2026-11-14)

- **Related payers left out.** Internal streams between companies of one master, or whose names share a
  distinctive word (Marathon Oil / MOC Portfolio wind-downs), are not standing relationships. Only the steps
  after the stream windows rerun (the 53-minute stream step is reused).
- **§5b by stratum:** the onset reading also for payer-wide stops from non-hub payers (outside the customer's
  control) and for single-link stops, to tell an early warning from a customer anticipating its own change.
- **New §5c, who absorbs and who spreads:** effects and pass-through by sector (NAICS2) and by baseline traits
  (payers, payees, concentration, external share, balance buffer, family size).
- **§7b oil shock against the same months a year earlier**, so seasonality drops out.

**Changes from v2** (after the second real run, 2026-11-14)

- **Steady streams only.** A stream whose last 3 months average more than 1.5x its 6-month median was surging,
  and its "loss" was the surge ending (the +28.6% inflow pre-trend in v2). Such streams are no longer standing.
- **Payer tenure:** a standing stream must also be paid in at least 8 of the last 12 months.
- **Family moves cancel.** A payer that stops paying one company and starts or grows paying a sister company
  (same master, months e-1..e+1) is not a loss for either (Dillon / Kroger, Marathon in v2).
- **Receivables and funding vehicles** (names with RECEIVABLE, SECURITIZATION, FUNDING LLC ...) move to the
  finance & public segment (Wesco Receivables in v2).
- **New §5b, what moves first:** for every outcome, the first month its effect is clear of zero, the month it
  reaches half its peak, and the peak.
- **New §6b, early warning (look-back):** real events (default, PD doubling, deposit collapse, a PNC-customer
  payer cutting many payees) and the signals that switched on in the 12 months before them, against matched
  controls: which fire more, which fire first, and in which order.
- **New §7b, real scenarios:** Change Healthcare (2024-02, descriptive) and the 2026 oil shock (2026-03,
  transport NAICS 48-49 and the customers that depend on them for income).

**Changes from v1** (after the first real run, 2026-11-14)

- **Timing shifts cancel.** Lost dollars are counted over months e-1, e, e+1, so a payment that only moved
  across a month end is no longer a shock. A break needs a net shortfall of at least 50% of the expected amount.
- **Events start 2025-01**, when the 6 shock-free months before an event can be checked.
- **Headline = operating businesses, non-hub payers.** PNC's own entities, NAICS 52 and 92 and hub
  (processor) payers become separate cuts.
- **Panel sampled by stratum** (headline / other payer-wide / single-link, per direction), so cuts and dose
  response compare like with like.
- **Dollars and counts as average log change** (shown as %), instead of the median index that sat at 100 for counts.
- **Both directions.** Losses and gains are netted against each other in the same month, so a payer that
  changes its bank (old stream stops, new one starts) cancels out.
- **Deposits, revenue and credit** from `pkg_dep_cust_month` and `pkg_tm_cust_month`.
- **Pass-through:** % change in outflow per 1% change in inflow, by direction, size band and segment. This is
  the transmission rate a network propagation model needs.

| § | What it does | Output |
|---|---|---|
| 0 | Inputs: required columns, months, outcome families | one table |
| 1 | Population: TM organisations, standing streams, baseline inflow, segment, size band | one table |
| 2 | Stream breaks and gains, timing shifts, customer events | one table |
| 3 | Pre-flight counts, gates, headline choice | tables + f01, f02 |
| 4 | Matched controls and the event panel | one table |
| 5 | Effects, dose response, cuts, pass-through, pre-trend check | tables + f03-f11, f15, f17 |
| 5b | What moves first after the shock (headline, payer-wide, single-link) | tables + f18 |
| 5c | Who absorbs and who spreads: sector and traits | tables + f22 |
| 5d | Robustness: payer weighting and clustering, event month, all-inflow events | tables + f23 |
| 5e | What makes a shock spread: response by event and customer traits (regression) | tables + f24 |
| 5f | Self-payment: own accounts at other banks | tables |
| 9 | HTML report | p1_report.html |
| 7c | Exposure score and watchlist (prototype), with backtest | tables + p1_score_*.parquet, p1_watchlist_*.csv |
| 6 | Second hop: do the customer's own regular payees feel it? | table + f14 |
| 6b | Early warning: signals before real events | tables + f12, f13 |
| 7 | Case panels (incl. United Consumer Financial Service) | table + f16 |
| 7b | Real scenarios: Change Healthcare, 2026 oil shock | tables + f20, f21 |
| 8 | Answer sheet, gates, files written | one table |

Tables are shown inline (pandas, one output per cell). **Figures are saved to disk, never shown inline.**
Nothing is written to cdi_tm. Work files go to the personal HDFS folder, results to `OUT_DIR`.

In [ ]:
# =============================================================================
# SETUP — environment shims, CONFIG, Spark session, helpers
# =============================================================================
import os, sys, time, json, math, hashlib, warnings, threading, contextlib
import numpy as np
import pandas as pd

# --- environment compat (must run before any toPandas / createDataFrame) ------
for _a, _t in [("object0", object), ("bool8", np.bool_), ("int0", np.intp),
               ("uint0", np.uintp), ("bool", np.bool_)]:
    if not hasattr(np, _a):
        setattr(np, _a, _t)
if not hasattr(pd.DataFrame, "iteritems"):
    pd.DataFrame.iteritems = pd.DataFrame.items
if not hasattr(pd.Series, "iteritems"):
    pd.Series.iteritems = pd.Series.items
for _w in (ResourceWarning, DeprecationWarning, FutureWarning):
    warnings.filterwarnings("ignore", category=_w)

import matplotlib
matplotlib.use("Agg")                       # figures go to disk, never inline
import matplotlib.pyplot as plt
import matplotlib.ticker

from pyspark.sql import SparkSession, functions as F, Window as W

DB = "bdahd01p_dlcdi1_cdi_tm"

CONFIG = dict(
    # --- inputs ---------------------------------------------------------------
    MET_TABLE  = f"{DB}.pkg_cust_metrics_monthly_v3",
    PAIR_TABLE = f"{DB}.pkg_cust_pair_month_v3",
    CPTY_TABLE = f"{DB}.pkg_cpty_metrics_monthly",     # ent_id, month, hub_tier
    DIM_TABLE  = f"{DB}.pkg_cust_dim",                 # mdm_id, customer_name, master_pwr_id (related payers)
    USE_HUB    = True,

    # --- outcome families outside the customer metrics (names from the §0 run of 2026-11-14)
    # kind: logpct = average log change, shown as %; diff = change from own baseline;
    #       scaled = change from own baseline / baseline monthly inflow, in pp.
    # subset_pos: measure only for customers whose baseline value of that output is above zero.
    # mask: drop a customer's window when `col` > 0 in one of `months` (acquisition months).
    EXTRA_OUTCOMES = [
        dict(name="deposits", table=f"{DB}.pkg_dep_cust_month", id_col="customer_pwr_id",
             id_kind="customer_pwr_id", month_col="month",
             mask=dict(col="n_acct_new", months=["2025-12", "2026-06"]),
             measures={
                 "dep_bal_mean":      dict(src="dep_bal_mean", kind="logpct", label="Average balance"),
                 "dep_bal_vs_inflow": dict(src="dep_bal_mean", kind="scaled",
                                           label="Average balance change (pp of monthly inflow)"),
                 "dep_bal_min":       dict(src="dep_bal_min", kind="scaled",
                                           label="Lowest balance change (pp of monthly inflow)"),
                 "dep_drawdown_pct":  dict(src="dep_drawdown_pct", kind="diff", label="Largest drawdown in month (change)"),
                 "days_negative":     dict(src="days_negative", kind="diff", label="Days with negative balance"),
                 "n_acct_closed":     dict(src="n_acct_closed", kind="diff", label="Accounts closed per month"),
             }),
        dict(name="revenue", table=f"{DB}.pkg_tm_cust_month", id_col="customer_pwr_id",
             id_kind="customer_pwr_id", month_col="month",
             measures={
                 "gross_rev_12m":      dict(src="gross_rev_12m", kind="logpct", label="Gross revenue, trailing 12m"),
                 "tm_rev_12m":         dict(src="tm_rev_12m", kind="logpct", label="TM revenue, trailing 12m"),
                 "credit_rev_12m":     dict(src="credit_rev_12m", kind="logpct", label="Credit revenue, trailing 12m"),
                 "non_credit_rev_12m": dict(src="non_credit_rev_12m", kind="logpct", label="Non-credit revenue, trailing 12m"),
             }),
        dict(name="credit", table=f"{DB}.pkg_tm_cust_month", id_col="customer_pwr_id",
             id_kind="customer_pwr_id", month_col="month", subset_pos="dhe",
             measures={
                 "dhe":           dict(src="dhe", kind="logpct", label="Direct hard exposure (DHE)"),
                 "pd_pct_now":    dict(src="pd_pct_now", kind="diff", label="PD (pd_pct_now, change)"),
                 "is_default":    dict(src="is_default", kind="diff", mult=100.0, label="In default (pp)"),
                 "past_due_days": dict(src="past_due_days", kind="diff", label="Days past due"),
             }),
    ],

    # --- window ---------------------------------------------------------------
    MONTH_MIN = "2024-01",
    MONTH_MAX = None,            # None = last month present in both inputs

    # --- run control ----------------------------------------------------------
    RUN_STAGE = "full",          # "preflight" stops after §3
    REBUILD   = False,           # True = ignore the work files of this configuration
    WORK_DIR  = "hdfs:///user/pk36814/pkg_contagion_p1/work",
    OUT_DIR   = "../metrics/contagion/p1",
    HDFS_REPLICATION   = 2,
    SHUFFLE_PARTITIONS = 800,

    # --- population -----------------------------------------------------------
    ORG_PARTY_TYPES = ["O"],
    BASE_MONTHS     = 6,         # baseline = the 6 months before the event month
    BASE_IN_MIN     = 10_000.0,  # minimum average monthly incoming dollars at baseline
    SIZE_BANDS      = [100_000.0, 1_000_000.0],   # baseline monthly inflow: small / mid / large
    FIN_PUBLIC_NAICS2 = ["52", "92"],
    PNC_NAME_REGEX  = r"\bPNC\b",
    SPV_NAME_REGEX  = r"RECEIVABLE|SECURITI[SZ]ATION|\bFUNDING (LLC|CORP|CO|INC|LP|LTD)\b",   # -> finance & public

    # --- stream events (monthly grain) ----------------------------------------
    STANDING_MIN = 4,            # paid in >= 4 of the last 6 months, incl. the last one
    STEADY_MAX   = 1.5,          # ... and steady: last-3-month average <= 1.5x the 6-month median (no surge)
    TENURE_MIN   = 8,            # ... and paid in >= 8 of the last 12 months (None = off)
    FAMILY_NET   = True,         # a payer moving money between companies of one master cancels out
    SELF_EXCLUDE    = True,      # payers that are the customer's own name at another bank are not event triggers
    PAIR_SELF_COL   = "o_self",  # pair-table flag for the customer's own name at another bank
    PNC_PAYER_HUB   = True,      # internal payers whose name matches PNC_NAME_REGEX count as hub payers
    RELATED_EXCLUDE = True,      # internal streams within one master or with a shared distinctive name word are left out
    NAME_TOKEN_MAX  = 50,        # a name word counts as distinctive when at most 50 customers start with it
    NAME_STOP = ["THE", "COMPANY", "COMPANIES", "CORPORATION", "CORP", "INCORPORATED", "LLC", "GROUP", "HOLDINGS",
                 "HOLDING", "SERVICES", "SERVICE", "INTERNATIONAL", "NATIONAL", "AMERICAN", "AMERICA", "UNITED",
                 "FIRST", "GENERAL", "GLOBAL", "CITY", "COUNTY", "STATE", "UNIVERSITY", "BANK", "TRUST", "FUND",
                 "CAPITAL", "PARTNERS", "MANAGEMENT", "ENTERPRISES", "INDUSTRIES", "SYSTEMS", "SOLUTIONS",
                 "NORTH", "SOUTH", "EAST", "WEST", "CENTRAL", "HEALTH", "MEDICAL", "COMMUNITY", "PROPERTIES",
                 "REALTY", "INVESTMENTS", "ASSOCIATES", "ASSOCIATION", "DEPARTMENT", "OFFICE", "SCHOOL", "CENTER"],
    LOSS_MIN     = 0.50,         # net shortfall over e-1..e+1 >= 50% of the expected amount = break
    GAIN_MIN     = 0.50,         # paid >= 150% of expected in both e and e+1 = growth
    NEW_MONTHS   = 3,            # new payer: paid in e, e+1, e+2 and not in the 6 months before

    # --- customer events ------------------------------------------------------
    TIER_EDGES    = [0.10, 0.20, 0.50],   # net event size / baseline inflow; first edge = minimum kept
    HEADLINE_MIN  = 0.20,
    CLEAN_MONTHS  = 6,           # event = first event after 6 months without one (either direction)
    MIN_FOLLOW    = 3,           # event months end 3 months before the last month
    PAYERWIDE_SHARE      = 0.50,
    PAYERWIDE_MIN_PAYEES = 5,
    HUB_TIER      = 2,

    # --- gates ------------------------------------------------------------------
    GATE_MIN_EVENTS    = 1000,
    GATE_MIN_PAYERWIDE = 0.10,

    # --- event study ----------------------------------------------------------
    PRE_MONTHS  = 6,
    POST_MONTHS = 6,
    N_CONTROLS  = 3,
    CONTROL_CLEAN_AFTER = 0,     # controls: no event in the 6 months before nor in the event month
    PANEL_PER_STRATUM   = 5000,  # events per stratum (direction x headline / payer-wide / single-link)
    EFFECT_T    = 3,
    BOOT_REPS   = 200,
    LOG_CAP     = 10.0,          # log changes capped at x10 / /10
    SEED        = 20261114,

    # --- cases ----------------------------------------------------------------
    CASE_TOP_LOSS = 6,
    CASE_TOP_GAIN = 4,
    CASE_NAME_PATTERNS = ["UNITED CONSUMER FINANCIAL"],

    # --- §5b what moves first ---------------------------------------------------
    ONSET_RUN = 2,               # onset = first month whose interval excludes zero, same sign, 2 months running

    # --- §5d robustness ------------------------------------------------------------------
    ALL_IN_DROP  = 0.30,         # all-inflow event: average of months e, e+1 at least 30% below / above the 6-month average
    ALL_IN_PREV  = 0.85,         # ... with month e-1 still within 15% of the average (not already moving)

    # --- §5e regression on event traits / §9 report -------------------------------------
    REG_REPS      = 200,         # bootstrap draws (clusters = main payer)
    REPORT_FILE   = "p1_report.html",
    REPORT_CASES  = False,       # case panels name customers: off by default for a shareable report

    # --- §10 exposure score (prototype) ---------------------------------------------
    RUN_SCORE          = True,
    STRESS_BREAK_SHARE = 0.25,   # a payer is under stress when it broke >= 25% of its standing PNC payees last month
    STRESS_MIN_PAYEES  = 3,      # ... with at least 3 standing PNC payees
    STRESS_OWN_MONTHS  = 6,      # ... or (PNC-customer payer) one of its own warning signals switched on in the last 6 months
    TIER_EDGES_SCORE   = (0.05, 0.10, 0.20),   # exposure share for tiers C / B / A
    SCORE_OWN_SIGNALS  = ["s_n", "s_brk", "s_neg", "s_pdue"],   # own warning signals counted over the last 3 months
    WATCH_TOP          = 40,     # rows shown per list (full lists saved to csv)

    # --- §5c who absorbs and who spreads ---------------------------------------------
    SUSC_TRAITS = {"ncpty_all_in": "payers (count)", "ncpty_all_out": "payees (count)",
                   "hhi_in": "payer concentration (HHI in)", "hhi_out": "payee concentration (HHI out)",
                   "ext_share_in": "external share of inflow", "ext_share_out": "external share of outflow",
                   "buffer_months": "balance buffer (months of outflow)", "family_size": "companies in its family"},
    SUSC_MIN_EVENTS = 150,       # smallest group shown

    # --- §6b early warning (look-back from real events) ---------------------------
    RUN_EARLY_WARNING = True,
    LOOKBACK      = 12,          # months looked back before each anchor event
    SIG_QUIET     = 6,           # a signal "switches on" when it fires after 6 months without firing
    SIG_DROP      = 0.70,        # "down 30%": below 70% of its own average over the 6 months before
    SIG_BROKEN    = 0.25,        # standing payees not paid: share >= 25%
    SIG_NETOUT    = 0.20,        # net outflow larger than 20% of average monthly inflow
    SIG_PD_MULT   = 1.5,         # PD up 50% against 6 months before
    ANCHOR_PD_MULT   = 2.0,      # anchor: PD doubles against 6 months before
    ANCHOR_DEP_DROP  = 0.50,     # anchor: average balance below 50% of its 6-month average, 2 months running
    ANCHOR_DEP_MIN   = 10_000.0, # ... with a 6-month average balance of at least 10K
    ANCHOR_MAX    = 20_000,      # anchors per type (stable hash sample)
    ANCHOR_CONTROLS = 3,

    # --- §7b real scenarios ----------------------------------------------------------
    CPTY_MAP_TABLE = f"{DB}.pkg_cpty_entity_map_v2",   # cpty_id -> ent_l2 (= pair table `other` for external)
    CPTY_DIM_TABLE = f"{DB}.pkg_cpty_dim",             # cpty_id -> dominant cpty_name
    SCENARIOS = [
        dict(kind="payer", name="Change Healthcare cyberattack", month="2024-02",
             patterns=["CHANGE HEALTHCARE"], note="data start 2024-01: one month before, descriptive only"),
        dict(kind="sector", name="2026 oil price shock (Strait of Hormuz, Feb 28 - Apr 6)", month="2026-03",
             naics2=["48", "49"], exposed_share=0.20, seasonal=True,   # minus the same months a year earlier
             control_exclude=["21", "22", "48", "49", "52", "92"]),
    ],
    SCEN_MAX_UNITS = 20_000,     # units per group in a sector scenario (stable hash sample)
)

DIRS = ["loss", "gain"]
LOSS_TYPES = ["stop", "skip", "short", "open"]
GAIN_TYPES = ["existing payer grew", "new payer"]

# Outcomes from the customer metrics: (label, kind, multiplier). Core ones are required;
# optional ones are dropped with a note if the table does not have them.
OUTCOMES = {
    "amt_all_in":               ("Incoming dollars",                          "logpct", 1.0),
    "amt_all_out":              ("Outgoing dollars",                          "logpct", 1.0),
    "ntxn_all_out":             ("Outgoing payment count",                    "logpct", 1.0),
    "active_days_out":          ("Days with outgoing payments",               "logpct", 1.0),
    "ncpty_ext_out":            ("External payees paid",                      "logpct", 1.0),
    "rec_standing_out":         ("Standing payees",                           "logpct", 1.0),
    "rec_broken_share_out":     ("Standing payees not paid (pp)",             "diff",   100.0),
    "rec_broken_amt_share_out": ("Standing payee dollars not paid (pp)",      "diff",   100.0),
    "net_flow_all":             ("Net flow (pp of monthly inflow)",           "scaled", 1.0),
}
OPTIONAL_OUTCOMES = {
    "new_cpty_ext_out":         ("New external payees per month",             "diff",   1.0),
    "lost_cpty_ext_out":        ("External payees dropped per month",         "diff",   1.0),
}
MET_REQUIRED  = ["mdm_id", "customer_pwr_id", "master_pwr_id", "customer_name", "party_type",
                 "naics2", "is_tm", "pwr_shared", "month", "rec_standing_in"] + list(OUTCOMES)
PAIR_REQUIRED = ["cust", "dir", "scope", "other", "amount", "month"]
CPTY_REQUIRED = ["ent_id", "month", "hub_tier"]

# --- Spark session (ACC code-review limits: 4g / 4g / 1g, 512 executors) --------
spark = (SparkSession.builder.appName("pkg_contagion_p1")
         .config("spark.yarn.queue", "root.ua.CIB-AMG.spark")
         .config("spark.executor.memory", "4g")
         .config("spark.driver.memory", "4g")
         .config("spark.driver.maxResultSize", "4g")
         .config("spark.executor.memoryOverhead", "1g")
         .config("spark.yarn.am.memory", "4g")
         .config("spark.yarn.am.memoryOverhead", "1g")
         .config("spark.dynamicAllocation.maxExecutors", "512")
         .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
         .config("spark.kryoserializer.buffer.max", "128m")
         .config("spark.sql.execution.arrow.pyspark.enabled", "true")
         .config("spark.ui.showConsoleProgress", "false")
         .config("spark.sql.shuffle.partitions", str(CONFIG["SHUFFLE_PARTITIONS"]))
         .config("spark.hadoop.dfs.replication", str(CONFIG["HDFS_REPLICATION"]))
         .enableHiveSupport().getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
try:    # cluster noise: Atlas lineage hook, Kafka, netty (harmless to data)
    _L = spark._jvm.org.apache.logging.log4j
    for _n in ("org.apache.atlas", "com.hortonworks.spark.atlas", "com.cloudera.spark.lineage",
               "org.apache.kafka", "org.apache.spark.sql.catalyst.util.package",
               "org.apache.spark.network", "io.netty"):
        _L.core.config.Configurator.setLevel(_n, _L.Level.OFF)
except Exception:
    pass
spark.conf.set("spark.sql.shuffle.partitions", str(CONFIG["SHUFFLE_PARTITIONS"]))

# --- display: one HTML output per cell; progress lines above the tables --------
try:
    from IPython.display import display, HTML
    _NB = "ipykernel" in sys.modules
except Exception:
    _NB = False


def _fmt(v):
    if v is None or (isinstance(v, float) and (math.isnan(v) or math.isinf(v))):
        return ""
    if isinstance(v, (bool, np.bool_)):
        return "yes" if v else "no"
    if isinstance(v, (int, np.integer)):
        return f"{int(v):,}"
    if isinstance(v, (float, np.floating)):
        a = abs(float(v))
        if a >= 1000:
            return f"{v:,.0f}"
        if a >= 10:
            return f"{v:,.1f}"
        return f"{v:,.3f}".rstrip("0").rstrip(".") if a else "0"
    return str(v)


def _html(df):
    d = df.copy()
    for c in d.columns:
        d[c] = d[c].map(_fmt)
    return d.to_html(index=False, border=0, escape=True)


class _Cell:
    """Progress lines and result tables of one cell, rendered as a single output."""
    def __init__(self):
        self.lines, self.tables, self.h = [], [], None

    def start(self, title):
        self.title, self.lines, self.tables = title, [], []
        self.h = display(HTML(""), display_id=True) if _NB else None
        self._draw()

    def log(self, txt, replace_last=False):
        if replace_last and self.lines:
            self.lines[-1] = txt
        else:
            self.lines.append(txt)
        self._draw()
        if not _NB:
            print("   ", txt, flush=True)

    def show(self, *blocks):
        self.tables = list(blocks)
        self._draw()
        if not _NB:
            for lab, df in blocks:
                print(f"\n== {lab}")
                print(df.to_string(index=False, max_rows=80))

    def _draw(self):
        if self.h is None:
            return
        css = ("<style>.pkgc table{border-collapse:collapse;font-size:12px;margin:2px 0 12px}"
               ".pkgc th{background:#f1f0ec;color:#0b0b0b;text-align:left;padding:3px 8px;border-bottom:1px solid #b8b7b0}"
               ".pkgc td{padding:2px 8px;border-bottom:1px solid #e6e5e0}"
               ".pkgc h4{margin:10px 0 3px;font-size:13px}.pkgc .pg{color:#8a8985;font-size:12px}</style>")
        body = f"<div class='pkgc'><h3 style='margin:4px 0'>{self.title}</h3>"
        body += "".join(f"<div class='pg'>{x}</div>" for x in self.lines)
        body += "".join(f"<h4>{lab}</h4>{_html(df)}" for lab, df in self.tables)
        self.h.update(HTML(css + body + "</div>"))


CELL = _Cell()
TIMINGS = []


@contextlib.contextmanager
def step(name):
    """Timed step with a live line: elapsed minutes and Spark tasks of the running stage."""
    t0 = time.time()
    CELL.log(f"running: {name} (started {time.strftime('%H:%M')})")
    stop = threading.Event()

    def _tick():
        while not stop.wait(30):
            msg = f"running: {name} ({(time.time() - t0) / 60:.1f} min"
            try:
                st = spark.sparkContext.statusTracker()
                ids = st.getActiveStageIds()
                if ids:
                    info = st.getStageInfo(max(ids))
                    msg += f", stage {info.stageId}: {info.numCompletedTasks:,}/{info.numTasks:,} tasks"
            except Exception:
                pass
            if _NB:
                CELL.log(msg + ")", replace_last=True)

    th = threading.Thread(target=_tick, daemon=True)
    th.start()
    try:
        yield
    finally:
        stop.set()
        mins = (time.time() - t0) / 60
        TIMINGS.append((name, round(mins, 2)))
        if _NB:
            CELL.log(f"done: {name} ({mins:.1f} min)", replace_last=True)
        else:
            print(f"    done: {name} ({mins:.1f} min)", flush=True)


GATES = []


def gate(name, ok, detail, action=""):
    GATES.append(dict(gate=name, result="PASS" if ok else "FAIL", detail=detail, action=action))


def gates_frame():
    return pd.DataFrame(GATES, columns=["gate", "result", "detail", "action"])


# --- months ---------------------------------------------------------------------
_BASE = int(CONFIG["MONTH_MIN"][:4]) * 12 + int(CONFIG["MONTH_MIN"][5:7])


def mi_col(c):
    """Integer month index, 0 = MONTH_MIN. Gaps are safe with rangeBetween windows."""
    c = F.col(c) if isinstance(c, str) else c
    return (F.substring(c, 1, 4).cast("int") * 12 + F.substring(c, 6, 2).cast("int") - F.lit(_BASE)).cast("int")


def mi2m(i):
    k = int(i) + _BASE
    return f"{(k - 1) // 12:04d}-{(k - 1) % 12 + 1:02d}"


def m2mi(m):
    return int(m[:4]) * 12 + int(m[5:7]) - _BASE


def table_months(tbl):
    try:
        p = spark.sql(f"SHOW PARTITIONS {tbl}").toPandas().iloc[:, 0]
        m = sorted({x.split("month=")[1].split("/")[0] for x in p if "month=" in x})
        if m:
            return m
    except Exception:
        pass
    return sorted(r[0] for r in spark.table(tbl).select("month").distinct().collect())


# --- work files: one folder per configuration, so a changed threshold never reuses old files
_SIG_KEYS = ["MET_TABLE", "PAIR_TABLE", "CPTY_TABLE", "USE_HUB", "MONTH_MIN", "ORG_PARTY_TYPES",
             "BASE_MONTHS", "BASE_IN_MIN", "SIZE_BANDS", "FIN_PUBLIC_NAICS2", "PNC_NAME_REGEX",
             "STANDING_MIN", "LOSS_MIN", "GAIN_MIN", "NEW_MONTHS", "TIER_EDGES", "CLEAN_MONTHS",
             "PAYERWIDE_SHARE", "PAYERWIDE_MIN_PAYEES", "HUB_TIER",
             "STEADY_MAX", "TENURE_MIN", "FAMILY_NET", "SPV_NAME_REGEX"]


_STAGE2 = {"stream_risk", "new_raw", "payer_month", "stream_events", "new_streams", "cust_events",
           "ew_signals", "ew_units", "score_standing", "score_stress", "score_all"}


def _work(name):
    if name in _STAGE2 or name.startswith("units_"):
        name = f"{name}_rp{SIG2}"
    return f"{CONFIG['WORK_DIR']}/v3_sig={SIG}/{name}"


def cache_parquet(sdf, name):
    """Write once, read back. Later runs with the same configuration skip the build."""
    path = _work(name)
    if not CONFIG["REBUILD"]:
        try:
            out = spark.read.parquet(path)
            out.limit(1).count()
            CELL.log(f"reused work file: {name}")
            return out
        except Exception:
            pass
    sdf.write.mode("overwrite").parquet(path)
    return spark.read.parquet(path)


# --- figures: saved to disk, indexed, never shown inline --------------------------
OUT_DIR = CONFIG["OUT_DIR"]
FIG_DIR = os.path.join(OUT_DIR, "figs")
os.makedirs(FIG_DIR, exist_ok=True)
for _f in os.listdir(FIG_DIR):                       # figures of an earlier run
    if _f[:1] == "f" and _f.endswith((".png", ".html")):
        os.remove(os.path.join(FIG_DIR, _f))
FIG_INDEX = []

INK, INK2, GRID, SURF = "#0b0b0b", "#52514e", "#e6e5e0", "#fcfcfb"
C_EVENT, C_CTRL = "#2a78d6", "#8a8985"          # event customers = blue, matched controls = gray
C_DIR = {"loss": "#2a78d6", "gain": "#eb6834"}  # direction (categorical slots 1, 2)
C_IN, C_OUT = "#008300", "#e34948"              # inflow green, outflow red (team chart rule)
TYPE_COLORS = {"stop": "#2a78d6", "skip": "#eb6834", "short": "#1baf7a", "open": "#eda100",
               "existing payer grew": "#2a78d6", "new payer": "#eb6834"}

plt.rcParams.update({
    "figure.facecolor": SURF, "axes.facecolor": SURF, "savefig.facecolor": SURF,
    "axes.edgecolor": "#b8b7b0", "axes.labelcolor": INK2, "text.color": INK,
    "xtick.color": INK2, "ytick.color": INK2, "axes.grid": True, "grid.color": GRID,
    "grid.linewidth": 0.8, "axes.axisbelow": True, "axes.spines.top": False, "axes.spines.right": False,
    "font.size": 10, "axes.titlesize": 11, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "legend.frameon": False, "lines.linewidth": 2.0, "figure.dpi": 110,
})


def savefig(fig, name, section, caption):
    path = os.path.join(FIG_DIR, f"{name}.png")
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    FIG_INDEX.append(dict(file=f"{name}.png", section=section, caption=caption))
    return path


def save_plotly(fig, name, section, caption):
    """Standalone HTML copy (opens in the browser; inline Plotly is empty in this JupyterLab)."""
    try:
        fig.write_html(os.path.join(FIG_DIR, f"{name}.html"), include_plotlyjs=True, full_html=True)
        FIG_INDEX.append(dict(file=f"{name}.html", section=section, caption=caption + " (interactive)"))
    except Exception as ex:
        CELL.log(f"interactive copy of {name} skipped: {type(ex).__name__}")


def usd(v):
    return "" if v is None or (isinstance(v, float) and math.isnan(v)) else f"${v:,.0f}"


def ci(e, lo, hi):
    """'-9.6 (-10.4 to -8.9)'; '(n.s.)' when the interval includes zero."""
    if e is None or not np.isfinite(e):
        return ""
    f = (lambda v: f"{v:+.3f}") if max(abs(e), abs(lo if np.isfinite(lo) else 0), abs(hi if np.isfinite(hi) else 0)) < 0.1 \
        else (lambda v: f"{v:+.1f}")
    s = f"{f(e)} ({f(lo)} to {f(hi)})" if np.isfinite(lo) and np.isfinite(hi) else f(e)
    return s + ("" if (np.isfinite(lo) and np.isfinite(hi) and (lo > 0 or hi < 0)) else " n.s.")


def did_boot(s, c, sg, cg, key, reps=None):
    """Group mean of s minus group mean of c, per column, with a cluster bootstrap (clusters sg / cg, e.g. masters).
    s, c: units x months arrays of each unit's change from its own baseline (NaN = not measured)."""
    reps = reps or CONFIG["BOOT_REPS"]
    s, c = np.asarray(s, float), np.asarray(c, float)
    uniq = np.unique(np.concatenate([np.asarray(sg, str), np.asarray(cg, str)]))
    si, cix = np.searchsorted(uniq, np.asarray(sg, str)), np.searchsorted(uniq, np.asarray(cg, str))
    K, T = len(uniq), s.shape[1]
    Ss, Ns, Sc, Nc = (np.zeros((K, T)) for _ in range(4))
    np.add.at(Ss, si, np.nan_to_num(s)); np.add.at(Ns, si, ~np.isnan(s))
    np.add.at(Sc, cix, np.nan_to_num(c)); np.add.at(Nc, cix, ~np.isnan(c))
    rng = np.random.default_rng(CONFIG["SEED"] + int(hashlib.md5(str(key).encode()).hexdigest()[:6], 16))
    Wt = rng.multinomial(K, np.full(K, 1.0 / K), size=reps).astype(float)
    with np.errstate(all="ignore"), warnings.catch_warnings():
        warnings.simplefilter("ignore")
        ms, mc = Ss.sum(0) / Ns.sum(0), Sc.sum(0) / Nc.sum(0)
        bd = (Wt @ Ss) / (Wt @ Ns) - (Wt @ Sc) / (Wt @ Nc)
        lo, hi = np.nanpercentile(bd, 2.5, axis=0), np.nanpercentile(bd, 97.5, axis=0)
    return dict(ms=ms, mc=mc, d=ms - mc, lo=lo, hi=hi, n=np.sum(~np.isnan(s), axis=0), bd=bd)


def onset_read(ts, eff, lo, hi, run=None, t_from=0):
    """When does an effect start, reach half its peak, and peak?
    peak  = largest |effect| from month t_from on; its sign is the direction of the move.
    onset = first month whose interval excludes zero on the peak's side for `run` months in a row
            (a run cut by the end of the window counts); may be before the event.
    half  = first month the effect reaches half the peak on the peak's side."""
    run = run or CONFIG["ONSET_RUN"]
    ts, eff, lo, hi = (np.asarray(x, float) for x in (ts, eff, lo, hi))
    ok = np.isfinite(eff)
    post = ok & (ts >= t_from)
    out = dict(onset=np.nan, half=np.nan, peak_t=np.nan, peak=np.nan, clear=False)
    if not post.any():
        return out
    k = int(np.nanargmax(np.where(post, np.abs(eff), -np.inf)))
    sgn = np.sign(eff[k]) or 1.0
    out.update(peak_t=ts[k], peak=eff[k])
    sig = ok & np.isfinite(lo) & np.isfinite(hi) & ((lo > 0) if sgn > 0 else (hi < 0))
    for i in range(len(ts)):
        j = min(i + run, len(ts))
        if sig[i:j].all() and sig[i]:
            out.update(onset=ts[i], clear=True)
            break
    reach = ok & (sgn * eff >= 0.5 * abs(eff[k]))
    if reach.any():
        out["half"] = ts[int(np.argmax(reach))]
    return out


def fam_table(name, keep):
    """Outcome family (EXTRA_OUTCOMES entry that §0 found) as (key, mi, measures...), measures named by their keys.
    key = customer_pwr_id or cust. A family with a mask gets `_bad` = 1 for every month of a customer that has the
    mask column above zero in one of the mask months (acquisition)."""
    x = {f["name"]: f for f in EXTRA_ACTIVE}.get(name)
    if x is None:
        return None, None
    ms = {k: m["src"] for k, m in x["measures"].items() if k in keep}
    if not ms:
        return None, None
    if x.get("mask"):
        ms["_mask"] = x["mask"]["col"]
    key = "cust" if x["id_kind"] == "mdm_id" else "customer_pwr_id"
    mc = F.col(x["month_col"])
    mstr = (F.when(mc.cast("string").rlike(r"^\d{4}-\d{2}$"), mc.cast("string"))
            .otherwise(F.date_format(mc.cast("date"), "yyyy-MM")))
    t = (spark.table(x["table"])
         .select(F.col(x["id_col"]).cast("string").alias(key), mi_col(mstr).alias("mi"),
                 *[F.col(src).cast("double").alias(k) for k, src in ms.items()])
         .where(F.col("mi").between(0, MAX_MI))
         .groupBy(key, "mi").agg(*[F.avg(k).alias(k) for k in ms]))
    if "_mask" in ms:
        mm = [m2mi(v) for v in x["mask"]["months"]]
        t = (t.withColumn("_bad", F.when(F.col("mi").isin(mm) & (F.col("_mask") > 0), 1).otherwise(0))
             .withColumn("_bad", F.max("_bad").over(W.partitionBy(key))).drop("_mask"))
    return t, key


ANSWERS = []


def ans(q, a):
    ANSWERS.append(dict(question=q, answer=a))

## §0 · Inputs

Confirms every column this notebook depends on exists and lists the months of each input. Outcome families
whose configured names are not in their table are listed and skipped, nothing else stops. A missing
required column stops the run here, before any heavy work.

In [ ]:
CELL.start("§0 · Inputs")
with step("schemas and months"):
    _req = [("customer metrics", CONFIG["MET_TABLE"], MET_REQUIRED),
            ("customer x counterparty pairs", CONFIG["PAIR_TABLE"], PAIR_REQUIRED)]
    if CONFIG["USE_HUB"]:
        _req.append(("counterparty metrics", CONFIG["CPTY_TABLE"], CPTY_REQUIRED))
    _rows, _missing, _months = [], {}, {}
    for _lab, _t, _need in _req:
        _cols = spark.table(_t).columns
        _miss = [c for c in _need if c not in _cols]
        _months[_t] = table_months(_t)
        _rows.append(dict(input=_lab, table=_t, columns=len(_cols),
                          months=f"{_months[_t][0]} .. {_months[_t][-1]} ({len(_months[_t])})",
                          missing_columns=", ".join(_miss) if _miss else "none"))
        if _miss:
            _missing[_t] = _miss
    _met_cols = set(spark.table(CONFIG["MET_TABLE"]).columns)
    for _c, _v in OPTIONAL_OUTCOMES.items():
        if _c in _met_cols:
            OUTCOMES[_c] = _v
    _opt_missing = [c for c in OPTIONAL_OUTCOMES if c not in _met_cols]
    _inputs = pd.DataFrame(_rows)

    _xrows, EXTRA_ACTIVE = [], []
    for _x in CONFIG["EXTRA_OUTCOMES"]:
        try:
            _names = spark.table(_x["table"]).columns
        except Exception:
            _xrows.append(dict(family=_x["name"], table=_x["table"], status="table not found", measures=""))
            continue
        _want = [_x["id_col"], _x["month_col"]] + [m["src"] for m in _x["measures"].values()]
        if _x.get("mask"):
            _want.append(_x["mask"]["col"])
        _miss = [w for w in _want if w not in _names]
        if _miss:
            _st = "skipped, not in table: " + ", ".join(_miss)
        else:
            _st = "active"
            EXTRA_ACTIVE.append(_x)
        try:
            _xm = table_months(_x["table"]) if not _miss else []
            _mtxt = f"{_xm[0]} .. {_xm[-1]} ({len(_xm)})" if _xm else ""
        except Exception:
            _mtxt = ""
        _xrows.append(dict(family=_x["name"], table=_x["table"], status=_st, months=_mtxt,
                           measures=", ".join(m["src"] for m in _x["measures"].values())))
    _extra = pd.DataFrame(_xrows)

CELL.show(("Required inputs", _inputs),
          ("Outcome families", _extra),
          ("Optional payment outcomes not in the metrics table", pd.DataFrame({"column": _opt_missing or ["none"]})))
if _missing:
    raise ValueError("Required columns not found, nothing was built: "
                     + "; ".join(f"{t}: {', '.join(m)}" for t, m in _missing.items()))

MONTH_MIN = CONFIG["MONTH_MIN"]
_common = sorted(set(_months[CONFIG["MET_TABLE"]]) & set(_months[CONFIG["PAIR_TABLE"]]))
_common = [m for m in _common if m >= MONTH_MIN and (CONFIG["MONTH_MAX"] is None or m <= CONFIG["MONTH_MAX"])]
BM, CLEAN = CONFIG["BASE_MONTHS"], CONFIG["CLEAN_MONTHS"]
if len(_common) < BM + CLEAN + CONFIG["MIN_FOLLOW"] + 3:
    raise ValueError(f"Only {len(_common)} common months between metrics and pairs: too few for the event study.")
MONTH_MAX = _common[-1]
MAX_MI = m2mi(MONTH_MAX)
EV_START, EV_END = BM + CLEAN, MAX_MI - CONFIG["MIN_FOLLOW"]
gate("months are continuous", len(_common) == MAX_MI + 1,
     f"{len(_common)} months present, {MAX_MI + 1} expected between {MONTH_MIN} and {MONTH_MAX}",
     "a missing month would look like a wave of stops")
SIG = hashlib.md5(json.dumps({k: CONFIG[k] for k in _SIG_KEYS} | {"max": MONTH_MAX},
                             sort_keys=True, default=str).encode()).hexdigest()[:10]
SIG2 = hashlib.md5(json.dumps([CONFIG["RELATED_EXCLUDE"], CONFIG["NAME_STOP"], CONFIG["NAME_TOKEN_MAX"],
                               CONFIG["PNC_PAYER_HUB"], CONFIG["SELF_EXCLUDE"]]).encode()).hexdigest()[:6]
ans("Window", f"{MONTH_MIN} .. {MONTH_MAX}; events {mi2m(EV_START)} .. {mi2m(EV_END)}; configuration {SIG}-{SIG2}")

## §1 · Population

TM organisations (`is_tm`, `party_type` O, power id not shared), one row per customer per month.
**Baseline inflow** = average monthly `amt_all_in` over the 6 months before. A customer-month is **eligible**
with at least one standing incoming relationship (`rec_standing_in`) and baseline inflow above the floor.

- **Segment:** PNC own (name matches PNC), finance & public (NAICS2 52 or 92, or a receivables / funding vehicle
  by name), operating (everything else).
- **Size band:** baseline monthly inflow under 100K, 100K to 1M, 1M and over.

In [ ]:
CELL.start("§1 · Population")
with step("TM organisations, baseline inflow, eligibility"):
    _met = spark.table(CONFIG["MET_TABLE"]).where(F.col("month").between(MONTH_MIN, MONTH_MAX))
    _tm = _met.where((F.col("is_tm").cast("boolean") == F.lit(True))
                     & F.col("party_type").isin(CONFIG["ORG_PARTY_TYPES"])
                     & (F.coalesce(F.col("pwr_shared").cast("boolean"), F.lit(False)) == F.lit(False)))
    _pop = _tm.select(
        F.col("mdm_id").cast("string").alias("cust"),
        F.col("customer_pwr_id").cast("string").alias("customer_pwr_id"),
        F.coalesce(F.col("master_pwr_id").cast("string"), F.col("mdm_id").cast("string")).alias("master"),
        F.col("customer_name"),
        F.coalesce(F.col("naics2").cast("string"), F.lit("NA")).alias("naics2n"),
        mi_col("month").alias("mi"),
        F.col("rec_standing_in").cast("double").alias("rec_standing_in"),
        *[F.col(c).cast("double").alias(c) for c in OUTCOMES])
    _w6 = W.partitionBy("cust").orderBy("mi").rangeBetween(-BM, -1)
    _pop = _pop.withColumn("base_in", F.sum("amt_all_in").over(_w6) / F.lit(float(BM)))
    pop = cache_parquet(_pop, "pop")

    _b1, _b2 = CONFIG["SIZE_BANDS"]
    _elig = (pop.where((F.col("mi") >= BM) & (F.col("rec_standing_in") >= 1)
                       & (F.col("base_in") >= CONFIG["BASE_IN_MIN"]))
             .withColumn("pbin", F.ntile(100).over(W.partitionBy("mi").orderBy("base_in")))
             .withColumn("segment",
                         F.when(F.upper(F.coalesce(F.col("customer_name"), F.lit(""))).rlike(CONFIG["PNC_NAME_REGEX"]),
                                F.lit("PNC own"))
                         .when(F.col("naics2n").isin(CONFIG["FIN_PUBLIC_NAICS2"])
                               | F.upper(F.coalesce(F.col("customer_name"), F.lit(""))).rlike(CONFIG["SPV_NAME_REGEX"]),
                               F.lit("finance & public"))
                         .otherwise(F.lit("operating")))
             .withColumn("size_band", F.when(F.col("base_in") < _b1, F.lit("small"))
                         .when(F.col("base_in") < _b2, F.lit("mid")).otherwise(F.lit("large")))
             .select("cust", "customer_pwr_id", "master", "customer_name", "naics2n", "mi", "base_in", "pbin",
                     "segment", "size_band"))
    elig = cache_parquet(_elig, "elig")

    _pm = (pop.groupBy("mi").agg(
        F.count(F.lit(1)).alias("tm_orgs"),
        F.sum((F.col("rec_standing_in") >= 1).cast("long")).alias("with_standing_in"),
        F.sum("amt_all_in").alias("in_usd"),
        F.sum(F.when(F.col("rec_standing_in") >= 1, F.col("amt_all_in")).otherwise(0.0)).alias("in_usd_standing"))
        .join(elig.groupBy("mi").agg(F.count(F.lit(1)).alias("eligible")), "mi", "left")
        .orderBy("mi").toPandas())
    _seg = (elig.where(F.col("mi") == MAX_MI).groupBy("segment", "size_band").count().toPandas()
            .pivot(index="segment", columns="size_band", values="count").fillna(0).astype(int)
            .reindex(columns=["small", "mid", "large"]).fillna(0).astype(int).reset_index())

if _pm["tm_orgs"].sum() == 0:
    raise ValueError("No TM organisation rows found. Check is_tm and ORG_PARTY_TYPES.")
_pm.insert(0, "month", _pm["mi"].map(mi2m))
_pm["eligible"] = _pm["eligible"].fillna(0).astype(int)
_pm["standing_share"] = _pm["with_standing_in"] / _pm["tm_orgs"]
_pm["standing_usd_share"] = _pm["in_usd_standing"] / _pm["in_usd"]
_last = _pm.iloc[-1]
CELL.show(("Population, last month", pd.DataFrame([dict(
              tm_orgs=int(_last.tm_orgs), with_standing_in=int(_last.with_standing_in),
              standing_share=float(_last.standing_share), standing_usd_share=float(_last.standing_usd_share),
              eligible=int(_last.eligible))])),
          ("Eligible customers, last month, by segment and size band", _seg),
          ("By month", _pm[["month", "tm_orgs", "with_standing_in", "standing_share", "standing_usd_share", "eligible"]]))
ans("TM organisations with a standing incoming relationship (last month)",
    f"{int(_last.with_standing_in):,} of {int(_last.tm_orgs):,} ({_last.standing_share:.1%}), "
    f"holding {_last.standing_usd_share:.1%} of incoming dollars")

## §2 · Stream breaks and gains, customer events

A **stream** is customer x payer (internal or external), incoming side of the pair table. A stream is
**standing** going into month e when it was paid in at least 4 of the 6 months before, including month e-1,
in at least 8 of the 12 months before, and was **steady**: its last 3 paid months average at most 1.5x the median
of its paid months in the last 6 (a surge ending is not a loss). Expected amount = median of the last 3 paid months.

**Family moves:** when the same payer starts or grows paying a sister company (same master) in months e-1..e+1,
a break is cancelled if the sister's gain covers at least half of it, and the same the other way round.

| Stream event | Rule |
|---|---|
| break | net shortfall over e-1, e, e+1 against 3 x expected is at least 50% of expected (shift-proof) |
| - stop | break, not paid in e nor e+1 |
| - skip | break, not paid in e, paid in e+1 |
| - short | break, paid in e but less |
| - open | break in the last month of data (cannot be typed yet) |
| grow | paid at least 150% of expected in both e and e+1 |
| new payer | paid in e, e+1, e+2, not paid in the 6 months before |

**Customer event** (per month e): net loss = lost dollars - grown dollars - new payers starting in e or e+1;
net gain = grown dollars + new payers starting in e - lost dollars. An event needs at least 10% of baseline
inflow. The first event after 6 event-free months is used, one per customer and direction, from 2025-01.
**Payer-wide** = the payer broke (or grew) at least half of its standing PNC payees that month, 5 payees or more.

In [ ]:
CELL.start("§2 · Stream breaks and gains, customer events")
SM, LOSS_MIN, GAIN_MIN = CONFIG["STANDING_MIN"], CONFIG["LOSS_MIN"], CONFIG["GAIN_MIN"]
TIER0 = CONFIG["TIER_EDGES"][0]
PWS, PWN = CONFIG["PAYERWIDE_SHARE"], CONFIG["PAYERWIDE_MIN_PAYEES"]

with step("stream windows (pair table, incoming side)"):
    _pair = (spark.table(CONFIG["PAIR_TABLE"])
             .where(F.col("month").between(MONTH_MIN, MONTH_MAX)
                    & (F.col("dir") == "in") & F.col("scope").isin("int", "ext"))
             .select(F.col("cust").cast("string").alias("cust"), "scope",
                     F.col("other").cast("string").alias("other"),
                     mi_col("month").alias("mi"), F.col("amount").cast("double").alias("amount"))
             .join(pop.select("cust").distinct(), "cust", "left_semi")
             .groupBy("cust", "scope", "other", "mi").agg(F.sum("amount").alias("amount"))
             .where(F.col("amount") > 0))
    _w = W.partitionBy("cust", "scope", "other").orderBy("mi")
    _win = (_pair
            .withColumn("cnt6", F.count(F.lit(1)).over(_w.rangeBetween(-5, 0)))
            .withColumn("cnt12", F.count(F.lit(1)).over(_w.rangeBetween(-11, 0)))
            .withColumn("last6", F.collect_list("amount").over(_w.rangeBetween(-5, 0)))
            .withColumn("prev_mi", F.lag("mi").over(_w))
            .withColumn("l1_mi", F.lead("mi").over(_w)).withColumn("l1_amt", F.lead("amount").over(_w))
            .withColumn("l2_mi", F.lead("mi", 2).over(_w)).withColumn("l2_amt", F.lead("amount", 2).over(_w))
            .withColumn("last3", F.collect_list("amount").over(_w.rowsBetween(-2, 0))))
    win = cache_parquet(_win, "stream_win")

with step("outgoing dollars to regular payees vs everyone else (pair table, outgoing side)"):
    _pc = set(spark.table(CONFIG["PAIR_TABLE"]).columns)
    _ntx = F.col("n_txn").cast("double") if "n_txn" in _pc else F.lit(None).cast("double")
    _po = (spark.table(CONFIG["PAIR_TABLE"])
           .where((F.col("dir") == "out") & F.col("scope").isin("int", "ext") & F.col("month").between(MONTH_MIN, MONTH_MAX))
           .select(F.col("cust").cast("string").alias("cust"), F.col("scope"), F.col("other").cast("string").alias("other"),
                   mi_col("month").alias("mi"), F.col("amount").cast("double").alias("amount"), _ntx.alias("n_txn"))
           .join(pop.select("cust").distinct(), "cust", "left_semi")
           .groupBy("cust", "scope", "other", "mi").agg(F.sum("amount").alias("amount"), F.sum("n_txn").alias("n_txn")))
    _wo = W.partitionBy("cust", "scope", "other").orderBy("mi").rangeBetween(-BM, -1)
    _po = _po.withColumn("reg", F.count(F.lit(1)).over(_wo) >= SM)
    _sw = lambda c, cond: F.sum(F.when(cond, F.col(c)).otherwise(F.lit(0.0)))
    out_reg = cache_parquet(_po.groupBy("cust", "mi").agg(
        _sw("amount", F.col("reg")).alias("amt_reg_out"), _sw("amount", ~F.col("reg")).alias("amt_irr_out"),
        _sw("n_txn", F.col("reg")).alias("ntxn_reg_out"), _sw("n_txn", ~F.col("reg")).alias("ntxn_irr_out")), "out_reg")
    REG_OUTCOMES = {"amt_reg_out": ("Outgoing dollars to regular payees", "logpct", 1.0),
                    "amt_irr_out": ("Outgoing dollars to other payees", "logpct", 1.0)}
    if "n_txn" in _pc:
        REG_OUTCOMES.update({"ntxn_reg_out": ("Payments to regular payees", "logpct", 1.0),
                             "ntxn_irr_out": ("Payments to other payees", "logpct", 1.0)})

with step("self-payment: own accounts at other banks (customer metrics)"):
    _mc0 = set(spark.table(CONFIG["MET_TABLE"]).columns)
    SELF_OUTCOMES = {}
    if {"selfpay_amt_in", "selfpay_amt_out"} <= _mc0:
        selfpay = cache_parquet(
            spark.table(CONFIG["MET_TABLE"]).where(F.col("month").between(MONTH_MIN, MONTH_MAX))
            .select(F.col("mdm_id").cast("string").alias("cust"), mi_col("month").alias("mi"),
                    F.coalesce(F.col("selfpay_amt_in").cast("double"), F.lit(0.0)).alias("selfpay_amt_in"),
                    F.coalesce(F.col("selfpay_amt_out").cast("double"), F.lit(0.0)).alias("selfpay_amt_out"))
            .join(pop.select("cust").distinct(), "cust", "left_semi")
            .withColumn("selfpay_net", F.col("selfpay_amt_in") - F.col("selfpay_amt_out")), "selfpay")
        SELF_OUTCOMES = {"selfpay_amt_in": ("Money in from own accounts at other banks", "logpct", 1.0),
                         "selfpay_amt_out": ("Money out to own accounts at other banks", "logpct", 1.0),
                         "selfpay_net": ("Net money in from own accounts elsewhere (pp of monthly inflow)", "scaled", 1.0)}
    else:
        selfpay = None

with step("standing streams: breaks (net of timing shifts) and growth"):
    _z = F.lit(0.0)
    _ten = (F.col("cnt12") >= CONFIG["TENURE_MIN"]) if CONFIG["TENURE_MIN"] else F.lit(True)
    _s = (win.where((F.col("cnt6") >= SM) & (F.col("mi") < MAX_MI) & _ten)
          .withColumn("med6", F.expr("element_at(array_sort(last6), cast(ceil(size(last6) / 2.0) as int))"))
          .where(F.expr("aggregate(last3, cast(0 as double), (a, x) -> a + x) / size(last3)")
                 <= CONFIG["STEADY_MAX"] * F.col("med6"))
          .withColumn("e", F.col("mi") + 1)
          .withColumn("expected", F.expr("element_at(array_sort(last3), cast(ceil(size(last3) / 2.0) as int))"))
          .withColumn("a_e", F.when(F.col("l1_mi") == F.col("e"), F.col("l1_amt")).otherwise(_z))
          .withColumn("a_e1", F.when(F.col("l1_mi") == F.col("e") + 1, F.col("l1_amt"))
                      .when(F.col("l2_mi") == F.col("e") + 1, F.col("l2_amt")).otherwise(_z))
          .withColumn("has_e1", F.col("e") + 1 <= F.lit(MAX_MI))
          .withColumn("gross", F.greatest(F.col("expected") - F.col("a_e"), _z))
          .withColumn("net", F.when(F.col("has_e1"), F.lit(3.0)).otherwise(F.lit(2.0)) * F.col("expected")
                      - F.col("amount") - F.col("a_e") - F.col("a_e1"))
          .withColumn("lost_raw", F.greatest(F.least(F.col("gross"), F.col("net")), _z))
          .withColumn("gross_break", F.col("gross") >= LOSS_MIN * F.col("expected"))
          .withColumn("broke", F.col("lost_raw") >= LOSS_MIN * F.col("expected"))
          .withColumn("ev", F.when(~F.col("broke"), F.lit(None).cast("string"))
                      .when(F.col("a_e") > 0, F.lit("short"))
                      .when(~F.col("has_e1"), F.lit("open"))
                      .when(F.col("a_e1") == 0, F.lit("stop"))
                      .otherwise(F.lit("skip")))
          .withColumn("lost", F.when(F.col("broke"), F.col("lost_raw")).otherwise(_z))
          .withColumn("g_raw", F.when(F.col("has_e1"), F.least(F.col("a_e"), F.col("a_e1")) - F.col("expected")).otherwise(_z))
          .withColumn("grew", F.col("g_raw") >= GAIN_MIN * F.col("expected"))
          .withColumn("gain", F.when(F.col("grew"), F.col("g_raw")).otherwise(_z))
          .select("cust", "scope", "other", "e", "expected", F.col("amount").alias("a_prev"), "a_e", "a_e1",
                  "gross_break", "broke", "ev", "lost", "grew", "gain"))
    risk = cache_parquet(_s, "stream_risk_raw")

with step("new regular payers"):
    _nm = CONFIG["NEW_MONTHS"]
    _new = (win.where((F.col("mi") >= BM) & (F.col("mi") + _nm - 1 <= MAX_MI)
                      & (F.col("prev_mi").isNull() | (F.col("mi") - F.col("prev_mi") > BM))
                      & (F.col("l1_mi") == F.col("mi") + 1)
                      & ((F.col("l2_mi") == F.col("mi") + 2) if _nm >= 3 else F.lit(True)))
            .select("cust", "scope", "other", F.col("mi").alias("e"),
                    (F.least(F.col("amount"), F.col("l1_amt"), F.col("l2_amt")) if _nm >= 3
                     else F.least(F.col("amount"), F.col("l1_amt"))).alias("new_amt")))
    new_raw = cache_parquet(_new, "new_raw0")

with step("related payers: same master or a shared distinctive name word"):
    _stop_sql = "array(" + ",".join(f"'{w}'" for w in CONFIG["NAME_STOP"]) + ")"
    _tokx = lambda c: F.expr(f"element_at(filter(split(trim(regexp_replace(upper(coalesce({c}, '')), '[^A-Z0-9]+', ' ')), ' '), "
                             f"x -> length(x) >= 4 AND NOT array_contains({_stop_sql}, x)), 1)")
    try:
        _dim = (spark.table(CONFIG["DIM_TABLE"]).select(F.col("mdm_id").cast("string").alias("id"),
                                                       F.col("customer_name").alias("nm"),
                                                       F.col("master_pwr_id").cast("string").alias("ms")))
        _dim_src = "pkg_cust_dim"
    except Exception:
        _dim = pop.select(F.col("cust").alias("id"), F.col("customer_name").alias("nm"), F.col("master").alias("ms"))
        _dim_src = "population (pkg_cust_dim not readable)"
    _dim = (_dim.dropDuplicates(["id"]).withColumn("tok", _tokx("nm"))
            .withColumn("ms", F.coalesce(F.col("ms"), F.col("id"))).drop("nm")
            # a word shared by many unrelated names (JOHNSON, SAINT ...) is not evidence of a relation
            .withColumn("tok", F.when(F.count(F.lit(1)).over(W.partitionBy("tok")) <= CONFIG["NAME_TOKEN_MAX"], F.col("tok"))))
    _rel_n = 0
    if CONFIG["RELATED_EXCLUDE"]:
        def _related(df):
            """internal (cust, payer) pairs in one master, or sharing the first distinctive name word"""
            x = (df.where(F.col("scope") == "int").select("cust", "other").distinct()
                 .join(_dim.select(F.col("id").alias("cust"), F.col("ms").alias("ms_c"), F.col("tok").alias("tk_c")), "cust", "left")
                 .join(_dim.select(F.col("id").alias("other"), F.col("ms").alias("ms_o"), F.col("tok").alias("tk_o")), "other", "left")
                 .where((F.col("ms_c") == F.col("ms_o")) | (F.col("tk_c").isNotNull() & (F.col("tk_c") == F.col("tk_o"))))
                 .select("cust", "other").withColumn("scope", F.lit("int")))
            return x
        _relp = _related(risk.select("cust", "scope", "other").unionByName(new_raw.select("cust", "scope", "other")))
        _rel_n = int(risk.join(_relp, ["cust", "scope", "other"], "left_semi").count())
        risk = risk.join(_relp, ["cust", "scope", "other"], "left_anti")
        new_raw = new_raw.join(_relp, ["cust", "scope", "other"], "left_anti")
    _self_n = dict(stream_months=0, breaks=0, dollars=0.0)
    _selfp = None
    if CONFIG["SELF_EXCLUDE"] and CONFIG["PAIR_SELF_COL"] in set(spark.table(CONFIG["PAIR_TABLE"]).columns):
        _sf = F.lower(F.col(CONFIG["PAIR_SELF_COL"]).cast("string")).isin("true", "1", "y", "yes")
        _selfp = cache_parquet(spark.table(CONFIG["PAIR_TABLE"])
                               .where(F.col("month").between(MONTH_MIN, MONTH_MAX) & (F.col("dir") == "in")
                                      & (F.col("scope") == "ext") & _sf)
                               .select(F.col("cust").cast("string").alias("cust"), F.lit("ext").alias("scope"),
                                       F.col("other").cast("string").alias("other")).distinct(), "self_pairs")
        _sr = (risk.join(_selfp, ["cust", "scope", "other"], "left_semi")
               .agg(F.count(F.lit(1)).alias("n"), F.sum(F.col("broke").cast("long")).alias("b"),
                    F.sum("lost").alias("d")).toPandas().iloc[0])
        _self_n = dict(stream_months=int(_sr.n or 0), breaks=int(_sr.b or 0), dollars=float(_sr.d or 0.0))
        risk = risk.join(_selfp, ["cust", "scope", "other"], "left_anti")
        new_raw = new_raw.join(_selfp, ["cust", "scope", "other"], "left_anti")
    _ex = [x for x in [_relp if CONFIG["RELATED_EXCLUDE"] else None, _selfp] if x is not None]
    EXCL_PAIRS = None
    if _ex:
        EXCL_PAIRS = _ex[0].select("cust", "scope", "other")
        for _x in _ex[1:]:
            EXCL_PAIRS = EXCL_PAIRS.unionByName(_x.select("cust", "scope", "other"))

with step("family moves: a payer shifting money between companies of one master cancels out"):
    _cm = pop.select("cust", "master").dropDuplicates(["cust"])
    _lo = (risk.where(F.col("broke")).select("cust", "scope", "other", "e", F.col("lost").alias("amt"))
           .join(_cm, "cust"))
    _ga = (risk.where(F.col("grew")).select("cust", "scope", "other", "e", F.col("gain").alias("amt"))
           .unionByName(new_raw.select("cust", "scope", "other", "e", F.col("new_amt").alias("amt")))
           .join(_cm, "cust"))

    def _fam(a, b, name):
        """Per stream in a: dollars of b from the same payer to a sister company within one month."""
        bb = b.select(F.col("master"), "scope", "other", F.col("cust").alias("c2"), F.col("e").alias("e2"),
                      F.col("amt").alias("a2"))
        return (a.join(bb, ["master", "scope", "other"], "inner")
                .where((F.col("c2") != F.col("cust")) & (F.abs(F.col("e2") - F.col("e")) <= 1))
                .groupBy("cust", "scope", "other", "e").agg(F.sum("a2").alias(name)))
    if CONFIG["FAMILY_NET"]:
        _fl, _fg = _fam(_lo, _ga, "fam_gain"), _fam(_ga, _lo, "fam_loss")
    else:
        _fl = spark.createDataFrame([], "cust string, scope string, other string, e int, fam_gain double")
        _fg = spark.createDataFrame([], "cust string, scope string, other string, e int, fam_loss double")
    _z = F.lit(0.0)
    _rk = (risk.join(_fl, ["cust", "scope", "other", "e"], "left").join(_fg, ["cust", "scope", "other", "e"], "left")
           .na.fill(0.0, ["fam_gain", "fam_loss"])
           .withColumn("fam_move", (F.col("broke") & (F.col("fam_gain") >= 0.5 * F.col("lost")))
                       | (F.col("grew") & (F.col("fam_loss") >= 0.5 * F.col("gain"))))
           .withColumn("broke", F.col("broke") & ~F.col("fam_move"))
           .withColumn("ev", F.when(F.col("broke"), F.col("ev")))
           .withColumn("lost", F.when(F.col("broke"), F.col("lost")).otherwise(_z))
           .withColumn("grew", F.col("grew") & ~F.col("fam_move"))
           .withColumn("gain", F.when(F.col("grew"), F.col("gain")).otherwise(_z)))
    risk = cache_parquet(_rk, "stream_risk")
    _n_new0 = new_raw.count()
    _nw = (new_raw.join(_fg, ["cust", "scope", "other", "e"], "left").na.fill(0.0, ["fam_loss"])
           .where(F.col("fam_loss") < 0.5 * F.col("new_amt")).drop("fam_loss"))
    new_raw = cache_parquet(_nw, "new_raw")
    _famn = int(risk.agg(F.sum(F.col("fam_move").cast("long")).alias("n")).toPandas().iloc[0]["n"] or 0)
    _famn += int(_n_new0 - new_raw.count())

with step("payer-month roll-up, payer-wide and hub flags"):
    _pr = risk.groupBy("scope", "other", "e").agg(
        F.count(F.lit(1)).alias("n_risk"),
        F.sum(F.col("broke").cast("int")).alias("n_broken"),
        F.sum(F.col("grew").cast("int")).alias("n_grow"))
    _pn = new_raw.groupBy("scope", "other", "e").agg(F.count(F.lit(1)).alias("n_new"))
    _pmo = (_pr.join(_pn, ["scope", "other", "e"], "full").na.fill(0, ["n_risk", "n_broken", "n_grow", "n_new"])
            .withColumn("pw_loss", ((F.col("n_risk") >= PWN) & (F.col("n_broken") >= PWS * F.col("n_risk"))).cast("int"))
            .withColumn("pw_gain", ((F.col("n_risk") >= PWN) & (F.col("n_grow") >= PWS * F.col("n_risk"))).cast("int"))
            .withColumn("pw_new", ((F.col("n_new") >= PWN)
                                   & (F.col("n_new") >= PWS * (F.col("n_risk") + F.col("n_new")))).cast("int")))
    if CONFIG["USE_HUB"]:
        _hub = (spark.table(CONFIG["CPTY_TABLE"])
                .where(F.col("month").between(MONTH_MIN, MONTH_MAX)
                       & (F.col("hub_tier").cast("int") >= CONFIG["HUB_TIER"]))
                .select(F.lit("ext").alias("scope"), F.col("ent_id").cast("string").alias("other"),
                        (mi_col("month") + 1).alias("e")).distinct().withColumn("is_hub", F.lit(1)))
        _pmo = _pmo.join(F.broadcast(_hub), ["scope", "other", "e"], "left")
    else:
        _pmo = _pmo.withColumn("is_hub", F.lit(None).cast("int"))
    if CONFIG["PNC_PAYER_HUB"]:
        _pncp = (pop.where(F.upper(F.coalesce(F.col("customer_name"), F.lit(""))).rlike(CONFIG["PNC_NAME_REGEX"]))
                 .select(F.lit("int").alias("scope"), F.col("cust").alias("other")).distinct().withColumn("is_pnc", F.lit(1)))
        _pmo = (_pmo.join(F.broadcast(_pncp), ["scope", "other"], "left")
                .withColumn("is_hub", F.greatest(F.coalesce(F.col("is_hub"), F.lit(0)), F.coalesce(F.col("is_pnc"), F.lit(0))))
                .drop("is_pnc"))
    payer_month = cache_parquet(_pmo.withColumn("is_hub", F.coalesce(F.col("is_hub"), F.lit(0))), "payer_month")

with step("stream events with flags"):
    _flags = payer_month.select("scope", "other", "e", "pw_loss", "pw_gain", "pw_new", "is_hub", "n_risk", "n_broken", "n_grow", "n_new")
    stream_events = cache_parquet(risk.where(F.col("broke") | F.col("grew")).join(_flags, ["scope", "other", "e"], "left"),
                                  "stream_events")
    new_streams = cache_parquet(new_raw.join(_flags, ["scope", "other", "e"], "left"), "new_streams")

with step("customer events (net of losses and gains)"):
    def _s_if(cond, col):
        return F.sum(F.when(cond, F.col(col)).otherwise(0.0))
    _cr = stream_events.groupBy("cust", "e").agg(
        F.sum("lost").alias("lost"),
        *[_s_if(F.col("ev") == t, "lost").alias(f"lost_{t}") for t in LOSS_TYPES],
        _s_if(F.col("scope") == "int", "lost").alias("lost_int"),
        _s_if(F.col("pw_loss") == 1, "lost").alias("lost_pw"),
        _s_if(F.col("is_hub") == 1, "lost").alias("lost_hub"),
        F.sum("gain").alias("grow"),
        _s_if(F.col("scope") == "int", "gain").alias("grow_int"),
        _s_if(F.col("pw_gain") == 1, "gain").alias("grow_pw"),
        _s_if(F.col("is_hub") == 1, "gain").alias("grow_hub"),
        F.sum(F.col("broke").cast("int")).alias("n_broken"),
        F.sum(F.col("grew").cast("int")).alias("n_grew"))
    _cn = new_streams.groupBy("cust", "e").agg(
        F.sum("new_amt").alias("new_amt"),
        _s_if(F.col("scope") == "int", "new_amt").alias("new_int"),
        _s_if(F.col("pw_new") == 1, "new_amt").alias("new_pw"),
        _s_if(F.col("is_hub") == 1, "new_amt").alias("new_hub"),
        F.count(F.lit(1)).alias("n_new"))
    _cn1 = (new_streams.withColumn("e", F.col("e") - 1).groupBy("cust", "e")
            .agg(F.sum("new_amt").alias("new_next")))
    _num = ["lost", "grow", "new_amt", "new_next", "lost_int", "lost_pw", "lost_hub", "grow_int", "grow_pw",
            "grow_hub", "new_int", "new_pw", "new_hub", "n_broken", "n_grew", "n_new"] + [f"lost_{t}" for t in LOSS_TYPES]
    _cm = (_cr.join(_cn, ["cust", "e"], "full").join(_cn1, ["cust", "e"], "full").na.fill(0, _num)
           .join(elig.withColumnRenamed("mi", "e"), ["cust", "e"], "inner")
           .withColumn("gain_tot", F.col("grow") + F.col("new_amt"))
           .withColumn("net_loss", F.col("lost") - F.col("grow") - F.col("new_amt") - F.col("new_next"))
           .withColumn("net_gain", F.col("gain_tot") - F.col("lost"))
           .withColumn("direction", F.when(F.col("net_loss") >= TIER0 * F.col("base_in"), F.lit("loss"))
                       .when(F.col("net_gain") >= TIER0 * F.col("base_in"), F.lit("gain")))
           .where(F.col("direction").isNotNull()))
    _isl = F.col("direction") == "loss"
    _top = F.greatest(*[F.col(f"lost_{t}") for t in LOSS_TYPES])
    _ltype = F.lit("open")
    for t in reversed(LOSS_TYPES):
        _ltype = F.when(F.col(f"lost_{t}") == _top, F.lit(t)).otherwise(_ltype)
    _sh = (_cm
           .withColumn("amount", F.when(_isl, F.col("net_loss")).otherwise(F.col("net_gain")))
           .withColumn("size", F.col("amount") / F.col("base_in"))
           .withColumn("type", F.when(_isl, _ltype).when(F.col("new_amt") >= 0.5 * F.col("gain_tot"), F.lit("new payer"))
                       .otherwise(F.lit("existing payer grew")))
           .withColumn("source", F.when(F.when(_isl, F.col("lost_int") >= 0.5 * F.col("lost"))
                                        .otherwise(F.col("grow_int") + F.col("new_int") >= 0.5 * F.col("gain_tot")),
                                        F.lit("PNC customer payer")).otherwise(F.lit("external payer")))
           .withColumn("breadth", F.when(F.when(_isl, F.col("lost_pw") >= 0.5 * F.col("lost"))
                                         .otherwise(F.col("grow_pw") + F.col("new_pw") >= 0.5 * F.col("gain_tot")),
                                         F.lit("payer-wide")).otherwise(F.lit("single-link")))
           .withColumn("hub", F.when(F.when(_isl, F.col("lost_hub") >= 0.5 * F.col("lost"))
                                     .otherwise(F.col("grow_hub") + F.col("new_hub") >= 0.5 * F.col("gain_tot")),
                                     F.lit("hub payer")).otherwise(F.lit("non-hub payer"))))
    _wc = W.partitionBy("cust").orderBy("e")
    _sh = (_sh.withColumn("prev_e", F.lag("e").over(_wc))
              .withColumn("clean", F.col("prev_e").isNull() | (F.col("e") - F.col("prev_e") > CLEAN)))
    shocks = cache_parquet(_sh, "cust_events")

    _ev = (shocks.where(F.col("clean") & F.col("e").between(EV_START, EV_END))
           .withColumn("rn", F.row_number().over(W.partitionBy("cust", "direction").orderBy("e")))
           .where(F.col("rn") == 1)
           .select("cust", "customer_pwr_id", "master", "customer_name", "naics2n", "e", "base_in", "pbin",
                   "segment", "size_band", "direction", "size", "amount", "type", "source", "breadth", "hub",
                   "lost", "gain_tot", "lost_stop", "new_amt"))
    events = _ev.toPandas()

    _st = risk.agg(F.count(F.lit(1)).alias("standing"),
                   F.sum(F.col("gross_break").cast("long")).alias("gross_breaks"),
                   F.sum(F.col("broke").cast("long")).alias("net_breaks"),
                   F.sum(F.col("grew").cast("long")).alias("grows")).toPandas().iloc[0]
    _bt = (stream_events.where(F.col("broke")).groupBy("ev").agg(F.count(F.lit(1)).alias("streams"),
                                                                  F.sum("lost").alias("dollars")).toPandas()
           .rename(columns={"ev": "type"}))
    _gt = stream_events.where(F.col("grew")).agg(F.count(F.lit(1)).alias("streams"), F.sum("gain").alias("dollars")).toPandas()
    _nt = new_streams.agg(F.count(F.lit(1)).alias("streams"), F.sum("new_amt").alias("dollars")).toPandas()
    _ndir = shocks.groupBy("direction").count().toPandas().set_index("direction")["count"]

if len(events) == 0:
    raise ValueError("No events found. Check the pair table values of dir / scope and the thresholds.")
_edges = CONFIG["TIER_EDGES"]
_tlabels = [f"{int(a * 100)}-{int(b * 100)}%" for a, b in zip(_edges[:-1], _edges[1:])] + [f"{int(_edges[-1] * 100)}%+"]
events["tier"] = pd.cut(events["size"], bins=_edges + [np.inf], labels=_tlabels, right=False).astype(str)
events["month"] = events["e"].map(mi2m)
events["evid"] = events["cust"].astype(str) + "|" + events["direction"]

_bt = _bt.set_index("type").reindex(LOSS_TYPES).fillna(0).reset_index()
_types = pd.concat([_bt.assign(kind="break"),
                    pd.DataFrame([dict(type="grow", streams=_gt.streams.iloc[0], dollars=_gt.dollars.iloc[0], kind="gain"),
                                  dict(type="new payer", streams=_nt.streams.iloc[0], dollars=_nt.dollars.iloc[0], kind="gain")])])
_types["streams"] = _types["streams"].fillna(0).astype(int)
_gb, _nb = int(_st.gross_breaks), int(_st.net_breaks)
_sum2 = pd.DataFrame([dict(
    standing_stream_months=int(_st.standing), breaks_before_netting=_gb, breaks_after_netting=_nb,
    share_removed_as_timing_shift=1 - _nb / max(_gb, 1), grows=int(_st.grows),
    family_moves_cancelled=_famn, related_stream_months_dropped=_rel_n,
    own_account_stream_months_dropped=_self_n["stream_months"], own_account_breaks_dropped=_self_n["breaks"],
    own_account_break_dollars=round(_self_n["dollars"]),
    loss_event_months=int(_ndir.get("loss", 0)), gain_event_months=int(_ndir.get("gain", 0)),
    first_loss_events=int((events.direction == "loss").sum()), first_gain_events=int((events.direction == "gain").sum()),
    event_months=f"{mi2m(EV_START)} .. {mi2m(EV_END)}")])
CELL.show(("Streams and events", _sum2), ("Stream events by type", _types[["kind", "type", "streams", "dollars"]]))
ans("Stream breaks removed as timing shifts", f"{1 - _nb / max(_gb, 1):.1%} of {_gb:,}")

## §3 · Pre-flight counts, gates and headline

**Headline**, per direction: operating businesses, non-hub payers, payer-wide, size 20%+. If fewer than 1,000
such events (or payer-wide under 10% of operating non-hub events at 20%+), the payer-wide condition is
dropped; if still under 1,000, the size floor drops to 10%. Figures f01 and f02 are saved here.

In [ ]:
CELL.start("§3 · Pre-flight counts, gates, headline")
HEAD, HEAD_LABEL = {}, {}
events["headline"] = False
for d in DIRS:
    E = events[events.direction == d]
    base = (E.segment == "operating") & (E.hub == "non-hub payer")
    big = base & (E["size"] >= CONFIG["HEADLINE_MIN"])
    pw = big & (E.breadth == "payer-wide")
    pw_share = float(pw.sum() / max(big.sum(), 1))
    if pw.sum() >= CONFIG["GATE_MIN_EVENTS"] and pw_share >= CONFIG["GATE_MIN_PAYERWIDE"]:
        sel, lab = pw, f"operating, non-hub, payer-wide, {CONFIG['HEADLINE_MIN']:.0%}+"
    elif big.sum() >= CONFIG["GATE_MIN_EVENTS"]:
        sel, lab = big, f"operating, non-hub, {CONFIG['HEADLINE_MIN']:.0%}+"
    else:
        sel, lab = base, f"operating, non-hub, {TIER0:.0%}+"
    gate(f"{d}: payer-wide headline has enough events", bool(sel is pw),
         f"{int(pw.sum()):,} payer-wide ({pw_share:.1%} of {int(big.sum()):,} operating non-hub at {CONFIG['HEADLINE_MIN']:.0%}+)",
         f"headline = {lab}")
    events.loc[E.index[sel.to_numpy()], "headline"] = True
    HEAD_LABEL[d] = lab
    ans(f"Headline group, {d}", f"{lab}: {int(sel.sum()):,} events")
    _mc = E.groupby("month").size()
    gate(f"{d}: events spread over months", bool(len(_mc) and _mc.max() <= 3 * _mc.median()),
         f"largest month {int(_mc.max()) if len(_mc) else 0:,}, median {float(_mc.median()) if len(_mc) else 0:,.0f}",
         "a month far above the rest points to a data or definition artefact")

_by_month = (events.pivot_table(index="month", columns=["direction", "type"], values="cust", aggfunc="count", fill_value=0))
_by_month.columns = [f"{a}: {b}" for a, b in _by_month.columns]
_by_month = _by_month.reset_index()
_tier_tab = (events.pivot_table(index=["direction", "tier"], columns="breadth", values="cust", aggfunc="count", fill_value=0)
             .reset_index())
_cuts = pd.concat([events.groupby(["direction", c]).agg(events=("cust", "count"), headline=("headline", "sum"),
                                                        median_size=("size", "median"), dollars=("amount", "sum"))
                   .reset_index().rename(columns={c: "value"}).assign(cut=c)
                   for c in ["type", "source", "breadth", "hub", "segment", "size_band"]])
_cuts = _cuts[["direction", "cut", "value", "events", "headline", "median_size", "dollars"]]

# f01: events per month by type, one panel per direction
fig, axes = plt.subplots(1, 2, figsize=(14, 4.4), sharey=False)
for ax, d in zip(axes, DIRS):
    E = events[events.direction == d]
    tps = [t for t in (LOSS_TYPES if d == "loss" else GAIN_TYPES) if t in set(E["type"])]
    pv = E.pivot_table(index="month", columns="type", values="cust", aggfunc="count", fill_value=0).reindex(columns=tps).fillna(0)
    bottom = np.zeros(len(pv))
    for t in tps:
        ax.bar(pv.index, pv[t], bottom=bottom, color=TYPE_COLORS[t], label=t, width=0.78, edgecolor=SURF, linewidth=1.2)
        bottom += pv[t].to_numpy()
    ax.set_title(f"{d.capitalize()} events per month (first per customer)", pad=26)
    ax.set_ylabel("customers"); ax.grid(axis="x", visible=False); ax.tick_params(axis="x", rotation=60)
    ax.legend(ncol=len(tps), loc="lower left", bbox_to_anchor=(0, 1.0), fontsize=9)
fig.tight_layout()
savefig(fig, "f01_events_by_month", "3", "Events per month by type, losses and gains")

# f02: event size by direction
fig, ax = plt.subplots(figsize=(8.5, 4.2))
_bins = np.linspace(TIER0, 1.5, 29)
for d in DIRS:
    ax.hist(events.loc[events.direction == d, "size"].clip(upper=1.5), bins=_bins, histtype="step",
            color=C_DIR[d], linewidth=2, label=d)
for _x in _edges[1:]:
    ax.axvline(_x, color=INK2, linewidth=1, linestyle=":")
ax.set_title("Event size: net dollars / baseline monthly inflow (capped at 150%)")
ax.set_xlabel("event size"); ax.set_ylabel("customers"); ax.legend(loc="lower right", bbox_to_anchor=(1, 1), ncol=2)
ax.xaxis.set_major_formatter(matplotlib.ticker.PercentFormatter(1.0, decimals=0))
savefig(fig, "f02_event_size", "3", "Distribution of event size by direction; dotted lines are the tier edges")

CELL.show(("Gates", gates_frame()), ("Events by tier and breadth", _tier_tab), ("Events by cut", _cuts),
          ("Events by month and type", _by_month))
RUN_FULL = CONFIG["RUN_STAGE"] == "full"
if not RUN_FULL:
    CELL.log("RUN_STAGE = preflight: stopping here. Set RUN_STAGE = 'full' to continue (work files are reused).")

## §4 · Matched controls and the event panel

Events are sampled per stratum (direction x headline / other payer-wide / single-link), up to 5,000 each,
by a stable hash. Up to 3 controls per event: same month, NAICS2 and baseline-inflow decile, nearest on
baseline inflow, a different master, no event of either direction in the 6 months before or in the event
month. Controls are free to have events later, like any customer. Months past the end of data stay empty;
months inside the data with no metrics row count as zero activity.

In [ ]:
if RUN_FULL:
    CELL.start("§4 · Matched controls and the event panel")
    PRE, POST, NC = CONFIG["PRE_MONTHS"], CONFIG["POST_MONTHS"], CONFIG["N_CONTROLS"]
    events["stratum"] = events["direction"] + ": " + np.where(
        events["headline"], "headline", np.where(events["breadth"] == "payer-wide", "other payer-wide", "single-link"))
    events["kind"] = "regular"

    with step("main payer of each event and how many events it shares"):
        _lp = (stream_events.where(F.col("broke")).select("cust", "e", "scope", "other", F.col("lost").alias("amt"), "expected")
               .withColumn("direction", F.lit("loss")))
        _gp = (stream_events.where(F.col("grew")).select("cust", "e", "scope", "other", F.col("gain").alias("amt"), "expected")
               .unionByName(new_streams.select("cust", "e", "scope", "other", F.col("new_amt").alias("amt"),
                                               F.lit(0.0).alias("expected")))
               .withColumn("direction", F.lit("gain")))
        _evk = spark.createDataFrame(events[["cust", "e", "direction"]].astype({"e": "int32"}))
        _mp = (_lp.unionByName(_gp).join(F.broadcast(_evk), ["cust", "e", "direction"], "inner")
               .withColumn("rn", F.row_number().over(W.partitionBy("cust", "e", "direction").orderBy(F.desc("amt"), "other")))
               .where(F.col("rn") == 1)
               .select("cust", "e", "direction", F.concat(F.col("scope"), F.lit(":"), F.col("other")).alias("payer"),
                       F.col("expected").alias("main_exp"))
               .toPandas())
        events = events.drop(columns=[c for c in ["payer", "main_exp"] if c in events]).merge(_mp, on=["cust", "e", "direction"], how="left")
        events["payer"] = events["payer"].fillna("cust:" + events["cust"].astype(str))
        events["payer_events"] = events.groupby(["payer", "e", "direction"])["evid"].transform("count")
        events["w_payer"] = 1.0 / events["payer_events"]
        _pnc_ids = set(pop.where(F.upper(F.coalesce(F.col("customer_name"), F.lit(""))).rlike(CONFIG["PNC_NAME_REGEX"]))
                       .select("cust").distinct().toPandas().cust.astype(str))
        events["pnc_entity_payer"] = events["payer"].map(lambda p_: p_.startswith("int:") and p_[4:] in _pnc_ids)

    with step("events on all inflow (second definition)"):
        T_, P_ = CONFIG["ALL_IN_DROP"], CONFIG["ALL_IN_PREV"]
        _ai = (elig.select("cust", "customer_pwr_id", "master", "customer_name", "naics2n", "mi", "base_in", "pbin",
                           "segment", "size_band")
               .join(pop.select("cust", "mi", "amt_all_in"), ["cust", "mi"], "inner")
               .join(pop.select("cust", (F.col("mi") - 1).alias("mi"), F.col("amt_all_in").alias("in_next")), ["cust", "mi"], "left")
               .join(pop.select("cust", (F.col("mi") + 1).alias("mi"), F.col("amt_all_in").alias("in_prev")), ["cust", "mi"], "left")
               .na.fill(0.0, ["in_next", "in_prev"])
               .withColumn("avg2", (F.col("amt_all_in") + F.col("in_next")) / 2.0)
               .withColumn("direction", F.when((F.col("avg2") <= (1 - T_) * F.col("base_in")) & (F.col("in_prev") >= P_ * F.col("base_in")), F.lit("loss"))
                           .when((F.col("avg2") >= (1 + T_) * F.col("base_in")) & (F.col("in_prev") <= (2 - P_) * F.col("base_in")), F.lit("gain")))
               .where(F.col("direction").isNotNull() & (F.col("segment") == "operating"))
               .withColumn("prev_e", F.lag("mi").over(W.partitionBy("cust", "direction").orderBy("mi")))
               .where(F.col("prev_e").isNull() | (F.col("mi") - F.col("prev_e") > CLEAN)))
        _aim = _ai.select("cust", F.col("mi").alias("e_all"), "direction")   # every all-inflow event month (control exclusion)
        _near = (shocks.select("cust", F.col("e").alias("e_reg"), "direction"))
        _ai1 = (_ai.where(F.col("mi").between(EV_START, EV_END))
                .withColumn("rn", F.row_number().over(W.partitionBy("cust", "direction").orderBy("mi"))).where(F.col("rn") == 1)
                .join(_near, ["cust", "direction"], "left")
                .withColumn("near", (F.abs(F.col("e_reg") - F.col("mi")) <= 1).cast("int"))
                .groupBy("cust", "customer_pwr_id", "master", "customer_name", "naics2n", "mi", "base_in", "pbin", "segment",
                         "size_band", "direction", "avg2")
                .agg(F.max(F.coalesce(F.col("near"), F.lit(0))).alias("near_regular")))
        alt = _ai1.toPandas().rename(columns={"mi": "e"})
        alt["evid"] = alt["cust"].astype(str) + "|" + alt["direction"] + "|all"
        alt["kind"] = "all-inflow"
        alt["headline"] = False
        alt["stratum"] = alt["direction"] + ": all-inflow"
        alt["size"] = (alt["avg2"] / alt["base_in"] - 1).abs()
        alt["payer"] = "all:" + alt["cust"].astype(str); alt["payer_events"] = 1; alt["w_payer"] = 1.0
        for _c in ["type", "source", "breadth", "hub", "tier"]:
            alt[_c] = "all-inflow"
        _alt_n = alt.groupby(["direction", "near_regular"]).size().rename("events").reset_index()

    _allev = pd.concat([events, alt.drop(columns=["avg2"])], ignore_index=True)
    _allev["_h"] = _allev["evid"].map(lambda s: int(hashlib.md5(str(s).encode()).hexdigest()[:8], 16))
    _pe = (_allev.sort_values("_h").groupby("stratum", group_keys=False).head(CONFIG["PANEL_PER_STRATUM"])
           .reset_index(drop=True))
    _pe["near_regular"] = _pe["near_regular"].fillna(1).astype(int)
    _pe["headline"] = _pe["headline"].fillna(False).astype(bool)
    with step("matching controls"):
        _evs = spark.createDataFrame(
            _pe[["evid", "cust", "master", "e", "naics2n", "pbin", "base_in"]]
            .rename(columns={"cust": "ev_cust", "master": "ev_master", "base_in": "ev_base"})
            .astype({"e": "int32", "pbin": "int32", "ev_base": "float64"}))
        _arr = (shocks.select("cust", "e").unionByName(_aim.select("cust", F.col("e_all").alias("e")))
                .groupBy("cust").agg(F.collect_set("e").alias("shock_arr")))
        _cand = (elig.select(F.col("cust").alias("unit"), F.col("master").alias("u_master"),
                             F.col("mi").alias("e"), "naics2n", "pbin", F.col("base_in").alias("ubase"))
                 .join(_arr.withColumnRenamed("cust", "unit"), "unit", "left")
                 .withColumn("shock_arr", F.coalesce(F.col("shock_arr"), F.array().cast("array<int>"))))
        _dec = lambda c: F.ceil(F.col(c) / 10.0).cast("int")
        _m = (_evs.withColumn("dec", _dec("pbin")).drop("pbin")
              .join(_cand.withColumn("dec", _dec("pbin")).drop("pbin"), ["e", "naics2n", "dec"], "inner")
              .where((F.col("unit") != F.col("ev_cust")) & (F.col("u_master") != F.col("ev_master"))
                     & F.expr(f"NOT exists(shock_arr, x -> x - e >= -{CLEAN} AND x - e <= {CONFIG['CONTROL_CLEAN_AFTER']})"))
              .withColumn("dist", F.abs(F.log(F.col("ubase")) - F.log(F.col("ev_base"))))
              .withColumn("rn", F.row_number().over(W.partitionBy("evid").orderBy("dist", F.xxhash64("unit", "evid"))))
              .where(F.col("rn") <= NC)
              .select("evid", "e", "unit", "ubase", F.lit("C").alias("role")))
        _units = (_evs.select("evid", "e", F.col("ev_cust").alias("unit"), F.col("ev_base").alias("ubase"),
                              F.lit("S").alias("role")).unionByName(_m))
        _ukey = hashlib.md5(("|".join(sorted(_pe["evid"])) + f"{NC}{CONFIG['CONTROL_CLEAN_AFTER']}").encode()).hexdigest()[:8]
        units = cache_parquet(_units, f"units_{_ukey}")

    with step("event panel (months -6 .. +6): payment, deposit, revenue and credit outcomes"):
        _grid = (units.withColumn("t", F.explode(F.sequence(F.lit(-PRE), F.lit(POST))))
                 .withColumn("mi", F.col("e") + F.col("t"))
                 .where((F.col("mi") >= 0) & (F.col("mi") <= MAX_MI)))
        _pan = (_grid.join(pop.select(F.col("cust").alias("unit"), "mi", *[F.col(c).cast("float").alias(c) for c in OUTCOMES]),
                           ["unit", "mi"], "left")
                .join(out_reg.select(F.col("cust").alias("unit"), "mi", *[F.col(c).cast("float").alias(c) for c in REG_OUTCOMES]),
                      ["unit", "mi"], "left"))
        if SELF_OUTCOMES:
            _pan = _pan.join(selfpay.select(F.col("cust").alias("unit"), "mi", *[F.col(c).cast("float").alias(c) for c in SELF_OUTCOMES]),
                             ["unit", "mi"], "left")
        EXTRA = {}          # output column -> dict(label, kind, mult, family, subset, mask)
        if EXTRA_ACTIVE:
            _ids = (pop.where(F.col("customer_pwr_id").isNotNull())
                    .select(F.col("cust").alias("unit"), "customer_pwr_id").dropDuplicates(["unit"])
                    .join(units.select("unit").distinct(), "unit", "left_semi"))
            for _x in EXTRA_ACTIVE:
                _key = "unit" if _x["id_kind"] == "mdm_id" else "customer_pwr_id"
                _mc = F.col(_x["month_col"])
                _mstr = (F.when(_mc.cast("string").rlike(r"^\d{4}-\d{2}$"), _mc.cast("string"))
                         .otherwise(F.date_format(_mc.cast("date"), "yyyy-MM")))
                _cols = {out: m["src"] for out, m in _x["measures"].items()}
                if _x.get("mask"):
                    _cols[f"_mask_{_x['name']}"] = _x["mask"]["col"]
                _xt = (spark.table(_x["table"])
                       .select(F.col(_x["id_col"]).cast("string").alias(_key), mi_col(_mstr).alias("mi"),
                               *[F.col(src).cast("double").alias(out) for out, src in _cols.items()])
                       .where(F.col("mi").between(0, MAX_MI))
                       .groupBy(_key, "mi").agg(*[F.avg(out).cast("float").alias(out) for out in _cols]))
                if _key == "customer_pwr_id":
                    _xt = _xt.join(_ids, "customer_pwr_id", "inner").drop("customer_pwr_id")
                _pan = _pan.join(_xt, ["unit", "mi"], "left")
                for out, m in _x["measures"].items():
                    EXTRA[out] = dict(label=m["label"], kind=m["kind"], mult=m.get("mult", 1.0), family=_x["name"],
                                      subset=_x.get("subset_pos"),
                                      mask=(f"_mask_{_x['name']}", [m2mi(v) for v in _x["mask"]["months"]]) if _x.get("mask") else None)
        panel = _pan.toPandas()

    _nctl = panel[panel.role == "C"].groupby("evid")["unit"].nunique().reindex(_pe["evid"]).fillna(0).astype(int)
    _pe["n_controls"] = _nctl.to_numpy()
    _match = _pe.groupby(["stratum", "n_controls"]).size().rename("events").reset_index()
    _n_all = len(_pe)
    _pe = _pe[_pe["n_controls"] > 0].reset_index(drop=True)
    _ub = panel.drop_duplicates(["evid", "role", "unit"])
    _bal = (_ub.merge(_pe[["evid", "stratum"]], on="evid").groupby(["stratum", "role"])
            .agg(units=("unit", "nunique"), median_base_in=("ubase", "median")).reset_index()
            .pivot(index="stratum", columns="role", values=["units", "median_base_in"]))
    _bal.columns = [f"{a}_{'event' if b == 'S' else 'controls'}" for a, b in _bal.columns]
    _bal = _bal.reset_index()
    gate("events with at least one control", len(_pe) >= 0.8 * max(_n_all, 1), f"{len(_pe):,} of {_n_all:,}",
         "unmatched events are left out of the effects")
    CELL.show(("Panel by stratum (events, controls, median baseline inflow)", _bal),
              ("Panel", pd.DataFrame([dict(events=len(_pe), panel_rows=len(panel), outcome_columns=len(OUTCOMES) + len(EXTRA))])),
              ("Controls found per event", _match))

## §5 · Effects

Every unit is measured against its own baseline (months -6 to -1). The effect is the event group's average
minus its matched controls' average, month by month.

- **logpct** (dollars, counts, balances, revenue, DHE): log change from baseline, capped at x10 either way;
  group averages are converted back to % change, and the effect is % relative to controls.
- **diff** (shares, flags, days, PD): change from baseline, in the column's own units (shares in pp).
- **scaled** (net flow, balance changes): change from baseline / baseline monthly inflow, in pp.
- Credit outcomes only for customers with DHE above zero at baseline; deposit outcomes leave out customers
  whose new accounts appear in the acquisition months.
- Intervals: cluster bootstrap by master (an event family counts once).
- **Pass-through** = log change in outflow / log change in inflow at month +3: the % of an inflow change the
  customer passes on to its own payees.
- **Pre-trend check:** baseline moved to months -6..-4, effect read at -3..-1. It should sit on zero.

In [ ]:
if RUN_FULL:
    CELL.start("§5 · Effects")
    ALL_OUT = {c: dict(label=v[0], kind=v[1], mult=v[2], family="payments", subset=None, mask=None)
               for c, v in {**OUTCOMES, **REG_OUTCOMES, **SELF_OUTCOMES}.items()}
    ALL_OUT.update(EXTRA)
    TS = np.arange(-PRE, POST + 1)
    ET, B, LCAP = CONFIG["EFFECT_T"], CONFIG["BOOT_REPS"], math.log(CONFIG["LOG_CAP"])
    _meta = _pe.set_index("evid")
    _keys = panel[["evid", "role", "unit"]].drop_duplicates()
    _midx = pd.MultiIndex.from_frame(_keys)
    _ubase = panel.drop_duplicates(["evid", "role", "unit"]).set_index(["evid", "role", "unit"])["ubase"]
    _wide_cache = {}
    _pidx = pd.MultiIndex.from_frame(panel[["evid", "role", "unit", "t"]])
    _SUBSET_COLS = {o["subset"] for o in ALL_OUT.values() if o["subset"]}

    def _wide(col):
        """units x months matrix of the raw value (core payment outcomes: missing month = 0)."""
        if col in _wide_cache:
            return _wide_cache[col]
        v = panel[col].astype("float64")
        if col in OUTCOMES or col in REG_OUTCOMES or col in SELF_OUTCOMES:
            v = v.fillna(0.0)
        m = pd.Series(v.to_numpy(), index=_pidx).unstack("t").reindex(index=_midx, columns=TS)
        if col in _SUBSET_COLS:
            _wide_cache[col] = m
        return m

    def _matrix(col, base_ts):
        o = ALL_OUT[col]
        m = _wide(col) * o["mult"]
        base = m[base_ts].mean(axis=1, skipna=True)
        if o["kind"] == "logpct":
            out = np.log((m.clip(lower=0) + 1.0).div(base.clip(lower=0) + 1.0, axis=0)).clip(-LCAP, LCAP)
        elif o["kind"] == "scaled":
            out = m.sub(base, axis=0).div(_ubase.reindex(m.index), axis=0).mul(100.0).clip(-500, 500)
        else:
            out = m.sub(base, axis=0)
        if o["subset"]:
            sb = _wide(o["subset"])[base_ts].mean(axis=1, skipna=True)
            out.loc[~(sb > 0).to_numpy()] = np.nan
        if o["mask"]:
            mc, mis = o["mask"]
            mm = panel[panel["mi"].isin(mis) & (panel[mc] > 0)][["evid", "role", "unit"]].drop_duplicates()
            if len(mm):
                out.loc[out.index.isin(pd.MultiIndex.from_frame(mm))] = np.nan
        return out

    def _sc(col, base_ts):
        m = _matrix(col, base_ts)
        S = m.xs("S", level="role").droplevel("unit")
        C = m.xs("C", level="role").droplevel("unit")
        return S, C

    def _effect(S, C, ids, gkey):
        ii = S.index.intersection(pd.Index(ids))
        if len(ii) < 20:
            return None
        Cg = C[C.index.isin(ii)]
        s, c = S.loc[ii].to_numpy(float), Cg.to_numpy(float)
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            ms, mc = np.nanmean(s, axis=0), np.nanmean(c, axis=0)
        uniq, cs = np.unique(_meta.loc[ii, "master"].astype(str).to_numpy(), return_inverse=True)
        cc = np.searchsorted(uniq, _meta.loc[Cg.index, "master"].astype(str).to_numpy())
        K, T = len(uniq), s.shape[1]
        Ss, Ns, Sc, Nc = (np.zeros((K, T)) for _ in range(4))
        np.add.at(Ss, cs, np.nan_to_num(s)); np.add.at(Ns, cs, ~np.isnan(s))
        np.add.at(Sc, cc, np.nan_to_num(c)); np.add.at(Nc, cc, ~np.isnan(c))
        rng = np.random.default_rng(CONFIG["SEED"] + int(hashlib.md5(str(gkey).encode()).hexdigest()[:6], 16))
        Wt = rng.multinomial(K, np.full(K, 1.0 / K), size=B).astype(float)
        with np.errstate(all="ignore"):
            bd = (Wt @ Ss) / (Wt @ Ns) - (Wt @ Sc) / (Wt @ Nc)
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            lo, hi = np.nanpercentile(bd, 2.5, axis=0), np.nanpercentile(bd, 97.5, axis=0)
        return dict(ms=ms, mc=mc, d=ms - mc, lo=lo, hi=hi, n=np.sum(~np.isnan(s), axis=0), bd=bd)

    def _effect_w(S, C, ids, gkey, w=None, clus="master", reps=None):
        """_effect with event weights (controls carry their event's weight split over its controls) and a chosen
        cluster column of the event table (master or payer)."""
        ii = S.index.intersection(pd.Index(ids))
        if len(ii) < 20:
            return None
        Cg = C[C.index.isin(ii)]
        s, c = S.loc[ii].to_numpy(float), Cg.to_numpy(float)
        we = (pd.Series(1.0, index=ii) if w is None else w.reindex(ii).fillna(0.0)).to_numpy(float)
        nc = pd.Series(Cg.index).value_counts()
        wc = ((pd.Series(1.0, index=ii) if w is None else w.reindex(ii).fillna(0.0)).reindex(Cg.index).to_numpy(float)
              / nc.reindex(Cg.index).to_numpy(float))
        uniq, cs = np.unique(_meta.loc[ii, clus].astype(str).to_numpy(), return_inverse=True)
        cc = np.searchsorted(uniq, _meta.loc[Cg.index, clus].astype(str).to_numpy())
        K, T = len(uniq), s.shape[1]
        Ss, Ns, Sc, Nc = (np.zeros((K, T)) for _ in range(4))
        np.add.at(Ss, cs, we[:, None] * np.nan_to_num(s)); np.add.at(Ns, cs, we[:, None] * ~np.isnan(s))
        np.add.at(Sc, cc, wc[:, None] * np.nan_to_num(c)); np.add.at(Nc, cc, wc[:, None] * ~np.isnan(c))
        rng = np.random.default_rng(CONFIG["SEED"] + int(hashlib.md5(str(gkey).encode()).hexdigest()[:6], 16))
        Wt = rng.multinomial(K, np.full(K, 1.0 / K), size=reps or B).astype(float)
        with np.errstate(all="ignore"), warnings.catch_warnings():
            warnings.simplefilter("ignore")
            ms, mc = Ss.sum(0) / Ns.sum(0), Sc.sum(0) / Nc.sum(0)
            bd = (Wt @ Ss) / (Wt @ Ns) - (Wt @ Sc) / (Wt @ Nc)
            lo, hi = np.nanpercentile(bd, 2.5, axis=0), np.nanpercentile(bd, 97.5, axis=0)
        return dict(ms=ms, mc=mc, d=ms - mc, lo=lo, hi=hi, n=np.sum(~np.isnan(s), axis=0), bd=bd, clusters=K)

    def _disp(kind, v):
        return 100.0 * (np.exp(v) - 1.0) if kind == "logpct" else v

    UNIT = {"logpct": "%", "diff": "change", "scaled": "pp of inflow"}

    def _ttl(label, kind):
        return label if "(" in label else f"{label} ({UNIT[kind]})"

    GROUPS, GCUT = {}, {}
    for d in DIRS:
        E = _pe[(_pe.direction == d) & (_pe.kind == "regular")]
        _A = _pe[(_pe.direction == d) & (_pe.kind == "all-inflow")]
        GROUPS[(d, "all-inflow: all")] = _A["evid"]; GCUT[(d, "all-inflow: all")] = "definition"
        GROUPS[(d, "all-inflow: no regular-payer event")] = _A.loc[_A.near_regular == 0, "evid"]
        GCUT[(d, "all-inflow: no regular-payer event")] = "definition"
        GROUPS[(d, "headline")] = E.loc[E.headline, "evid"]; GCUT[(d, "headline")] = "headline"
        GROUPS[(d, "all in panel")] = E["evid"]; GCUT[(d, "all in panel")] = "headline"
        for _c, _pref, _cut in [("tier", "tier ", "tier"), ("type", "", "type"), ("source", "", "cut"),
                                ("breadth", "", "cut"), ("hub", "", "cut"), ("segment", "", "segment"),
                                ("size_band", "size ", "size")]:
            for _v in sorted(E[_c].unique()):
                GROUPS[(d, f"{_pref}{_v}")] = E.loc[E[_c] == _v, "evid"]; GCUT[(d, f"{_pref}{_v}")] = _cut
        # timing strata for §5b: stops outside the customer's control vs single relationships ending
        _nh = E["hub"] != "hub payer"
        GROUPS[(d, "payer-wide, non-hub")] = E.loc[(E.breadth == "payer-wide") & _nh, "evid"]; GCUT[(d, "payer-wide, non-hub")] = "timing"
        GROUPS[(d, "single-link, operating non-hub")] = E.loc[(E.breadth == "single-link") & _nh & (E.segment == "operating"), "evid"]
        GCUT[(d, "single-link, operating non-hub")] = "timing"
    TIMING_GROUPS = ["headline", "payer-wide, non-hub", "single-link, operating non-hub", "all-inflow: all"]

    _rows, _pre_rows, BOOT = [], [], {}
    with step(f"effects: {len(ALL_OUT)} outcomes x {len(GROUPS)} groups, {B} bootstrap draws"):
        for col, o in ALL_OUT.items():
            S, C = _sc(col, list(range(-PRE, 0)))
            for g, ids in GROUPS.items():
                r = _effect(S, C, ids, g)
                if r is None:
                    continue
                if col in ("amt_all_in", "amt_all_out"):
                    BOOT[(g, col)] = (r["d"], r["bd"])
                for j, t in enumerate(TS):
                    _rows.append(dict(direction=g[0], group=g[1], cut=GCUT[g], outcome=col, label=o["label"],
                                      family=o["family"], kind=o["kind"], t=int(t), n=int(r["n"][j]),
                                      event=_disp(o["kind"], r["ms"][j]), control=_disp(o["kind"], r["mc"][j]),
                                      effect=_disp(o["kind"], r["d"][j]), lo=_disp(o["kind"], r["lo"][j]),
                                      hi=_disp(o["kind"], r["hi"][j])))
            Sp, Cp = _sc(col, list(range(-PRE, -3)))
            for d in DIRS:
              for _tg in TIMING_GROUPS:
                if (d, _tg) not in GROUPS:
                    continue
                r = _effect(Sp, Cp, GROUPS[(d, _tg)], (d, "pre", _tg, col))
                if r is None:
                    continue
                for t in (-3, -2, -1):
                    j = int(np.where(TS == t)[0][0])
                    _pre_rows.append(dict(direction=d, group=_tg, outcome=col, label=o["label"], kind=o["kind"], t=t, n=int(r["n"][j]),
                                          effect=_disp(o["kind"], r["d"][j]), lo=_disp(o["kind"], r["lo"][j]),
                                          hi=_disp(o["kind"], r["hi"][j])))
    effects = pd.DataFrame(_rows)
    pretrend_all = pd.DataFrame(_pre_rows)
    pretrend = (pretrend_all[pretrend_all.group == "headline"].drop(columns="group").reset_index(drop=True)
                if len(pretrend_all) else pretrend_all)

    # pass-through: log outflow effect / log inflow effect at month +3, same bootstrap draws
    _j = int(np.where(TS == ET)[0][0])
    _pt = []
    for g in GROUPS:
        if (g, "amt_all_in") not in BOOT or (g, "amt_all_out") not in BOOT:
            continue
        din, bin_ = BOOT[(g, "amt_all_in")]; dout, bout = BOOT[(g, "amt_all_out")]
        with np.errstate(all="ignore"):
            rb = bout[:, _j] / bin_[:, _j]
        ok_in = (np.nanpercentile(bin_[:, _j], 2.5) > 0) or (np.nanpercentile(bin_[:, _j], 97.5) < 0)
        _pt.append(dict(direction=g[0], group=g[1], cut=GCUT[g], events=len(GROUPS[g]),
                        inflow_pct=_disp("logpct", din[_j]), outflow_pct=_disp("logpct", dout[_j]),
                        pass_through=(dout[_j] / din[_j]) if ok_in else np.nan,
                        lo=np.nanpercentile(rb, 2.5) if ok_in else np.nan,
                        hi=np.nanpercentile(rb, 97.5) if ok_in else np.nan))
    passthru = pd.DataFrame(_pt)

    effects.to_csv(os.path.join(OUT_DIR, "p1_effects.csv"), index=False)
    pretrend_all.to_csv(os.path.join(OUT_DIR, "p1_pretrend.csv"), index=False)
    passthru.to_csv(os.path.join(OUT_DIR, "p1_pass_through.csv"), index=False)
    _pe.drop(columns=["_h"]).to_parquet(os.path.join(OUT_DIR, "p1_events.parquet"), index=False)

    KEY = [c for c in ["amt_all_in", "amt_all_out", "ntxn_all_out", "ncpty_ext_out", "new_cpty_ext_out",
                       "rec_broken_share_out", "net_flow_all", "dep_bal_mean", "dep_bal_vs_inflow", "days_negative",
                       "tm_rev_12m", "gross_rev_12m", "pd_pct_now", "past_due_days"] if c in ALL_OUT]

    def _event_fig(cols, d, name, title):
        """Left: event customers and controls. Right: the difference with its interval. One row per outcome."""
        e = effects[(effects.direction == d) & (effects.group == "headline") & effects.outcome.isin(cols)]
        cols = [c for c in cols if c in set(e.outcome)]
        if not cols:
            return
        fig, axes = plt.subplots(len(cols), 2, figsize=(11, 3.3 * len(cols)), squeeze=False)
        for r, col in enumerate(cols):
            x = e[e.outcome == col].sort_values("t")
            a, b = axes[r]
            a.plot(x.t, x.control, color=C_CTRL, marker="o", markersize=4, label="matched controls")
            a.plot(x.t, x.event, color=C_DIR[d], marker="o", markersize=4, label=f"{d} event")
            a.axvline(0, color=INK2, linewidth=1, linestyle=":")
            a.set_title(x.label.iloc[0]); a.set_ylabel(f"change from baseline ({UNIT[x.kind.iloc[0]]})")
            a.legend(loc="best")
            b.fill_between(x.t, x.lo, x.hi, color=C_DIR[d], alpha=0.18, linewidth=0)
            b.plot(x.t, x.effect, color=C_DIR[d], marker="o", markersize=4)
            b.axhline(0, color=INK2, linewidth=1); b.axvline(0, color=INK2, linewidth=1, linestyle=":")
            b.set_title("Event minus controls (95% interval)"); b.set_ylabel(UNIT[x.kind.iloc[0]])
            for ax_ in (a, b):
                ax_.set_xticks(TS)
            if r == len(cols) - 1:
                a.set_xlabel("months from the event"); b.set_xlabel("months from the event")
        fig.suptitle(f"{title} — {d}: {HEAD_LABEL[d]}, n = {int(e.n.max()):,}", x=0.01, ha="left",
                     fontsize=12, fontweight="bold")
        fig.tight_layout(rect=(0, 0, 1, 0.97))
        savefig(fig, name, "5", f"{title}, {d} events against matched controls, months -{PRE} to +{POST}")
        return e, cols

    def _event_html(cols, d, name, title):
        try:
            import plotly.graph_objects as go
            from plotly.subplots import make_subplots
        except ImportError:
            return
        e = effects[(effects.direction == d) & (effects.group == "headline") & effects.outcome.isin(cols)]
        cols = [c for c in cols if c in set(e.outcome)]
        if not cols:
            return
        rgb = {"loss": "42,120,214", "gain": "235,104,52"}[d]
        pf = make_subplots(rows=len(cols), cols=2, subplot_titles=sum(
            [[e[e.outcome == c].label.iloc[0], "Event minus controls"] for c in cols], []))
        for r, col in enumerate(cols, start=1):
            x = e[e.outcome == col].sort_values("t")
            pf.add_trace(go.Scatter(x=x.t, y=x.control, name="matched controls", mode="lines+markers",
                                    line=dict(color=C_CTRL), showlegend=(r == 1)), row=r, col=1)
            pf.add_trace(go.Scatter(x=x.t, y=x.event, name=f"{d} event", mode="lines+markers",
                                    line=dict(color=C_DIR[d]), showlegend=(r == 1)), row=r, col=1)
            pf.add_trace(go.Scatter(x=list(x.t) + list(x.t[::-1]), y=list(x.hi) + list(x.lo[::-1]), fill="toself",
                                    fillcolor=f"rgba({rgb},0.18)", line=dict(width=0), hoverinfo="skip",
                                    showlegend=False), row=r, col=2)
            pf.add_trace(go.Scatter(x=x.t, y=x.effect, mode="lines+markers", line=dict(color=C_DIR[d]), showlegend=False,
                                    customdata=np.stack([x.lo, x.hi, x.n], axis=1),
                                    hovertemplate="month %{x}: %{y:.2f} (%{customdata[0]:.2f} to %{customdata[1]:.2f}), n=%{customdata[2]:,}<extra></extra>"),
                         row=r, col=2)
        pf.update_layout(title=f"{title} — {d}: {HEAD_LABEL[d]}", template="simple_white", hovermode="x unified",
                         height=330 * len(cols), paper_bgcolor=SURF, plot_bgcolor=SURF)
        save_plotly(pf, name, "5", f"{title}, {d}")

    def _dot_fig(d, cuts, name, title, caption, groups=None):
        sub = effects[(effects.direction == d) & (effects.t == ET) & effects.cut.isin(cuts)]
        groups = groups or [g for (dd, g) in GROUPS if dd == d and GCUT[(dd, g)] in cuts and g in set(sub.group)]
        cols = [c for c in KEY if c in set(sub.outcome)]
        if not cols or not groups:
            return
        ncol = 3; nrow = int(math.ceil(len(cols) / ncol))
        fig, axes = plt.subplots(nrow, ncol, figsize=(13.5, 0.9 + (0.36 * len(groups) + 1.4) * nrow), squeeze=False)
        for k, col in enumerate(cols):
            ax = axes[k // ncol][k % ncol]
            x = sub[sub.outcome == col].set_index("group").reindex(groups)
            y = np.arange(len(groups))[::-1]
            ax.hlines(y, x.lo, x.hi, color=C_DIR[d], linewidth=2, alpha=0.55)
            ax.plot(x.effect, y, "o", color=C_DIR[d], markersize=7, markeredgecolor=SURF, markeredgewidth=1.5)
            ax.axvline(0, color=INK2, linewidth=1)
            ax.set_yticks(y); ax.set_yticklabels(groups if k % ncol == 0 else [])
            lab = x.label.dropna().iloc[0] if x.label.notna().any() else col
            knd = x.kind.dropna().iloc[0] if x.kind.notna().any() else "diff"
            ax.set_title(_ttl(lab, knd), fontsize=9.5)
            ax.grid(axis="y", visible=False)
        for k in range(len(cols), nrow * ncol):
            axes[k // ncol][k % ncol].axis("off")
        fig.suptitle(title, x=0.01, ha="left", fontsize=12, fontweight="bold")
        fig.tight_layout(rect=(0, 0, 1, 0.96))
        savefig(fig, name, "5", caption)

    with step("figures"):
        for d in DIRS:
            _event_fig(["amt_all_in"], d, f"f03_inflow_{d}", "Incoming dollars around the event")
            _event_html(["amt_all_in"], d, f"f03_inflow_{d}", "Incoming dollars around the event")
            _event_fig(["amt_all_out", "ntxn_all_out", "net_flow_all"], d, f"f04_outflow_{d}", "Own outgoing payments around the event")
            _event_html(["amt_all_out", "ntxn_all_out", "net_flow_all"], d, f"f04_outflow_{d}", "Own outgoing payments around the event")
            _event_fig(["rec_broken_share_out", "rec_broken_amt_share_out", "new_cpty_ext_out", "lost_cpty_ext_out"], d,
                       f"f05_own_payees_{d}", "Own payees around the event")
            for fam, fn, tt in [("deposits", "f06_deposits", "Deposits around the event"),
                                ("credit", "f08_credit", "Credit around the event (customers with DHE)"),
                                ("revenue", "f09_revenue", "Revenue around the event")]:
                _fc = [c for c, o in ALL_OUT.items() if o["family"] == fam]
                if _fc:
                    _event_fig(_fc, d, f"{fn}_{d}", tt)
                    if fam == "deposits":
                        _event_html(_fc, d, f"{fn}_{d}", tt)
            _dot_fig(d, ["tier"], f"f10_dose_response_{d}",
                     f"{d.capitalize()}: effect at month +{ET} by event size (event minus controls, 95% interval)",
                     f"Dose response, {d}: effect at month +{ET} by event size",
                     groups=[f"tier {l}" for l in _tlabels if (d, f"tier {l}") in GROUPS])
            _dot_fig(d, ["type", "cut", "segment", "size"], f"f11_cuts_{d}",
                     f"{d.capitalize()}: effect at month +{ET} by type, source, breadth, hub, segment and size",
                     f"Cuts, {d}: effect at month +{ET}")
            if not pretrend.empty and (pretrend.direction == d).any():
                _p = pretrend[pretrend.direction == d]
                fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
                for ax, col in zip(axes, [c for c in ["amt_all_in", "amt_all_out", "rec_broken_share_out"] if c in set(_p.outcome)]):
                    x = _p[_p.outcome == col].sort_values("t")
                    ax.fill_between(x.t, x.lo, x.hi, color=C_DIR[d], alpha=0.18, linewidth=0)
                    ax.plot(x.t, x.effect, color=C_DIR[d], marker="o")
                    ax.axhline(0, color=INK2, linewidth=1); ax.set_xticks([-3, -2, -1])
                    ax.set_title(_ttl(x.label.iloc[0], x.kind.iloc[0]), fontsize=9.5)
                fig.suptitle(f"{d.capitalize()} pre-trend check: baseline months -6..-4, effect before the event (should sit on zero)",
                             x=0.01, ha="left", fontsize=11, fontweight="bold")
                fig.tight_layout(rect=(0, 0, 1, 0.9))
                savefig(fig, f"f15_pretrend_{d}", "5", f"Pre-trend check, {d} headline")

        # f17: pass-through by group, both directions
        _ptg = ["headline", "all in panel", "size small", "size mid", "size large", "operating", "finance & public",
                "PNC customer payer", "external payer", "payer-wide", "single-link"]
        _pp = passthru[passthru.group.isin(_ptg)]
        if not _pp.empty:
            fig, ax = plt.subplots(figsize=(9, 0.45 * len(_ptg) + 1.6))
            y0 = np.arange(len(_ptg))[::-1]
            for k, d in enumerate(DIRS):
                x = _pp[_pp.direction == d].set_index("group").reindex(_ptg)
                y = y0 + (0.16 if d == "loss" else -0.16)
                ax.hlines(y, 100 * x.lo, 100 * x.hi, color=C_DIR[d], linewidth=2, alpha=0.55)
                ax.plot(100 * x.pass_through, y, "o", color=C_DIR[d], markersize=7, markeredgecolor=SURF,
                        markeredgewidth=1.5, label=d)
            ax.axvline(0, color=INK2, linewidth=1)
            ax.set_yticks(y0); ax.set_yticklabels(_ptg); ax.grid(axis="y", visible=False)
            ax.set_xlabel("% of the inflow change passed on to own outflow (month +3)")
            ax.set_title(f"Pass-through at month +{ET}, with 95% interval")
            ax.legend(loc="lower right", bbox_to_anchor=(1, 1), ncol=2)
            savefig(fig, "f17_pass_through", "5", "Pass-through: % of an inflow change passed on to outflow, by direction and group")

    # tables
    def _fmt_eff(sub):
        return {(r.outcome): ci(r.effect, r.lo, r.hi) for r in sub.itertuples()}

    _lab = {c: _ttl(o["label"], o["kind"]) for c, o in ALL_OUT.items()}
    _t3 = pd.DataFrame({"outcome": [_lab[c] for c in ALL_OUT]})
    for d in DIRS:
        h = effects[(effects.direction == d) & (effects.group == "headline") & (effects.t == ET)].set_index("outcome")
        _t3[f"{d}: n"] = [int(h.n.get(c, 0)) if c in h.index else 0 for c in ALL_OUT]
        _t3[f"{d}: effect (95% interval)"] = [ci(h.effect[c], h.lo[c], h.hi[c]) if c in h.index else "" for c in ALL_OUT]
    _bym = (effects[(effects.group == "headline") & effects.t.isin([-1, 0, 1, 3, 6])]
            .pivot_table(index="outcome", columns=["direction", "t"], values="effect"))
    _bym.columns = [f"{a} t={b:+d}" for a, b in _bym.columns]
    _bym = _bym.reindex([c for c in ALL_OUT if c in _bym.index]).reset_index()
    _bym["outcome"] = _bym["outcome"].map(_lab)
    _dose = (effects[(effects.cut == "tier") & (effects.t == ET)]
             .pivot_table(index="outcome", columns=["direction", "group"], values="effect"))
    _dose.columns = [f"{a}: {b}" for a, b in _dose.columns]
    _dose = _dose.reindex([c for c in KEY if c in _dose.index]).reset_index()
    _dose["outcome"] = _dose["outcome"].map(_lab)
    _ptab = passthru.assign(pass_through=lambda x: x.pass_through.map(lambda v: f"{100 * v:.0f}%" if np.isfinite(v) else ""),
                            interval=lambda x: [f"{100 * a:.0f}% to {100 * b:.0f}%" if np.isfinite(a) else "inflow effect n.s."
                                                for a, b in zip(x.lo, x.hi)])
    _ptab = _ptab[_ptab.group.isin(_ptg)][["direction", "group", "events", "inflow_pct", "outflow_pct", "pass_through", "interval"]]
    _prt = pd.DataFrame({"outcome": [_lab[c] for c in ALL_OUT]})
    for d in DIRS:
        p = pretrend[(pretrend.direction == d) & (pretrend.t == -1)].set_index("outcome") if not pretrend.empty else pd.DataFrame()
        _prt[f"{d}: effect at -1"] = [ci(p.effect[c], p.lo[c], p.hi[c]) if c in p.index else "" for c in ALL_OUT]
        if "amt_all_out" in p.index:
            gate(f"{d}: pre-trend flat for outgoing dollars", bool(p.lo["amt_all_out"] <= 0 <= p.hi["amt_all_out"]),
                 ci(p.effect["amt_all_out"], p.lo["amt_all_out"], p.hi["amt_all_out"]) + " % at month -1",
                 "if it fails, read effects as event plus an earlier drift")
        if "amt_all_in" in p.index:
            gate(f"{d}: pre-trend flat for incoming dollars", bool(p.lo["amt_all_in"] <= 0 <= p.hi["amt_all_in"]),
                 ci(p.effect["amt_all_in"], p.lo["amt_all_in"], p.hi["amt_all_in"]) + " % at month -1",
                 "a spike before the event means timing shifts are still in the events")
    CELL.show((f"Headline effect at month +{ET}, event minus matched controls", _t3),
              ("Headline effect by month", _bym),
              (f"Dose response: effect at month +{ET} by event size", _dose),
              (f"Pass-through at month +{ET}: % of the inflow change passed on to outflow", _ptab),
              ("Pre-trend check at month -1 (should include zero)", _prt))
    for d in DIRS:
        h = effects[(effects.direction == d) & (effects.group == "headline") & (effects.t == ET)].set_index("outcome")
        for c in [c for c in KEY if c in h.index]:
            ans(f"{d} +{ET}: {_lab[c]}", ci(h.effect[c], h.lo[c], h.hi[c]))
        p = passthru[(passthru.direction == d) & (passthru.group == "headline")]
        if len(p) and np.isfinite(p.pass_through.iloc[0]):
            ans(f"{d}: pass-through at +{ET}", f"{100 * p.pass_through.iloc[0]:.0f}% ({100 * p.lo.iloc[0]:.0f}% to {100 * p.hi.iloc[0]:.0f}%)")

## §5b · What moves first after the shock

For the headline events of each direction, every outcome's effect path is read for three moments:

- **onset**: first month the 95% interval is clear of zero on the side the outcome finally moves, 2 months running.
  Months -3..-1 come from the pre-trend run (baseline -6..-4), so an onset before 0 means the outcome was
  already moving before the inflow broke: an early signal, or a sign the event was anticipated.
- **half**: first month the effect reaches half of its peak.
- **peak**: month of the largest effect from month 0 on, and its size.

Trailing-12-month revenue moves slowly by construction (each month adds 1/12), so it is marked "lagging"
and kept out of the ranking. Outcomes that never clear zero are listed last as "no clear move".

In [ ]:
if RUN_FULL:
    CELL.start("§5b · What moves first")
    _LAG = {c for c in ALL_OUT if c.endswith("_12m")}
    _fm = []
    for d, _tg in [(d, g) for d in DIRS for g in TIMING_GROUPS]:
        for col, o in ALL_OUT.items():
            e = effects[(effects.direction == d) & (effects.group == _tg) & (effects.outcome == col)]
            if e.empty:
                continue
            p = (pretrend_all[(pretrend_all.direction == d) & (pretrend_all.group == _tg) & (pretrend_all.outcome == col)]
                 if not pretrend_all.empty else pretrend_all)
            x = pd.concat([p[["t", "effect", "lo", "hi"]], e[e.t >= 0][["t", "effect", "lo", "hi"]]]).sort_values("t")
            r = onset_read(x.t, x.effect, x.lo, x.hi)
            _fm.append(dict(direction=d, group=_tg, outcome=_lab[col], col=col, family=o["family"],
                            n=int(e[e.t == 0].n.iloc[0]) if (e.t == 0).any() else 0,
                            onset=r["onset"], half=r["half"], peak_month=r["peak_t"], peak=r["peak"],
                            effect_at_0=float(e[e.t == 0].effect.iloc[0]) if (e.t == 0).any() else np.nan,
                            reading=("lagging by construction" if col in _LAG else
                                     ("moves before the event" if r["clear"] and r["onset"] < 0 else
                                      ("moves" if r["clear"] else "no clear move")))))
    first_movers = pd.DataFrame(_fm)
    if not first_movers.empty:
        first_movers["_rank"] = first_movers.reading.map({"moves before the event": 0, "moves": 1,
                                                          "no clear move": 2, "lagging by construction": 3})
        first_movers = (first_movers.sort_values(["direction", "group", "_rank", "onset", "half", "peak_month"])
                        .drop(columns="_rank").reset_index(drop=True))
        fm_all = first_movers
        first_movers = fm_all[fm_all.group == "headline"].drop(columns="group").reset_index(drop=True)
        fm_all.to_csv(os.path.join(OUT_DIR, "p1_first_movers.csv"), index=False)

        # f18: share of its own peak reached, month by month; ringed cells = interval clear of zero
        from matplotlib.colors import LinearSegmentedColormap
        _cmap = LinearSegmentedColormap.from_list("div", ["#eb6834", "#ecebe7", "#2a78d6"])
        fig, axes = plt.subplots(1, 2, figsize=(15, 0.36 * max(len(ALL_OUT), 6) + 2.2), squeeze=False)
        for ax, d in zip(axes[0], DIRS):
            fmd = first_movers[(first_movers.direction == d) & (first_movers.reading != "lagging by construction")]
            cols = list(fmd.col)
            if not cols:
                ax.axis("off"); continue
            tt = np.arange(-3, POST + 1)
            M = np.full((len(cols), len(tt)), np.nan); Sg = np.zeros_like(M, dtype=bool)
            for i, c in enumerate(cols):
                e = effects[(effects.direction == d) & (effects.group == "headline") & (effects.outcome == c) & (effects.t >= 0)]
                p = pretrend[(pretrend.direction == d) & (pretrend.outcome == c)] if not pretrend.empty else pretrend
                x = pd.concat([p[["t", "effect", "lo", "hi"]], e[["t", "effect", "lo", "hi"]]]).set_index("t").reindex(tt)
                pk = fmd[fmd.col == c].peak.iloc[0]
                if np.isfinite(pk) and pk != 0:
                    M[i] = x.effect.to_numpy() / pk          # 1 = at its peak, negative = moving against it
                Sg[i] = ((x.lo > 0) | (x.hi < 0)).to_numpy()
            ax.imshow(np.clip(M, -1, 1), cmap=_cmap, vmin=-1, vmax=1, aspect="auto")
            yy, xx = np.nonzero(Sg)
            ax.scatter(xx, yy, s=14, facecolors="none", edgecolors=INK, linewidths=0.8)
            ax.set_xticks(range(len(tt))); ax.set_xticklabels([f"{v:+d}" for v in tt])
            ax.set_yticks(range(len(cols)))
            ax.set_yticklabels([f"{r.outcome}  [{'-' if not np.isfinite(r.onset) else f'{int(r.onset):+d}'}]" for r in fmd.itertuples()],
                               fontsize=8.5)
            ax.axvline(2.5, color=INK, linewidth=1.2)
            ax.grid(False)
            ax.set_title(f"{d.capitalize()}: share of each outcome's peak reached (onset month in brackets)", fontsize=10)
            ax.set_xlabel("months from the event (ringed = 95% interval clear of zero)")
        fig.tight_layout()
        savefig(fig, "f18_what_moves_first", "5b",
                "What moves first: each outcome's effect as a share of its own peak (blue = towards the peak, orange = "
                "against), months -3..+6, ringed where the interval is clear of zero; sorted by onset")

    _show = first_movers.assign(
        onset=lambda x: x.onset.map(lambda v: "" if not np.isfinite(v) else f"{int(v):+d}"),
        half=lambda x: x.half.map(lambda v: "" if not np.isfinite(v) else f"{int(v):+d}"),
        peak_month=lambda x: x.peak_month.map(lambda v: "" if not np.isfinite(v) else f"{int(v):+d}"))
    _blocks = [(f"{d.capitalize()} headline ({HEAD_LABEL[d]}): order in which outcomes move",
                _show[_show.direction == d][["outcome", "family", "reading", "onset", "half", "peak_month", "peak", "effect_at_0", "n"]])
               for d in DIRS]
    # the same reading side by side: headline, payer-wide stops (outside the customer's control), single links
    _f = lambda v: "" if not np.isfinite(v) else f"{int(v):+d}"
    _cmp = fm_all.assign(cell=lambda x: np.where(x.reading.isin(["moves", "moves before the event"]),
                                                 x.onset.map(_f), np.where(x.reading == "no clear move", "-", "lag")))
    for d in DIRS:
        c = (_cmp[_cmp.direction == d].pivot_table(index="outcome", columns="group", values="cell", aggfunc="first")
             .reindex(columns=[g for g in TIMING_GROUPS if g in set(_cmp[_cmp.direction == d].group)]))
        _n = fm_all[(fm_all.direction == d)].groupby("group").n.max().reindex(c.columns)
        c.columns = [f"{g} (n={int(v):,})" if np.isfinite(v) else g for g, v in zip(c.columns, _n)]
        _ord = first_movers[first_movers.direction == d].outcome.tolist()
        _blocks.append((f"{d.capitalize()}: onset month by stratum (- = no clear move, lag = lagging by construction)",
                        c.reindex([o for o in _ord if o in c.index]).reset_index()))
    CELL.show(*_blocks)
    for d in DIRS:
        for _tg in TIMING_GROUPS[1:]:
            x = fm_all[(fm_all.direction == d) & (fm_all.group == _tg) & (fm_all.reading == "moves before the event")]
            ans(f"{d}, {_tg}: outcomes moving before the event", ", ".join(f"{r.outcome} ({int(r.onset):+d})" for r in x.itertuples()) or "none")
    for d in DIRS:
        fmd = first_movers[(first_movers.direction == d) & first_movers.reading.isin(["moves before the event", "moves"])]
        if len(fmd):
            first_t = fmd.onset.min()
            ans(f"{d}: first outcome(s) to move", f"{', '.join(fmd[fmd.onset == first_t].outcome)} (month {int(first_t):+d})")
            ans(f"{d}: outcomes that move, in order", " > ".join(f"{r.outcome} ({int(r.onset):+d})" for r in fmd.itertuples()))

## §5c · Who absorbs and who spreads

The headline events of each direction split by the customer's **sector** (NAICS2) and by its **traits over the
6 months before** (thirds within the headline events): number of payers and payees, concentration of payers and
payees (HHI), external share of inflow and outflow, balance buffer (average balance / average monthly outflow),
and whether it sits in a family of companies (master power id with 2+ customers).

For each group at month +3: the inflow effect, how much of it is passed on to outflow (**pass-through**, the
spreading side), the balance effect (the **absorbing** side) and the change in standing payees not paid (the
customer's own regular payees left unpaid, i.e. what the next customers feel). Groups under 150 events are left
out. Read differences between groups as associations: sectors and traits travel together.

In [ ]:
if RUN_FULL:
    CELL.start("§5c · Who absorbs and who spreads")
    NAICS2_NAME = {"11": "agriculture", "21": "mining, oil & gas", "22": "utilities", "23": "construction",
                   "31": "manufacturing", "32": "manufacturing", "33": "manufacturing", "42": "wholesale",
                   "44": "retail", "45": "retail", "48": "transport & warehousing", "49": "transport & warehousing",
                   "51": "information", "52": "finance & insurance", "53": "real estate", "54": "professional services",
                   "55": "management of companies", "56": "admin & support", "61": "education", "62": "health care",
                   "71": "arts & recreation", "72": "accommodation & food", "81": "other services", "92": "public admin"}
    _TR = CONFIG["SUSC_TRAITS"]
    with step("baseline traits of headline event customers"):
        _hd = _pe[_pe.headline]
        _mc = set(spark.table(CONFIG["MET_TABLE"]).columns)
        _tcols = [c for c in _TR if c in _mc]
        _ev = spark.createDataFrame(_hd[["evid", "cust", "e"]].astype({"e": "int32"}))
        _tr = (spark.table(CONFIG["MET_TABLE"]).where(F.col("month").between(MONTH_MIN, MONTH_MAX))
               .select(F.col("mdm_id").cast("string").alias("cust"), mi_col("month").alias("mi"),
                       *[F.col(c).cast("double").alias(c) for c in _tcols])
               .join(F.broadcast(_ev), "cust")
               .where(F.col("mi").between(F.col("e") - BM, F.col("e") - 1))
               .groupBy("evid").agg(*[F.avg(c).alias(c) for c in _tcols]).toPandas().set_index("evid"))
        # balance buffer from the panel: baseline average balance / baseline monthly outflow
        if "dep_bal_mean" in panel.columns:
            _bp = panel[(panel.role == "S") & (panel.t < 0)].groupby("evid")[["dep_bal_mean", "amt_all_out"]].mean()
            _tr["buffer_months"] = (_bp["dep_bal_mean"] / _bp["amt_all_out"].where(_bp["amt_all_out"] > 0)).reindex(_tr.index)
    _tr = _tr.reindex(_hd["evid"])

    SUSG = {}
    for d in DIRS:
        Ed = _hd[_hd.direction == d].set_index("evid")
        sec = Ed["naics2n"].astype(str).str[:2].map(NAICS2_NAME).fillna("other / unknown")
        for v, ix in sec.groupby(sec).groups.items():
            SUSG[(d, "sector", v)] = list(ix)
        for c, lab in _TR.items():
            if c not in _tr.columns:
                continue
            x = _tr.loc[Ed.index, c]
            if c == "family_size":
                b = pd.Series(np.where(x >= 2, "in a family (2+)", np.where(x.notna(), "on its own", None)), index=x.index)
            else:
                ok = x.notna()
                if ok.sum() < 3 * CONFIG["SUSC_MIN_EVENTS"] or x[ok].nunique() < 3:
                    continue
                q = x[ok].rank(method="first", pct=True)
                b = pd.Series(None, index=x.index, dtype=object)
                b[ok] = np.select([q <= 1 / 3, q <= 2 / 3], ["low third", "middle third"], "high third")
            for v, ix in b.dropna().groupby(b.dropna()).groups.items():
                SUSG[(d, lab, v)] = list(ix)
    SUSG = {k: v for k, v in SUSG.items() if len(v) >= CONFIG["SUSC_MIN_EVENTS"]}

    SUS_COLS = [c for c in ["amt_all_in", "amt_all_out", "dep_bal_mean", "rec_broken_share_out"] if c in ALL_OUT]
    _j = int(np.where(TS == ET)[0][0])
    _sr, _sb = {}, {}
    with step(f"effects for {len(SUSG)} sector and trait groups"):
        for col in SUS_COLS:
            o = ALL_OUT[col]
            S, C = _sc(col, list(range(-PRE, 0)))
            for g, ids in SUSG.items():
                r = _effect(S, C, ids, ("susc",) + g + (col,))
                if r is None:
                    continue
                _sr[(g, col)] = (_disp(o["kind"], r["d"][_j]), _disp(o["kind"], r["lo"][_j]), _disp(o["kind"], r["hi"][_j]))
                if col in ("amt_all_in", "amt_all_out"):
                    _sb[(g, col)] = (r["d"][_j], r["bd"][:, _j])
    _rows = []
    for g, ids in SUSG.items():
        row = dict(direction=g[0], cut=g[1], group=g[2], events=len(ids))
        for col, nm in [("amt_all_in", "inflow_pct"), ("amt_all_out", "outflow_pct"), ("dep_bal_mean", "balance_pct"),
                        ("rec_broken_share_out", "payees_unpaid_pp")]:
            if (g, col) in _sr:
                row[nm], row[nm + "_lo"], row[nm + "_hi"] = _sr[(g, col)]
        if (g, "amt_all_in") in _sb and (g, "amt_all_out") in _sb:
            din, bin_ = _sb[(g, "amt_all_in")]; dout, bout = _sb[(g, "amt_all_out")]
            ok_in = (np.nanpercentile(bin_, 2.5) > 0) or (np.nanpercentile(bin_, 97.5) < 0)
            with np.errstate(all="ignore"):
                rb = bout / bin_
            row["pass_through"] = dout / din if ok_in else np.nan
            row["pt_lo"] = np.nanpercentile(rb, 2.5) if ok_in else np.nan
            row["pt_hi"] = np.nanpercentile(rb, 97.5) if ok_in else np.nan
        _rows.append(row)
    susc = pd.DataFrame(_rows)
    if len(susc):
        susc.to_csv(os.path.join(OUT_DIR, "p1_susceptibility.csv"), index=False)

        # f22: pass-through by group with intervals, one panel per direction
        fig, axes = plt.subplots(1, len(DIRS), figsize=(14, 0.27 * susc.groupby("direction").size().max() + 2), squeeze=False)
        for ax, d in zip(axes[0], DIRS):
            x = susc[(susc.direction == d) & susc.pass_through.notna()].copy()
            x = x.sort_values(["cut", "pass_through"])
            y = np.arange(len(x))
            ax.errorbar(100 * x.pass_through, y, xerr=[100 * (x.pass_through - x.pt_lo).clip(lower=0),
                                                       100 * (x.pt_hi - x.pass_through).clip(lower=0)],
                        fmt="o", color=C_DIR[d], ecolor=C_DIR[d], elinewidth=1.2, markersize=5, capsize=0)
            ax.set_yticks(y); ax.set_yticklabels([f"{r.cut}: {r.group}" for r in x.itertuples()], fontsize=8)
            hp = passthru[(passthru.direction == d) & (passthru.group == "headline")]
            if len(hp):
                ax.axvline(100 * hp.pass_through.iloc[0], color=INK2, linewidth=1, linestyle=":")
            ax.axvline(0, color=INK2, linewidth=0.8)
            ax.set_xlim(-50, 200)
            ax.set_xlabel("% of the inflow change passed on to outflow at +3 (dotted = headline)")
            ax.set_title(f"{d.capitalize()}: pass-through by sector and trait", fontsize=10)
        fig.tight_layout()
        savefig(fig, "f22_susceptibility", "5c", "Pass-through by sector and baseline trait, headline events, month +3")

        _fmtc = lambda v, lo, hi: "" if not np.isfinite(v) else f"{v:+.1f} ({lo:+.1f} to {hi:+.1f})"
        _blocks = []
        for d in DIRS:
            x = susc[susc.direction == d].copy()
            _go = {"low third": 0, "middle third": 1, "high third": 2, "on its own": 0, "in a family (2+)": 1}
            x = x.assign(_c=x.cut.map(lambda v: 0 if v == "sector" else 1), _g=x.group.map(_go).fillna(-1),
                         _p=-x.events).sort_values(["_c", "cut", "_g", "_p"]).drop(columns=["_c", "_g", "_p"])
            x["pass-through"] = [("" if not np.isfinite(r.get("pass_through", np.nan)) else
                                  f"{100 * r['pass_through']:.0f}% ({100 * r['pt_lo']:.0f} to {100 * r['pt_hi']:.0f})")
                                 for _, r in x.iterrows()]
            for nm, lab in [("inflow_pct", "inflow %"), ("balance_pct", "balance %"), ("payees_unpaid_pp", "payees unpaid pp")]:
                if nm in x:
                    x[lab] = [_fmtc(r[nm], r[nm + "_lo"], r[nm + "_hi"]) for _, r in x.iterrows()]
            keep = ["cut", "group", "events", "inflow %", "pass-through", "balance %", "payees unpaid pp"]
            _blocks.append((f"{d.capitalize()} headline: effect at month +3 by sector and trait", x[[k for k in keep if k in x]]))
        CELL.show(*_blocks)
        for d in DIRS:
            x = susc[(susc.direction == d) & susc.pass_through.notna() & (susc.cut == "sector")]
            if len(x) >= 2:
                a, b = x.loc[x.pass_through.idxmax()], x.loc[x.pass_through.idxmin()]
                ans(f"{d}: sector passing on most / least", f"{a.group} {100 * a.pass_through:.0f}% / {b.group} {100 * b.pass_through:.0f}%")
            x = susc[(susc.direction == d) & (susc.cut == _TR.get("buffer_months", ""))]
            if len(x) == 3 and x.pass_through.notna().all():
                xs = x.set_index("group").pass_through
                ans(f"{d}: pass-through by balance buffer (low / middle / high third)",
                    " / ".join(f"{100 * xs.get(k, np.nan):.0f}%" for k in ["low third", "middle third", "high third"]))
    else:
        CELL.show(("Who absorbs and who spreads", pd.DataFrame([dict(status="no group reached the minimum size")])))

## §5d · Robustness: events that are not independent, event month, a second event definition

- **Payers:** a payer cutting or raising many PNC customers in one month creates many events. Each event's main
  payer is its largest lost (or gained) stream. Key effects are repeated with every payer-month counting once
  (weight 1 / events sharing that payer and month; controls carry their event's weight) and with intervals
  clustered by payer instead of by the customer's master.
- **Event month:** headline effects at +3 by event month, and the range when each month is left out in turn.
- **All-inflow events:** total incoming dollars at least 30% below (above) their 6-month average over months e and
  e+1, with month e-1 still near normal; operating customers. Split by whether a regular-payer event of the same
  direction happened within a month (no = driven by irregular money).
- Outcomes include the outflow split: dollars and payments to **regular payees** (paid in at least 4 of the 6
  months before) and to **other payees**.

In [ ]:
if RUN_FULL:
    CELL.start("§5d · Robustness")
    _RB_COLS = [c for c in ["amt_all_in", "amt_all_out", "amt_reg_out", "amt_irr_out", "ntxn_all_out", "ntxn_reg_out",
                            "ntxn_irr_out", "dep_bal_mean", "rec_broken_share_out"] if c in ALL_OUT]
    _RB_GROUPS = TIMING_GROUPS + ["all-inflow: no regular-payer event"]
    _VAR = [("as before: master clusters", None, "master"), ("payer-month once: payer clusters", _meta["w_payer"], "payer")]
    _jE = int(np.where(TS == ET)[0][0])
    _rb, _rbd = [], {}
    with step(f"robustness: {len(_RB_COLS)} outcomes x {len(_RB_GROUPS)} groups x 2 weightings"):
        for col in _RB_COLS:
            o = ALL_OUT[col]
            S, C = _sc(col, list(range(-PRE, 0)))
            Sp, Cp = _sc(col, list(range(-PRE, -3)))
            for d in DIRS:
                for g in _RB_GROUPS:
                    if (d, g) not in GROUPS or len(GROUPS[(d, g)]) < 20:
                        continue
                    for vname, w, cl in _VAR:
                        r = _effect_w(S, C, GROUPS[(d, g)], (d, g, col, vname), w, cl)
                        rp = _effect_w(Sp, Cp, GROUPS[(d, g)], (d, g, col, vname, "pre"), w, cl)
                        if r is None:
                            continue
                        tt = [-3, -2, -1] + list(range(0, POST + 1))
                        ix = [int(np.where(TS == t)[0][0]) for t in tt]
                        src = [rp if (t < 0 and rp is not None) else r for t in tt]
                        ee = np.array([_disp(o["kind"], q["d"][j]) if q is not None else np.nan for q, j in zip(src, ix)])
                        ll = np.array([_disp(o["kind"], q["lo"][j]) if q is not None else np.nan for q, j in zip(src, ix)])
                        hh = np.array([_disp(o["kind"], q["hi"][j]) if q is not None else np.nan for q, j in zip(src, ix)])
                        on = onset_read(tt, ee, ll, hh)
                        _rb.append(dict(direction=d, group=g, outcome=o["label"], col=col, variant=vname,
                                        events=len(GROUPS[(d, g)]), clusters=r["clusters"],
                                        effect_3=_disp(o["kind"], r["d"][_jE]), lo=_disp(o["kind"], r["lo"][_jE]),
                                        hi=_disp(o["kind"], r["hi"][_jE]),
                                        onset=on["onset"] if on["clear"] else np.nan))
                        if col in ("amt_all_in", "amt_all_out"):
                            _rbd[(d, g, vname, col)] = (r["d"][_jE], r["bd"][:, _jE])
    robust = pd.DataFrame(_rb)
    _ptr = []
    for (d, g, vname, col), (din, bin_) in _rbd.items():
        if col != "amt_all_in" or (d, g, vname, "amt_all_out") not in _rbd:
            continue
        dout, bout = _rbd[(d, g, vname, "amt_all_out")]
        ok_in = (np.nanpercentile(bin_, 2.5) > 0) or (np.nanpercentile(bin_, 97.5) < 0)
        with np.errstate(all="ignore"):
            rb = bout / bin_
        _ptr.append(dict(direction=d, group=g, variant=vname,
                         pass_through=(f"{100 * dout / din:.0f}% ({100 * np.nanpercentile(rb, 2.5):.0f} to "
                                       f"{100 * np.nanpercentile(rb, 97.5):.0f})") if ok_in else "inflow effect not clear"))
    _ptr = pd.DataFrame(_ptr)
    if len(robust):
        robust.to_csv(os.path.join(OUT_DIR, "p1_robustness.csv"), index=False)

    # how concentrated are events on payers (all regular events in the window)
    _hev = events[events.headline]
    _conc = (_hev.groupby("direction").apply(lambda x: pd.Series(dict(
        events=len(x), payers=x.payer.nunique(), payer_months=x.groupby(["payer", "e"]).ngroups,
        share_sharing_2plus=(x.payer_events >= 2).mean(), share_sharing_5plus=(x.payer_events >= 5).mean(),
        share_sharing_20plus=(x.payer_events >= 20).mean(),
        effective_events=x.w_payer.sum()))).reset_index())
    _top = (events.groupby(["direction", "month", "payer"]).agg(events=("evid", "count"),
                                                                 headline=("headline", "sum")).reset_index()
            .sort_values("events", ascending=False).head(12))
    _ints = [p.split(":", 1)[1] for p in _top.payer if p.startswith("int:")]
    if _ints:
        _nm = (pop.where(F.col("cust").isin(_ints)).select("cust", "customer_name").dropDuplicates(["cust"]).toPandas()
               .set_index("cust")["customer_name"])
        _top["name"] = _top.payer.map(lambda p: _nm.get(p.split(":", 1)[1], "") if p.startswith("int:") else "external")
    else:
        _top["name"] = "external"

    # event month stability (headline, point estimates and leave-one-month-out range)
    _STAB = [c for c in ["amt_all_in", "amt_all_out", "amt_reg_out", "amt_irr_out", "dep_bal_mean"] if c in ALL_OUT]
    _st_rows, _loo = [], []
    with step("effects by event month and leave-one-month-out"):
        for col in _STAB:
            o = ALL_OUT[col]
            S, C = _sc(col, list(range(-PRE, 0)))
            for d in DIRS:
                hd = _meta[(_meta.direction == d) & _meta.headline.astype(bool)]
                for m_, ids in hd.groupby("e").groups.items():
                    r = _effect_w(S, C, list(ids), (d, col, "month", m_), reps=50)
                    if r is not None:
                        _st_rows.append(dict(direction=d, month=mi2m(int(m_)), outcome=o["label"], col=col, events=len(ids),
                                             effect_3=_disp(o["kind"], r["d"][_jE]), lo=_disp(o["kind"], r["lo"][_jE]),
                                             hi=_disp(o["kind"], r["hi"][_jE])))
                vals = []
                for m_ in sorted(hd.e.unique()):
                    r = _effect_w(S, C, list(hd.index[hd.e != m_]), (d, col, "loo", m_), reps=2)
                    if r is not None:
                        vals.append(_disp(o["kind"], r["d"][_jE]))
                if vals:
                    _loo.append(dict(direction=d, outcome=o["label"], months=len(vals),
                                     smallest=float(np.nanmin(vals)), largest=float(np.nanmax(vals))))
    stab = pd.DataFrame(_st_rows)
    _loo = pd.DataFrame(_loo)

    if len(stab):
        fig, axes = plt.subplots(1, len(DIRS), figsize=(14, 4.6), squeeze=False)
        _cc = {"Incoming dollars": C_IN, "Outgoing dollars": C_OUT, "Average balance": C_EVENT,
               "Outgoing dollars to regular payees": "#8a5cd6", "Outgoing dollars to other payees": "#eda100"}
        for ax, d in zip(axes[0], DIRS):
            for lab, x in stab[stab.direction == d].groupby("outcome"):
                x = x.sort_values("month")
                ax.plot(x.month, x.effect_3, marker="o", markersize=4, color=_cc.get(lab, INK2), label=lab)
            ax.axhline(0, color=INK2, linewidth=1)
            ax.set_title(f"{d.capitalize()} headline: effect at +3 by event month (%)", fontsize=10)
            ax.tick_params(axis="x", rotation=60, labelsize=8)
        axes[0][0].legend(loc="lower left", bbox_to_anchor=(0, 1.1), ncol=3, fontsize=8.5)
        fig.tight_layout()
        savefig(fig, "f23_effect_by_event_month", "5d", "Headline effects at +3 by event month (stability)")

    _f = lambda v, lo, hi: "" if not np.isfinite(v) else f"{v:+.1f} ({lo:+.1f} to {hi:+.1f})"
    _rv = robust.assign(cell=[_f(r.effect_3, r.lo, r.hi) + ("" if not np.isfinite(r.onset) else f" [{int(r.onset):+d}]")
                              for r in robust.itertuples()]) if len(robust) else robust
    _blocks = [("How concentrated headline events are on a few payers (main payer = largest lost or gained stream)", _conc),
               ("Payer-months with the most events (all event types)", _top),
               ("All-inflow events in the window: with or without a regular-payer event within a month", _alt_n)]
    for d in DIRS:
        if len(_rv):
            pv = (_rv[_rv.direction == d].pivot_table(index=["group", "outcome"], columns="variant", values="cell", aggfunc="first")
                  .reset_index())
            _blocks.append((f"{d.capitalize()}: effect at +3 (95% interval) [onset month]: as before vs each payer-month once", pv))
    _blocks += [("Pass-through at +3 by weighting", _ptr.pivot_table(index=["direction", "group"], columns="variant",
                                                                       values="pass_through", aggfunc="first").reset_index()
                 if len(_ptr) else _ptr),
                ("Headline effect at +3: range when each event month is left out", _loo)]
    CELL.show(*_blocks)
    for d in DIRS:
        x = _ptr[(_ptr.direction == d) & (_ptr.group == "headline")] if len(_ptr) else _ptr
        if len(x):
            ans(f"{d}: pass-through, as before vs each payer-month once (payer clusters)",
                " / ".join(x.sort_values("variant").pass_through))
        for col in ["amt_reg_out", "amt_irr_out"]:
            x = robust[(robust.direction == d) & (robust.group == "headline") & (robust.col == col)
                       & (robust.variant.str.startswith("as before"))] if len(robust) else robust
            if len(x):
                r = x.iloc[0]
                ans(f"{d} +3: {r.outcome}", _f(r.effect_3, r.lo, r.hi) + ("" if not np.isfinite(r.onset) else f", onset {int(r.onset):+d}"))
        x = robust[(robust.direction == d) & (robust.group == "payer-wide, non-hub") & (robust.col == "ntxn_all_out")
                   & robust.variant.str.startswith("payer")] if len(robust) else robust
        if len(x):
            ans(f"{d}, payer-wide non-hub with payer clusters: outgoing payment count onset",
                "no clear move" if not np.isfinite(x.onset.iloc[0]) else f"{int(x.onset.iloc[0]):+d}")

## §5e · What makes a shock spread

Rather than more event types, one regression per direction and outcome on the regular-payer events of operating
customers in the panel (headline, payer-wide and single-link strata together):

**response = shock × (b + Σ c·trait)**  (proportional: no shock, no response; traits change the scale)

- **response**: the customer's change from its own 6 months before at +3, minus the average change of its
  matched controls (log for dollars and balances, pp for shares).
- **shock**: the event as the payers delivered it, log(1 − size) for losses and log(1 + size) for gains, where
  size = net lost (gained) regular-payer dollars / usual monthly inflow. Using the event itself rather than the
  customer's realised inflow keeps the customer's own reaction out of the right-hand side.
- **traits** (percentile rank within the direction, so the low end = 0.1, high end = 0.9; yes/no traits 0 or 1):
  event size, main payer's share of usual inflow, share of the event from payers gone (stops; new payers for gains),
  regular share of inflow, number of payers, balance buffer (months of outflow), money in from own accounts
  elsewhere (share of inflow), money out to own accounts elsewhere (share of outflow), payer-wide, payer is a PNC
  customer. Missing traits sit at the middle.
- Reported as the response to **a 10% income loss (gain)** at the low and high end of each trait, others at average,
  weighted so every payer-month counts once, 95% interval from a bootstrap over main payers.
- Read the differences, not the levels: an individual response is noisy, and traits travel together.

In [ ]:
if RUN_FULL:
    CELL.start("§5e · What makes a shock spread")
    _RG = _pe[(_pe.kind == "regular") & (_pe.segment == "operating")].copy().set_index("evid")
    with step("event and customer traits"):
        _keys = spark.createDataFrame(_RG.reset_index()[["evid", "cust", "e"]].astype({"e": "int32"}))
        _rs = (risk.groupBy("cust", "e").agg(F.sum("expected").alias("reg_exp"))
               .join(F.broadcast(_keys), ["cust", "e"], "inner").select("evid", "reg_exp").toPandas().set_index("evid")["reg_exp"])
        _mcx = set(spark.table(CONFIG["MET_TABLE"]).columns)
        _np = pd.Series(np.nan, index=_RG.index)
        if "ncpty_all_in" in _mcx:
            _np = (spark.table(CONFIG["MET_TABLE"]).where(F.col("month").between(MONTH_MIN, MONTH_MAX))
                   .select(F.col("mdm_id").cast("string").alias("cust"), mi_col("month").alias("mi"),
                           F.col("ncpty_all_in").cast("double").alias("v"))
                   .join(F.broadcast(_keys), "cust").where(F.col("mi").between(F.col("e") - BM, F.col("e") - 1))
                   .groupBy("evid").agg(F.avg("v").alias("v")).toPandas().set_index("evid")["v"])
        _bs = panel[(panel.role == "S") & (panel.t < 0)].groupby("evid").mean(numeric_only=True)
        T = pd.DataFrame(index=_RG.index)
        T["size"] = _RG["size"]
        T["main_share"] = (_RG["main_exp"] / _RG["base_in"]).clip(upper=2)
        _isl = _RG.direction == "loss"
        T["gone_share"] = np.where(_isl, _RG["lost_stop"] / _RG["lost"].where(_RG["lost"] > 0),
                                   _RG["new_amt"] / _RG["gain_tot"].where(_RG["gain_tot"] > 0))
        T["reg_share"] = (_rs.reindex(_RG.index) / _RG["base_in"]).clip(upper=1.5)
        T["n_payers"] = np.log1p(_np.reindex(_RG.index))
        if "dep_bal_mean" in _bs:
            T["buffer"] = (_bs["dep_bal_mean"] / _bs["amt_all_out"].where(_bs["amt_all_out"] > 0)).reindex(_RG.index)
        if "selfpay_amt_in" in _bs:
            T["self_in"] = (_bs["selfpay_amt_in"] / _bs["amt_all_in"].where(_bs["amt_all_in"] > 0)).reindex(_RG.index)
            T["self_out"] = (_bs["selfpay_amt_out"] / _bs["amt_all_out"].where(_bs["amt_all_out"] > 0)).reindex(_RG.index)
        T["payer_wide"] = (_RG["breadth"] == "payer-wide").astype(float)
        T["pnc_payer"] = (_RG["source"] == "PNC customer payer").astype(float)
    TRAIT_LAB = {"size": "event size (share of usual inflow)", "main_share": "main payer's share of usual inflow",
                 "gone_share": "share from payers gone (losses) / new payers (gains)", "reg_share": "regular share of inflow",
                 "n_payers": "number of payers", "buffer": "balance buffer (months of outflow)",
                 "self_in": "money in from own accounts elsewhere", "self_out": "money out to own accounts elsewhere",
                 "payer_wide": "payer-wide event (yes vs no)", "pnc_payer": "payer is a PNC customer (yes vs no)"}
    BIN = {"payer_wide", "pnc_payer"}
    TRAIT_LAB_DIR = {("loss", "gone_share"): "share of the loss from payers that stopped",
                     ("gain", "gone_share"): "share of the gain from new payers",
                     ("loss", "size"): "loss size (share of usual inflow)", ("gain", "size"): "gain size (share of usual inflow)"}
    _tl = lambda d, k: TRAIT_LAB_DIR.get((d, k), TRAIT_LAB[k])
    REG_MODELS = {}

    def _indiv(col):
        S, C = _sc(col, list(range(-PRE, 0)))
        sv = S[ET]
        cv = C[ET].groupby(level=0).mean()
        return (sv - cv.reindex(sv.index)).reindex(_RG.index)

    REG_Y = [c for c in ["amt_all_in", "amt_all_out", "amt_reg_out", "amt_irr_out", "dep_bal_mean", "rec_broken_share_out",
                         "selfpay_amt_in", "selfpay_amt_out"] if c in ALL_OUT]
    _rows, _base, _coef_store = [], [], {}
    with step(f"regressions: {len(REG_Y)} outcomes x 2 directions, {CONFIG['REG_REPS']} bootstrap draws"):
        Y = {c: _indiv(c) for c in REG_Y}
        for d in DIRS:
            ix = _RG.index[_RG.direction == d]
            if len(ix) < 200:
                continue
            sz = _RG.loc[ix, "size"].clip(upper=0.95 if d == "loss" else 10)
            shock = np.log(1 - sz) if d == "loss" else np.log(1 + sz)
            Z, zq = {}, {}
            for k in T.columns:
                v = T.loc[ix, k]
                if k in BIN:
                    if v.nunique() < 2 or min(v.sum(), (1 - v).sum()) < 50:
                        continue
                    Z[k] = v.fillna(0).to_numpy(float); zq[k] = (0.0, 1.0)
                else:
                    if v.notna().sum() < 0.5 * len(v) or v.nunique() < 5:
                        continue
                    Z[k] = v.rank(pct=True).fillna(0.5).to_numpy(float); zq[k] = (0.1, 0.9)
            ks = list(Z)
            zm = {k: Z[k].mean() for k in ks}
            Zc = np.column_stack([Z[k] - zm[k] for k in ks]) if ks else np.zeros((len(ix), 0))
            sh = shock.to_numpy(float)
            X = np.column_stack([sh, sh[:, None] * Zc])          # proportional: no shock, no response
            w = _RG.loc[ix, "w_payer"].fillna(1.0).to_numpy(float)
            uniq, cl = np.unique(_RG.loc[ix, "payer"].astype(str).to_numpy(), return_inverse=True)
            rng = np.random.default_rng(CONFIG["SEED"] + (7 if d == "loss" else 11))
            Wb = rng.multinomial(len(uniq), np.full(len(uniq), 1.0 / len(uniq)), size=CONFIG["REG_REPS"]).astype(float)
            u10 = math.log(0.9) if d == "loss" else math.log(1.1)
            for col in REG_Y:
                y = Y[col].reindex(ix).to_numpy(float)
                ok = np.isfinite(y) & np.isfinite(sh)
                if ok.sum() < 200:
                    continue
                Xo, yo, wo, co = X[ok], y[ok], w[ok], cl[ok]

                def _fit(rw):
                    A = Xo.T @ (rw[:, None] * Xo); bb = Xo.T @ (rw * yo)
                    try:
                        return np.linalg.solve(A + 1e-9 * np.eye(A.shape[0]), bb)
                    except np.linalg.LinAlgError:
                        return np.full(Xo.shape[1], np.nan)
                beta = _fit(wo)
                REG_MODELS[(d, col)] = dict(beta=beta, ks=list(ks), zm=dict(zm), kind=ALL_OUT[col]["kind"],
                                            ref={k: np.sort(T.loc[ix, k].dropna().to_numpy(float)) for k in ks if k not in BIN})
                boots = np.array([_fit(wo * Wb[r][co]) for r in range(CONFIG["REG_REPS"])])
                kind = ALL_OUT[col]["kind"]
                conv = (lambda v: 100 * (np.exp(v) - 1)) if kind == "logpct" else (lambda v: v)

                def _resp(bv, k=None, q=None):
                    """response to a 10% shock with trait k at value q (others at average)"""
                    slope = bv[..., 0] + (0 if k is None else bv[..., 1 + ks.index(k)] * (q - zm[k]))
                    return conv(slope * u10)
                r0, b0 = _resp(beta), _resp(boots)
                _base.append(dict(direction=d, outcome=ALL_OUT[col]["label"], col=col, events=int(ok.sum()),
                                  per_10pct=r0, lo=np.nanpercentile(b0, 2.5), hi=np.nanpercentile(b0, 97.5)))
                for k in ks:
                    qlo, qhi = zq[k]
                    a, ab = _resp(beta, k, qlo), _resp(boots, k, qlo)
                    c, cb = _resp(beta, k, qhi), _resp(boots, k, qhi)
                    dlo, dhi = np.nanpercentile(cb - ab, 2.5), np.nanpercentile(cb - ab, 97.5)
                    _rows.append(dict(direction=d, outcome=ALL_OUT[col]["label"], col=col, trait=_tl(d, k), key=k,
                                      low=a, low_lo=np.nanpercentile(ab, 2.5), low_hi=np.nanpercentile(ab, 97.5),
                                      high=c, high_lo=np.nanpercentile(cb, 2.5), high_hi=np.nanpercentile(cb, 97.5),
                                      diff=c - a, diff_lo=dlo, diff_hi=dhi, clear=bool(dlo > 0 or dhi < 0)))
    reg_base, reg_traits = pd.DataFrame(_base), pd.DataFrame(_rows)
    for _n, _d in [("p1_regression_base", reg_base), ("p1_regression_traits", reg_traits)]:
        _d.to_csv(os.path.join(OUT_DIR, f"{_n}.csv"), index=False)

    # f24: outflow response to a 10% shock, low vs high end of each trait (dumbbell), one panel per direction
    if len(reg_traits):
        fig, axes = plt.subplots(1, 2, figsize=(14, 5.2), squeeze=False)
        for ax, d in zip(axes[0], DIRS):
            x = reg_traits[(reg_traits.direction == d) & (reg_traits.col == "amt_all_out")].reset_index(drop=True)
            if x.empty:
                ax.axis("off"); continue
            y = np.arange(len(x))[::-1]
            for yy, r in zip(y, x.itertuples()):
                ax.plot([r.low, r.high], [yy, yy], color=INK2 if r.clear else GRID, linewidth=2, zorder=1)
                ax.errorbar(r.low, yy, xerr=[[max(r.low - r.low_lo, 0)], [max(r.low_hi - r.low, 0)]], fmt="o",
                            color="#8a8985", markersize=5, elinewidth=1, zorder=2)
                ax.errorbar(r.high, yy, xerr=[[max(r.high - r.high_lo, 0)], [max(r.high_hi - r.high, 0)]], fmt="o",
                            color=C_DIR[d], markersize=6, elinewidth=1, zorder=3)
            b = reg_base[(reg_base.direction == d) & (reg_base.col == "amt_all_out")]
            if len(b):
                ax.axvline(b.per_10pct.iloc[0], color=INK2, linewidth=1, linestyle=":")
            ax.axvline(0, color=INK2, linewidth=0.8)
            ax.set_yticks(y); ax.set_yticklabels(x.trait, fontsize=8.5)
            ax.set_xlabel(f"% change in outflow at +3 for a 10% income {'loss' if d == 'loss' else 'gain'}"
                          " (grey = low end / no, colour = high end / yes)")
            ax.set_title(f"{d.capitalize()}: how traits change what is passed on (dark line = clear difference)", fontsize=10)
        fig.tight_layout()
        savefig(fig, "f24_traits_regression", "5e", "Outflow response to a 10% income shock at the low and high end of each trait")

    _fci = lambda v, lo, hi: "" if not np.isfinite(v) else f"{v:+.2f} ({lo:+.2f} to {hi:+.2f})"
    _bt = reg_base.assign(**{"response to a 10% shock": [_fci(r.per_10pct, r.lo, r.hi) for r in reg_base.itertuples()]})
    _blocks = [("Response at +3 to a 10% income shock, traits at average (% for dollars and balance, pp for shares)",
                _bt[["direction", "outcome", "events", "response to a 10% shock"]] if len(_bt) else _bt)]
    for d in DIRS:
        for col in ["amt_all_out", "amt_reg_out", "dep_bal_mean", "rec_broken_share_out"]:
            x = reg_traits[(reg_traits.direction == d) & (reg_traits.col == col)] if len(reg_traits) else reg_traits
            if len(x):
                _blocks.append((f"{d.capitalize()}: {ALL_OUT[col]['label']} per 10% shock, low vs high end of each trait",
                                x.assign(low_end=[_fci(r.low, r.low_lo, r.low_hi) for r in x.itertuples()],
                                         high_end=[_fci(r.high, r.high_lo, r.high_hi) for r in x.itertuples()],
                                         difference=[_fci(r.diff, r.diff_lo, r.diff_hi) for r in x.itertuples()])
                                [["trait", "low_end", "high_end", "difference", "clear"]]))
    CELL.show(*_blocks)
    for d in DIRS:
        x = reg_traits[(reg_traits.direction == d) & (reg_traits.col == "amt_all_out") & reg_traits.clear] if len(reg_traits) else reg_traits
        if len(x):
            x = x.reindex(x["diff"].abs().sort_values(ascending=False).index)
            ans(f"{d}: traits that clearly change outflow response (per 10% shock, low → high end)",
                "; ".join(f"{r.trait}: {r.low:+.1f}% at the low end, {r.high:+.1f}% at the high end" for r in x.head(4).itertuples()))

## §5f · Self-payment: money between a customer's own accounts at PNC and at other banks

Self-payment = payments to or from the customer's own name at another bank (customer metrics `selfpay_amt_in/out`).
It can be a liquidity line outside PNC (money pulled in after a loss) or leakage (money sent out after a gain).

1. How common: share of TM organisations in the population with any self-payment in the last 6 months, and its share
   of their inflow and outflow dollars.
2. How it reacts: event effects for money in from and out to own accounts elsewhere, and the net (pp of monthly
   inflow), by direction and stratum.
3. Does it change absorption: headline effects for customers with and without self-payment in the 6 months before,
   and the self-payment traits in §5e.

In [ ]:
if RUN_FULL and SELF_OUTCOMES:
    CELL.start("§5f · Self-payment")
    with step("self-payment prevalence"):
        _lm = MAX_MI
        _pv = (pop.where(F.col("mi").between(_lm - BM + 1, _lm)).select("cust", "mi", "amt_all_in", "amt_all_out")
               .join(selfpay, ["cust", "mi"], "left").na.fill(0.0, ["selfpay_amt_in", "selfpay_amt_out"])
               .join(elig.where(F.col("mi") == _lm).select("cust", "segment", "size_band"), "cust", "inner")
               .groupBy("cust", "segment", "size_band").agg(*[F.sum(c).alias(c) for c in
                                                             ["amt_all_in", "amt_all_out", "selfpay_amt_in", "selfpay_amt_out"]])
               .groupBy("size_band").agg(
                   F.count(F.lit(1)).alias("customers"),
                   F.avg((F.col("selfpay_amt_in") > 0).cast("double")).alias("with_selfpay_in"),
                   F.avg((F.col("selfpay_amt_out") > 0).cast("double")).alias("with_selfpay_out"),
                   (F.sum("selfpay_amt_in") / F.sum("amt_all_in")).alias("selfpay_share_of_inflow"),
                   (F.sum("selfpay_amt_out") / F.sum("amt_all_out")).alias("selfpay_share_of_outflow"))
               .orderBy("size_band").toPandas())
    _SC = list(SELF_OUTCOMES)
    _se = effects[effects.outcome.isin(_SC) & effects.group.isin(["headline", "payer-wide, non-hub",
                                                                 "single-link, operating non-hub", "all-inflow: all"])]
    _se = _se[_se.t.isin([-3, -1, 0, 1, 3, 6])]
    _se_tab = (_se.assign(cell=lambda x: [f"{r.effect:+.1f} ({r.lo:+.1f} to {r.hi:+.1f})" for r in x.itertuples()])
               .pivot_table(index=["direction", "group", "label"], columns="t", values="cell", aggfunc="first").reset_index())

    # with vs without self-payment before the event (headline)
    _hb = panel[(panel.role == "S") & (panel.t < 0)].groupby("evid")[["selfpay_amt_in", "selfpay_amt_out"]].sum()
    _hb = _hb.reindex(_meta.index).fillna(0.0)
    _rows = []
    for d in DIRS:
        hd = _meta[(_meta.direction == d) & _meta.headline.astype(bool) & (_meta.kind == "regular")]
        has = (_hb.loc[hd.index].sum(axis=1) > 0)
        for col in [c for c in ["amt_all_in", "amt_all_out", "dep_bal_mean", "rec_broken_share_out", "selfpay_net"] if c in ALL_OUT]:
            o = ALL_OUT[col]
            S, C = _sc(col, list(range(-PRE, 0)))
            for lab, ids in [("with self-payment", hd.index[has.to_numpy()]), ("without", hd.index[~has.to_numpy()])]:
                r = _effect_w(S, C, list(ids), (d, col, "self", lab))
                if r is not None:
                    _rows.append(dict(direction=d, outcome=o["label"], group=lab, events=len(ids),
                                      effect_3=f"{_disp(o['kind'], r['d'][_jE]):+.1f} ({_disp(o['kind'], r['lo'][_jE]):+.1f} "
                                               f"to {_disp(o['kind'], r['hi'][_jE]):+.1f})"))
    _sw = pd.DataFrame(_rows)
    _swt = _sw.pivot_table(index=["direction", "outcome"], columns="group", values="effect_3", aggfunc="first").reset_index() if len(_sw) else _sw
    _swn = _sw.groupby(["direction", "group"]).events.max().reset_index() if len(_sw) else _sw
    _pvs = _pv.copy()
    for _c in ["with_selfpay_in", "with_selfpay_out", "selfpay_share_of_inflow", "selfpay_share_of_outflow"]:
        _pvs[_c] = _pvs[_c].map(lambda v: "" if pd.isna(v) else f"{v:.1%}")
    CELL.show(("TM organisations in the population, last 6 months: self-payment by size band", _pvs),
              ("Self-payment around events: effect by month from the event (% for dollars, pp for net)", _se_tab),
              ("Headline events: effect at +3 with vs without self-payment in the 6 months before", _swt),
              ("Events per group", _swn))
    for d in DIRS:
        x = effects[(effects.direction == d) & (effects.group == "headline") & (effects.outcome == "selfpay_net") & (effects.t == ET)]
        if len(x):
            r = x.iloc[0]
            ans(f"{d} +3: net money in from own accounts elsewhere (pp of monthly inflow)", f"{r.effect:+.1f} ({r.lo:+.1f} to {r.hi:+.1f})")
    if len(_pv):
        ans("TM organisations with any self-payment in (out), last 6 months",
            f"{(_pv.with_selfpay_in * _pv.customers).sum() / _pv.customers.sum():.0%} "
            f"({(_pv.with_selfpay_out * _pv.customers).sum() / _pv.customers.sum():.0%})")

## §6 · Second hop

Does it pass on? For event customers and their controls that pay other PNC customers through standing
streams: the share of those streams that break (losses) or grow (gains), month by month. Read from the
receivers' side, so this is the next customer's own event.

In [ ]:
if RUN_FULL:
    CELL.start("§6 · Second hop")
    with step("standing internal streams paid by event customers and controls"):
        _rg = _pe[_pe.kind == "regular"].copy()
        _big = (_rg.segment == "operating") & (_rg["size"] >= CONFIG["HEADLINE_MIN"])
        _rg["h2grp"] = np.select(
            [_rg.headline.astype(bool), _big & _rg.pnc_entity_payer.fillna(False).astype(bool), _big & (_rg.hub == "hub payer")],
            ["headline", "operating 20%+, PNC entity payer", "operating 20%+, other hub payer"], "other events in panel")
        _u = units.select("evid", "e", "unit", "role").join(
            spark.createDataFrame(_rg[["evid", "direction", "h2grp"]].assign(headline=_rg["headline"].astype(int))), "evid", "inner")
        _hop = (payer_month.where(F.col("scope") == "int")
                .select(F.col("other").alias("unit"), F.col("e").alias("e2"), "n_risk", "n_broken", "n_grow")
                .join(F.broadcast(_u), "unit", "inner")
                .withColumn("t", F.col("e2") - F.col("e"))
                .where(F.col("t").between(-PRE, POST))
                .groupBy("direction", "h2grp", "headline", "role", "t").agg(
                    F.countDistinct("unit").alias("payers"), F.sum("n_risk").alias("streams"),
                    F.sum("n_broken").alias("broken"), F.sum("n_grow").alias("grown"))
                .toPandas())
    _h2rows, _bymo, _h2grp_rows = [], [], []
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
    for ax, d in zip(axes, DIRS):
        hd = _hop[_hop.direction == d]
        use = hd[hd.headline == 1]
        lab = HEAD_LABEL[d]
        if use[use.role == "S"].streams.sum() < 200:
            use, lab = hd, f"all {d} events in panel"
        g = use.groupby(["role", "t"])[["payers", "streams", "broken", "grown"]].sum().reset_index()
        for _gn, _gg in hd.groupby("h2grp"):
            _g2 = _gg.groupby(["role", "t"])[["streams", "broken", "grown"]].sum().reset_index()
            _r2 = lambda role, a, b: (lambda x: float((x.broken if d == "loss" else x.grown).sum() / max(x.streams.sum(), 1)))(
                _g2[(_g2.role == role) & _g2.t.between(a, b)])
            _h2grp_rows.append(dict(direction=d, group=_gn, streams_event=int(_g2[_g2.role == "S"].streams.sum()),
                                    extra_pp=100 * ((_r2("S", 1, 3) - _r2("S", -3, -1)) - (_r2("C", 1, 3) - _r2("C", -3, -1)))))
        if g.empty or g[g.role == "S"].streams.sum() == 0:
            _h2rows.append(dict(direction=d, group="no event customer pays other PNC customers through standing streams"))
            ax.axis("off")
            continue
        g["rate"] = (g.broken if d == "loss" else g.grown) / g.streams.where(g.streams > 0)
        for role, colr, nm in [("C", C_CTRL, "payees of matched controls"), ("S", C_DIR[d], f"payees of {d}-event customers")]:
            x = g[g.role == role].sort_values("t")
            ax.plot(x.t, 100 * x.rate, color=colr, marker="o", markersize=4, label=nm)
        ax.axvline(0, color=INK2, linewidth=1, linestyle=":")
        ax.set_title(f"{d.capitalize()}: own standing streams to PNC customers that {'break' if d == 'loss' else 'grow'}")
        ax.set_xlabel("months from the event"); ax.set_ylabel("% of standing streams"); ax.set_xticks(TS); ax.legend()

        def _rate(role, lo_t, hi_t):
            x = g[(g.role == role) & g.t.between(lo_t, hi_t)]
            return float((x.broken if d == "loss" else x.grown).sum() / max(x.streams.sum(), 1))
        sb, sa, cb, ca = _rate("S", -3, -1), _rate("S", 1, 3), _rate("C", -3, -1), _rate("C", 1, 3)
        _h2rows.append(dict(direction=d, group=lab, measure="break rate" if d == "loss" else "growth rate",
                            event_before=sb, event_after=sa, controls_before=cb, controls_after=ca,
                            extra_pp=100 * ((sa - sb) - (ca - cb)),
                            streams_event=int(g[g.role == "S"].streams.sum())))
        w = g.pivot(index="t", columns="role", values="rate").rename(columns={"S": f"{d}: event", "C": f"{d}: controls"})
        _bymo.append(w)
    fig.tight_layout()
    savefig(fig, "f14_second_hop", "6", "Second hop: own standing streams to other PNC customers that break (losses) or grow (gains)")
    _h2 = pd.DataFrame(_h2rows)
    _bym2 = pd.concat(_bymo, axis=1).reset_index() if _bymo else pd.DataFrame()
    _h2g = pd.DataFrame(_h2grp_rows)
    CELL.show(("Second hop: months -3..-1 against +1..+3", _h2), ("By month", _bym2),
              ("Second hop by event group (extra pp, +1..+3 vs -3..-1, event minus controls)", _h2g))
    for r in _h2.itertuples():
        if hasattr(r, "extra_pp") and np.isfinite(getattr(r, "extra_pp", np.nan)):
            ans(f"Second hop, {r.direction}: extra {r.measure} of own streams to PNC customers, +1..+3",
                f"{r.extra_pp:+.2f} pp ({r.group})")

## §6b · Early warning: what switches on before real events

The look-back half of the spec. Four kinds of real events (**anchors**), each against matched customers without
that event (same month, NAICS2 and inflow decile, nearest inflow, another master, no such event in the 12 months
before):

| Anchor | Rule | Unit looked at |
|---|---|---|
| default | `is_default` switches on after 6 months off | the customer |
| PD doubles | `pd_pct_now` at least 2x its value 6 months before | the customer |
| deposit collapse | average balance below 50% of its 6-month average, 2 months running (6-month average 10K+) | the customer |
| payer-wide cut by a PNC customer | the customer stops at least half of its standing PNC payees (5+) | the **payer**: its signals warn everyone it pays |

**Signals** (monthly; a signal **switches on** when it fires after 6 months without firing): inflow shock (loss
event 10%+), incoming / outgoing dollars and payment count below 70% of their own 6-month average, standing payees
not paid 25%+, net outflow over 20% of inflow, average balance below 70% of its 6-month average, negative-balance
days, past due, PD up 50%, in default.

For each anchor and signal, over the 12 months before: the share of anchored customers where the signal switched
on against the share among controls (**excess**, pp), the median lead, which signal switched on **first**, and for
every pair of signals which one usually comes first (spec's lead-lag table). The signal that defines an anchor is
shown but left out of "first to fire" for that anchor.

In [ ]:
if RUN_FULL and CONFIG["RUN_EARLY_WARNING"]:
    CELL.start("§6b · Early warning")
    LB, Q = CONFIG["LOOKBACK"], CONFIG["SIG_QUIET"]
    SIGNALS = {"s_shock": "inflow shock (loss event 10%+)", "s_in": "incoming dollars down 30%",
               "s_out": "outgoing dollars down 30%", "s_n": "outgoing payment count down 30%",
               "s_brk": "standing payees not paid 25%+", "s_net": "net outflow over 20% of inflow"}
    ANCH = {"a_pay": "payer-wide cut by a PNC customer"}
    with step("monthly signals for TM organisations"):
        _b = pop.select("cust", "customer_pwr_id", "master", "naics2n", "mi", "amt_all_in", "amt_all_out",
                        "ntxn_all_out", "rec_broken_share_out", "net_flow_all")
        _dep, _dk = fam_table("deposits", ["dep_bal_mean", "days_negative"])
        _cre, _ck = fam_table("credit", ["pd_pct_now", "is_default", "past_due_days"])
        for _t, _k in [(_dep, _dk), (_cre, _ck)]:
            if _t is not None:
                _b = _b.join(_t, [_k, "mi"], "left")
        _cols = set(_b.columns)
        _w6 = W.partitionBy("cust").orderBy("mi").rangeBetween(-BM, -1)
        _wq = W.partitionBy("cust").orderBy("mi").rangeBetween(-Q, -1)
        _wl6 = W.partitionBy("cust").orderBy("mi").rangeBetween(-6, -6)
        _wn1 = W.partitionBy("cust").orderBy("mi").rangeBetween(1, 1)
        D, FLOOR = CONFIG["SIG_DROP"], CONFIG["BASE_IN_MIN"]
        _i = lambda cond, ok: F.when(ok, cond.cast("int"))
        _b = (_b.withColumn("a_in", F.avg("amt_all_in").over(_w6)).withColumn("a_out", F.avg("amt_all_out").over(_w6))
              .withColumn("a_n", F.avg("ntxn_all_out").over(_w6))
              .withColumn("s_in", _i(F.col("amt_all_in") < D * F.col("a_in"), F.col("a_in") >= FLOOR))
              .withColumn("s_out", _i(F.col("amt_all_out") < D * F.col("a_out"), F.col("a_out") >= FLOOR))
              .withColumn("s_n", _i(F.col("ntxn_all_out") < D * F.col("a_n"), F.col("a_n") >= 5))
              .withColumn("s_brk", _i(F.col("rec_broken_share_out") >= CONFIG["SIG_BROKEN"], F.col("rec_broken_share_out").isNotNull()))
              .withColumn("s_net", _i(F.col("net_flow_all") < -CONFIG["SIG_NETOUT"] * F.col("a_in"), F.col("a_in") >= FLOOR)))
        _sh = (shocks.where(F.col("direction") == "loss").select("cust", F.col("e").alias("mi"))
               .distinct().withColumn("s_shock", F.lit(1)))
        _b = (_b.join(_sh, ["cust", "mi"], "left")
              .withColumn("s_shock", F.when(F.col("a_in") >= FLOOR, F.coalesce(F.col("s_shock"), F.lit(0)))))
        if "dep_bal_mean" in _cols:
            SIGNALS["s_dep"] = "average balance down 30%"
            ANCH["a_dep"] = "deposit collapse"
            _okd = F.col("a_dep") >= CONFIG["ANCHOR_DEP_MIN"]
            if "_bad" in _cols:
                _okd = _okd & (F.coalesce(F.col("_bad"), F.lit(0)) == 0)
            _b = (_b.withColumn("a_dep", F.avg("dep_bal_mean").over(_w6))
                  .withColumn("s_dep", _i(F.col("dep_bal_mean") < D * F.col("a_dep"), _okd))
                  .withColumn("_dc", _i(F.col("dep_bal_mean") < CONFIG["ANCHOR_DEP_DROP"] * F.col("a_dep"), _okd))
                  .withColumn("a_dep_f", F.when(F.col("_dc") == 1, (F.max("_dc").over(_wn1) == 1).cast("int"))
                              .otherwise(F.col("_dc"))))
        if "days_negative" in _cols:
            SIGNALS["s_neg"] = "days with negative balance"
            _b = _b.withColumn("s_neg", _i(F.col("days_negative") > 0, F.col("days_negative").isNotNull()))
        if "past_due_days" in _cols:
            SIGNALS["s_pdue"] = "past due"
            _b = _b.withColumn("s_pdue", _i(F.col("past_due_days") > 0, F.col("past_due_days").isNotNull()))
        if "pd_pct_now" in _cols:
            SIGNALS["s_pd"] = "PD up 50%"
            ANCH["a_pd"] = "PD doubles"
            _b = (_b.withColumn("pd6", F.max("pd_pct_now").over(_wl6))
                  .withColumn("s_pd", _i(F.col("pd_pct_now") >= CONFIG["SIG_PD_MULT"] * F.col("pd6"), F.col("pd6") > 0))
                  .withColumn("a_pd_f", _i(F.col("pd_pct_now") >= CONFIG["ANCHOR_PD_MULT"] * F.col("pd6"), F.col("pd6") > 0)))
        if "is_default" in _cols:
            SIGNALS["s_def"] = "in default"
            ANCH["a_def"] = "default"
            _b = (_b.withColumn("s_def", _i(F.col("is_default") >= 1, F.col("is_default").isNotNull()))
                  .withColumn("a_def_f", F.col("s_def")))
        # switch-on = fires after Q months without firing, with at least 3 of those months measured
        _fires = {**{k: k for k in SIGNALS}, "a_dep": "a_dep_f", "a_pd": "a_pd_f", "a_def": "a_def_f"}
        for k, src in _fires.items():
            if src in _b.columns:
                _b = _b.withColumn(f"on_{k}", ((F.col(src) == 1) & (F.coalesce(F.max(src).over(_wq), F.lit(0)) == 0)
                                               & (F.count(src).over(_wq) >= 3)).cast("int"))
        _b = _b.withColumn("dec", F.ntile(10).over(W.partitionBy("mi").orderBy(F.coalesce(F.col("a_in"), F.lit(0.0)))))
        _keep = ["cust", "master", "naics2n", "mi", "a_in", "dec"] + list(SIGNALS) + \
                [f"on_{k}" for k in _fires if f"on_{k}" in _b.columns]
        sigs = cache_parquet(_b.select(*_keep), "ew_signals")

    with step("anchors and matched controls"):
        A0, A1 = BM + LB, MAX_MI
        _pw = (payer_month.where((F.col("scope") == "int") & (F.col("pw_loss") == 1))
               .select(F.col("other").alias("cust"), F.col("e").alias("mi")).distinct().withColumn("on_a_pay", F.lit(1)))
        sigs_a = sigs.join(_pw, ["cust", "mi"], "left").withColumn("on_a_pay", F.coalesce(F.col("on_a_pay"), F.lit(0)))
        _an = None
        for a in ANCH:
            x = (sigs_a.where((F.col(f"on_{a}") == 1) & F.col("mi").between(A0, A1) & (F.col("a_in") > 0))
                 .withColumn("rn", F.row_number().over(W.partitionBy("cust").orderBy("mi"))).where(F.col("rn") == 1)
                 .select(F.lit(a).alias("atype"), "cust", "master", "naics2n", "dec", F.col("mi").alias("m"), "a_in")
                 .withColumn("_h", F.abs(F.xxhash64("cust", F.lit(a))))
                 .withColumn("rn", F.row_number().over(W.partitionBy("atype").orderBy("_h")))
                 .where(F.col("rn") <= CONFIG["ANCHOR_MAX"]).drop("rn", "_h"))
            _an = x if _an is None else _an.unionByName(x)
        _arr = sigs_a.select("cust", *[F.when(F.col(f"on_{a}") == 1, F.col("mi")).alias(a) for a in ANCH]) \
            .groupBy("cust").agg(*[F.collect_set(a).alias(f"arr_{a}") for a in ANCH])
        _cand = (sigs.where(F.col("a_in") > 0).select(F.col("cust").alias("unit"), F.col("master").alias("u_master"),
                                                      F.col("mi").alias("m"), "naics2n", "dec", F.col("a_in").alias("u_in"))
                 .join(_arr.withColumnRenamed("cust", "unit"), "unit", "left"))
        _arrc = F.lit(None).cast("array<int>")
        for a in ANCH:
            _arrc = F.when(F.col("atype") == a, F.col(f"arr_{a}")).otherwise(_arrc)
        _mt = (_an.join(_cand, ["m", "naics2n", "dec"], "inner")
               .where((F.col("unit") != F.col("cust")) & (F.col("u_master") != F.col("master")))
               .withColumn("_arr", F.coalesce(_arrc, F.array().cast("array<int>")))
               .where(F.expr(f"NOT exists(_arr, x -> x >= m - {LB} AND x <= m)"))
               .withColumn("dist", F.abs(F.log(F.col("u_in") + 1.0) - F.log(F.col("a_in") + 1.0)))
               .withColumn("rn", F.row_number().over(W.partitionBy("atype", "cust").orderBy("dist", F.xxhash64("unit"))))
               .where(F.col("rn") <= CONFIG["ANCHOR_CONTROLS"])
               .select("atype", F.col("cust").alias("aid"), "m", "unit", F.lit("C").alias("role")))
        _U = (_an.select("atype", F.col("cust").alias("aid"), "m", F.col("cust").alias("unit"), F.lit("S").alias("role"))
              .unionByName(_mt))
        ew_units = cache_parquet(_U, "ew_units")
        _na = ew_units.groupBy("atype", "role").agg(F.countDistinct("aid").alias("anchors"),
                                                    F.count(F.lit(1)).alias("units")).toPandas()

    with step("look-back: signal switch-ons in the 12 months before each anchor"):
        _sk = list(SIGNALS)
        _lbk = (ew_units.join(sigs.withColumnRenamed("cust", "unit"), "unit", "inner")
                .where(F.col("mi").between(F.col("m") - LB, F.col("m")))
                .withColumn("t", F.col("mi") - F.col("m")))
        _first = (_lbk.groupBy("atype", "aid", "role", "unit").agg(
            *[F.min(F.when(F.col(f"on_{k}") == 1, F.col("t"))).alias(f"t_{k}") for k in _sk],
            *[F.max(F.col(k).isNotNull().cast("int")).alias(f"d_{k}") for k in _sk]).toPandas())
        _byt = (_lbk.select("atype", "role", "t", F.explode(F.array(*[F.struct(F.lit(k).alias("sig"), F.col(f"on_{k}").alias("on"),
                                                                              F.col(k).isNotNull().cast("int").alias("has"))
                                                                     for k in _sk])).alias("x"))
                .groupBy("atype", "role", "t", F.col("x.sig").alias("sig"))
                .agg(F.sum("x.on").alias("on"), F.sum("x.has").alias("has")).toPandas())

    EXCL = {"a_def": ["s_def"], "a_pd": ["s_pd"], "a_dep": ["s_dep"], "a_pay": []}
    _ex, _ff, _ll = [], [], []
    for a, alab in ANCH.items():
        fa = _first[_first.atype == a]
        S_, C_ = fa[fa.role == "S"], fa[fa.role == "C"]
        if len(S_) == 0:
            continue
        for k in _sk:
            sa, ca = S_[S_[f"d_{k}"] == 1], C_[C_[f"d_{k}"] == 1]
            if len(sa) < 20 or len(ca) < 20:
                continue
            pa = float((sa[f"t_{k}"] <= -1).mean()); pc = float((ca[f"t_{k}"] <= -1).mean())
            p0 = float((sa[f"t_{k}"] == 0).mean()); c0 = float((ca[f"t_{k}"] == 0).mean())
            se = math.sqrt(pa * (1 - pa) / len(sa) + pc * (1 - pc) / len(ca))
            lead = sa.loc[sa[f"t_{k}"] <= -1, f"t_{k}"]
            _ex.append(dict(anchor=alab, signal=SIGNALS[k], sig=k, defines_anchor=k in EXCL.get(a, []),
                            anchored=len(sa), controls=len(ca), anchored_before=pa, controls_before=pc,
                            excess_pp=100 * (pa - pc), lo_pp=100 * (pa - pc - 1.96 * se), hi_pp=100 * (pa - pc + 1.96 * se),
                            lift=pa / pc if pc > 0 else np.nan, median_lead_months=float(-lead.median()) if len(lead) else np.nan,
                            anchored_same_month=p0, controls_same_month=c0))
        # first to fire (anchored units, signals switched on in -LB..-1, anchor-defining signals left out)
        use = [k for k in _sk if k not in EXCL.get(a, [])]
        T_ = S_[[f"t_{k}" for k in use]].where(S_[[f"t_{k}" for k in use]] <= -1)
        tmin = T_.min(axis=1)
        any_ = tmin.notna()
        if any_.sum():
            first = T_[any_].eq(tmin[any_], axis=0)
            w = first.div(first.sum(axis=1), axis=0).sum()
            for col_, v in w.items():
                _ff.append(dict(anchor=alab, signal=SIGNALS[col_[2:]], share_first=float(v / any_.sum()),
                                anchored_with_any=int(any_.sum()), anchored=len(S_)))
        # pairwise order among anchored units, window -LB..0
        _pk = [k for k in _sk if k not in EXCL.get(a, [])]
        Tw = S_[[f"t_{k}" for k in _pk]]
        for i, ka in enumerate(_pk):
            for kb in _pk[i + 1:]:
                ta, tb = Tw[f"t_{ka}"], Tw[f"t_{kb}"]
                both = ta.notna() & tb.notna()
                nb = int(both.sum())
                if nb < 20:
                    continue
                _ll.append(dict(anchor=alab, a=SIGNALS[ka], b=SIGNALS[kb], both=nb,
                                a_first=float((ta[both] < tb[both]).mean()), same_month=float((ta[both] == tb[both]).mean()),
                                b_first=float((ta[both] > tb[both]).mean()), median_b_minus_a=float((tb[both] - ta[both]).median())))
    early = pd.DataFrame(_ex)
    first_fire = pd.DataFrame(_ff)
    leadlag = pd.DataFrame(_ll)
    for _n, _d in [("p1_early_warning", early), ("p1_first_to_fire", first_fire), ("p1_lead_lag", leadlag)]:
        _d.to_csv(os.path.join(OUT_DIR, f"{_n}.csv"), index=False)

    for a, alab in ANCH.items():
        n_a = int(_na[(_na.atype == a) & (_na.role == "S")].anchors.sum()) if len(_na) else 0
        gate(f"early warning: {alab} anchors", n_a >= 200, f"{n_a:,} anchors with matched controls",
             "under 200: read that anchor as a case list, not a rate")

    # f12: excess switch-on rate by month before the anchor (anchored minus controls, pp)
    if not _byt.empty:
        from matplotlib.colors import LinearSegmentedColormap
        _cm2 = LinearSegmentedColormap.from_list("div2", ["#2a78d6", "#ecebe7", "#eb6834"])
        _an_list = [a for a in ANCH if (_byt.atype == a).any()]
        fig, axes = plt.subplots(1, len(_an_list), figsize=(5.2 * len(_an_list), 0.42 * len(_sk) + 2.0), squeeze=False)
        tt = np.arange(-LB, 1)
        for ax, a in zip(axes[0], _an_list):
            g = _byt[_byt.atype == a]
            rate = lambda r: (g[g.role == r].groupby(["sig", "t"]).on.sum() / g[g.role == r].groupby(["sig", "t"]).has.sum()
                              ).unstack("t").reindex(index=_sk, columns=tt)
            M = 100 * (rate("S") - rate("C"))
            vmax = max(np.nanpercentile(np.abs(M.to_numpy()), 95) if np.isfinite(M.to_numpy()).any() else 1, 0.5)
            ax.imshow(M.to_numpy(), cmap=_cm2, vmin=-vmax, vmax=vmax, aspect="auto")
            ax.set_xticks(range(len(tt))[::2]); ax.set_xticklabels([f"{v:+d}" for v in tt[::2]])
            ax.set_yticks(range(len(_sk))); ax.set_yticklabels([SIGNALS[k] for k in _sk] if ax is axes[0][0] else [], fontsize=8.5)
            ax.grid(False)
            ax.set_title(f"{ANCH[a]}\n(scale up to {vmax:.1f} pp)", fontsize=9.5)
            ax.set_xlabel("months before the anchor")
        fig.suptitle("Signals switching on before real events: anchored minus matched controls, pp per month "
                     "(orange = more often than controls, blue = less)",
                     x=0.01, ha="left", fontsize=12, fontweight="bold")
        fig.tight_layout(rect=(0, 0, 1, 0.92))
        savefig(fig, "f12_early_warning", "6b", "Excess switch-on rate of each signal by month before each anchor event")

    # f13: pairwise order (row signal before column signal minus the reverse), per anchor
    if not leadlag.empty:
        _al = [alab for alab in ANCH.values() if (leadlag.anchor == alab).any()]
        labs = [SIGNALS[k] for k in _sk]
        fig, axes = plt.subplots(1, len(_al), figsize=(5.6 * len(_al), 0.42 * len(labs) + 2.6), squeeze=False)
        from matplotlib.colors import LinearSegmentedColormap
        _cm3 = LinearSegmentedColormap.from_list("div3", ["#eb6834", "#ecebe7", "#2a78d6"])
        for ax, alab in zip(axes[0], _al):
            L = leadlag[leadlag.anchor == alab]
            M = pd.DataFrame(np.nan, index=labs, columns=labs)
            for r in L.itertuples():
                M.loc[r.a, r.b] = 100 * (r.a_first - r.b_first)
                M.loc[r.b, r.a] = 100 * (r.b_first - r.a_first)
            ax.imshow(M.to_numpy(float), cmap=_cm3, vmin=-60, vmax=60, aspect="auto")
            ax.set_xticks(range(len(labs))); ax.set_xticklabels(labs, rotation=70, ha="right", fontsize=8)
            ax.set_yticks(range(len(labs))); ax.set_yticklabels(labs if ax is axes[0][0] else [], fontsize=8)
            ax.grid(False); ax.set_title(alab, fontsize=10)
        fig.suptitle("Which comes first: % of anchored customers where the row signal switched on before the column "
                     "signal, minus the reverse (blue = row first)", x=0.01, ha="left", fontsize=11, fontweight="bold")
        fig.tight_layout(rect=(0, 0, 1, 0.92))
        savefig(fig, "f13_lead_lag", "6b", "Pairwise order of signals before each anchor (spec lead-lag table)")

    _exs = early.sort_values(["anchor", "excess_pp"], ascending=[True, False]) if len(early) else early
    _ffs = (first_fire.sort_values(["anchor", "share_first"], ascending=[True, False]).groupby("anchor").head(5)
            if len(first_fire) else first_fire)
    _lls = (leadlag.assign(gap=lambda x: (x.a_first - x.b_first).abs()).sort_values(["anchor", "gap"], ascending=[True, False])
            .groupby("anchor").head(6).drop(columns="gap") if len(leadlag) else leadlag)
    _nat = pd.DataFrame([dict(anchor=alab,
                              anchors=int(_na[(_na.atype == a) & (_na.role == "S")].anchors.sum()) if len(_na) else 0,
                              control_units=int(_na[(_na.atype == a) & (_na.role == "C")].units.sum()) if len(_na) else 0)
                         for a, alab in ANCH.items()])
    CELL.show(("Anchors and controls (anchor months " + f"{mi2m(A0)} .. {mi2m(A1)})", _nat),
              ("Signals in the 12 months before each anchor, anchored against controls (excess in pp, 95% interval)", _exs),
              ("Which signal switched on first (anchored customers with at least one, ties shared)", _ffs),
              ("Clearest pairwise orders (share of anchored customers where a came first / same month / b first)", _lls))
    for alab in ANCH.values():
        e = early[(early.anchor == alab) & ~early.defines_anchor & (early.lo_pp > 0)].sort_values("excess_pp", ascending=False)
        if len(e):
            ans(f"Early warning, {alab}: strongest signals before it",
                "; ".join(f"{r.signal} +{r.excess_pp:.1f} pp, {r.median_lead_months:.0f} months ahead" for r in e.head(3).itertuples()))
        f_ = first_fire[first_fire.anchor == alab].sort_values("share_first", ascending=False)
        if len(f_):
            ans(f"Early warning, {alab}: usually first to switch on", f"{f_.signal.iloc[0]} ({f_.share_first.iloc[0]:.0%})")

## §7 · Case panels

The largest events among operating businesses (6 losses, 4 gains) plus named cases: monthly inflow as green
bars above zero, outflow as red bars below, the event month marked. The table lists the streams behind each event.

In [ ]:
if RUN_FULL:
    CELL.start("§7 · Case panels")
    with step("case selection and series"):
        _op = events[events.segment == "operating"].sort_values("amount", ascending=False)
        _cases = pd.concat([_op[_op.direction == "loss"].head(CONFIG["CASE_TOP_LOSS"]),
                            _op[_op.direction == "gain"].head(CONFIG["CASE_TOP_GAIN"])])
        _cases = _cases[["cust", "customer_name", "segment", "e", "direction", "amount", "size", "type", "breadth"]]
        for _p in CONFIG["CASE_NAME_PATTERNS"]:
            _nm = (shocks.where(F.upper(F.col("customer_name")).contains(_p.upper()))
                   .orderBy(F.desc("amount")).limit(1).toPandas())
            if len(_nm) and _nm.cust.iloc[0] not in set(_cases.cust):
                _cases = pd.concat([_cases, _nm[["cust", "customer_name", "segment", "e", "direction", "amount", "size",
                                                 "type", "breadth"]]], ignore_index=True)
        _ids = list(_cases.cust)
        _ser = pop.where(F.col("cust").isin(_ids)).select("cust", "mi", "amt_all_in", "amt_all_out").toPandas()
        _keys = spark.createDataFrame(_cases[["cust", "e"]].astype({"e": "int32"}))
        _br = (stream_events.join(F.broadcast(_keys), ["cust", "e"], "inner")
               .select("cust", "scope", "other", F.coalesce(F.col("ev"), F.lit("grow")).alias("event"), "expected",
                       (F.col("lost") + F.col("gain")).alias("dollars"), "pw_loss", "pw_gain", "n_risk", "n_broken", "n_grow")
               .unionByName(new_streams.join(F.broadcast(_keys), ["cust", "e"], "inner")
                            .select("cust", "scope", "other", F.lit("new payer").alias("event"),
                                    F.lit(None).cast("double").alias("expected"), F.col("new_amt").alias("dollars"),
                                    "pw_loss", "pw_gain", "n_risk", "n_broken", "n_grow"))
               .withColumn("rn", F.row_number().over(W.partitionBy("cust").orderBy(F.desc("dollars"))))
               .where(F.col("rn") <= 5).drop("rn").toPandas())
    for k, c in enumerate(_cases.itertuples(), start=1):
        s = _ser[_ser.cust == c.cust].set_index("mi").reindex(range(MAX_MI + 1)).fillna(0.0)
        fig, ax = plt.subplots(figsize=(11, 4))
        x = np.arange(MAX_MI + 1)
        ax.bar(x, s.amt_all_in / 1e6, color=C_IN, width=0.78, label="inflow")
        ax.bar(x, -s.amt_all_out / 1e6, color=C_OUT, width=0.78, label="outflow")
        ax.axhline(0, color=INK2, linewidth=1)
        ax.axvline(c.e, color=INK, linewidth=1.2, linestyle="--")
        ax.annotate(f"{c.direction} {mi2m(c.e)}: {usd(c.amount)} net ({c.size:.0%} of baseline), {c.type}",
                    xy=(c.e, ax.get_ylim()[1]), xytext=(4, -12), textcoords="offset points", fontsize=9, color=INK)
        ax.set_xticks(x[::2]); ax.set_xticklabels([mi2m(i) for i in x[::2]], rotation=60)
        ax.set_ylabel("USD millions per month"); ax.grid(axis="x", visible=False)
        ax.legend(loc="lower left", bbox_to_anchor=(0, 1.0), ncol=2, fontsize=9)
        ax.set_title(f"Case {k}: {c.customer_name}", pad=26)
        savefig(fig, f"f16_case_{k:02d}", "7", f"Case panel: {c.customer_name} ({c.direction}), inflow green, outflow red")
    _cases_show = _cases.assign(month=_cases.e.map(mi2m)).drop(columns=["e", "cust"])
    _br = _br.merge(_cases[["cust", "customer_name"]], on="cust").drop(columns=["cust"])
    CELL.show(("Cases", _cases_show[["customer_name", "segment", "month", "direction", "amount", "size", "type", "breadth"]]),
              ("Largest streams behind each case (other = payer id; n_risk / n_broken / n_grow = that payer's standing PNC payees)",
               _br[["customer_name", "scope", "other", "event", "expected", "dollars", "n_risk", "n_broken", "n_grow"]]))

## §7b · Real scenarios

Shocks we know happened, from the spec's test cases (United Consumer Financial Service is a case panel in §7).

- **Change Healthcare (2024-02).** Customers paid by a counterparty named CHANGE HEALTHCARE in the months before,
  month by month: dollars from it, and their own inflow, outflow, payment count and balance as change from 2024-01,
  against customers of the same NAICS2 it did not pay. Data start 2024-01, so there is one baseline month:
  descriptive only.
- **2026 oil price shock (2026-03).** Two groups against customers in sectors the shock should not hit directly
  (leaving out mining, utilities, transport, finance and public), matched on size decile (3 controls per unit):
  **transport** (NAICS 48-49) and **dependent on transport** (at least 20% of baseline inflow from PNC-customer
  payers in NAICS 48-49, the second hop). Change from each unit's own 6 months before, cluster bootstrap by
  master, and the same onset reading as §5b (months -3..-1 against baseline -6..-4).
  **Seasonality:** the same calculation for the same months a year earlier (2025-03) is subtracted, so a
  sector's usual winter-to-spring pattern drops out; the table shows the adjusted, unadjusted and year-before
  effects at +1.

In [ ]:
if RUN_FULL and CONFIG["SCENARIOS"]:
    import re
    CELL.start("§7b · Real scenarios")
    SC_OUT = {"amt_all_in": ("Incoming dollars", "logpct", 1.0), "amt_all_out": ("Outgoing dollars", "logpct", 1.0),
              "ntxn_all_out": ("Outgoing payment count", "logpct", 1.0),
              "rec_broken_share_out": ("Standing payees not paid (pp)", "diff", 100.0),
              "net_flow_all": ("Net flow (pp of monthly inflow)", "scaled", 1.0),
              "dep_bal_mean": ("Average balance", "logpct", 1.0), "days_negative": ("Days with negative balance", "diff", 1.0),
              "pd_pct_now": ("PD (pd_pct_now, change)", "diff", 1.0), "past_due_days": ("Days past due", "diff", 1.0),
              "is_default": ("In default (pp)", "diff", 100.0)}

    def _sc_series(units_pdf, m_lo, m_hi):
        """Monthly outcomes for the given units (cust), months m_lo..m_hi, payments + deposits + credit."""
        u = spark.createDataFrame(units_pdf[["cust"]].drop_duplicates())
        b = (pop.join(F.broadcast(u), "cust").where(F.col("mi").between(m_lo, m_hi))
             .select("cust", "customer_pwr_id", "mi", "base_in", *[c for c in SC_OUT if c in pop.columns]))
        for fam, keep in [("deposits", ["dep_bal_mean", "days_negative"]),
                          ("credit", ["pd_pct_now", "past_due_days", "is_default"])]:
            t, k = fam_table(fam, keep)
            if t is not None:
                b = b.join(t, [k, "mi"], "left")
                if "_bad" in t.columns:
                    b = b.withColumn("dep_bal_mean", F.when(F.coalesce(F.col("_bad"), F.lit(0)) == 0, F.col("dep_bal_mean"))).drop("_bad")
        return b.drop("customer_pwr_id").toPandas()

    def _sc_effects(ser, S_ids, C_ids, master, tt, base_ts, key, draws=None):
        """Per outcome: event-minus-control change from baseline months base_ts, by month tt.
        draws: optional dict filled with col -> (d, bootstrap draws) on the log / raw scale."""
        rows = []
        idx = pd.MultiIndex.from_frame(ser[["cust", "t"]])
        ub = (ser.pivot_table(index="cust", columns="t", values="amt_all_in", aggfunc="sum")
              .reindex(columns=base_ts).fillna(0.0).mean(axis=1))
        for col, (lab, kind, mult) in SC_OUT.items():
            if col not in ser.columns or ser[col].notna().sum() == 0:
                continue
            v = ser[col].astype(float) * mult
            if col in OUTCOMES:
                v = v.fillna(0.0)
            m = pd.Series(v.to_numpy(), index=idx).unstack("t").reindex(columns=tt)
            base = m[base_ts].mean(axis=1, skipna=True)
            if kind == "logpct":
                x = np.log((m.clip(lower=0) + 1).div(base.clip(lower=0) + 1, axis=0)).clip(-math.log(CONFIG["LOG_CAP"]), math.log(CONFIG["LOG_CAP"]))
            elif kind == "scaled":
                x = m.sub(base, axis=0).div(ub.reindex(m.index), axis=0).mul(100).clip(-500, 500)
            else:
                x = m.sub(base, axis=0)
            S, C = x.reindex(S_ids), x.reindex(C_ids)
            r = did_boot(S.to_numpy(), C.to_numpy(), master.reindex(S_ids).to_numpy(), master.reindex(C_ids).to_numpy(), (key, col))
            if draws is not None:
                draws[col] = (r["d"], r["bd"], lab, kind)
            f = (lambda z: 100 * (np.exp(z) - 1)) if kind == "logpct" else (lambda z: z)
            for j, t in enumerate(tt):
                rows.append(dict(outcome=lab, col=col, kind=kind, t=int(t), n=int(r["n"][j]), event=f(r["ms"][j]),
                                 control=f(r["mc"][j]), effect=f(r["d"][j]), lo=f(r["lo"][j]), hi=f(r["hi"][j])))
        return pd.DataFrame(rows)

    _sc_blocks, _sc_eff, _sc_first = [], [], []
    _m_master = pop.select("cust", "master").dropDuplicates(["cust"])
    for sc in CONFIG["SCENARIOS"]:
        m0 = m2mi(sc["month"])
        if m0 < 0 or m0 > MAX_MI:
            _sc_blocks.append((sc["name"], pd.DataFrame([dict(status=f"{sc['month']} is outside the data window")])))
            continue

        if sc["kind"] == "payer":
            with step(f"scenario: {sc['name']}"):
                pat = "|".join(re.escape(x.upper()) for x in sc["patterns"])
                try:
                    _ents = (spark.table(CONFIG["CPTY_DIM_TABLE"])
                             .where(F.upper(F.col("cpty_name")).rlike(pat)).select(F.col("cpty_id").cast("string").alias("cpty_id"))
                             .join(spark.table(CONFIG["CPTY_MAP_TABLE"]).select(F.col("cpty_id").cast("string").alias("cpty_id"),
                                                                                F.col("ent_l2").cast("string").alias("other")), "cpty_id")
                             .select("other").distinct().withColumn("scope", F.lit("ext")))
                except Exception as ex:
                    _sc_blocks.append((sc["name"], pd.DataFrame([dict(status=f"counterparty name tables not readable: {type(ex).__name__}")])))
                    continue
                _ints = (pop.where(F.upper(F.coalesce(F.col("customer_name"), F.lit(""))).rlike(pat))
                         .select(F.col("cust").alias("other")).distinct().withColumn("scope", F.lit("int")))
                _src = _ents.unionByName(_ints)
                _from = (spark.table(CONFIG["PAIR_TABLE"])
                         .where((F.col("dir") == "in") & F.col("scope").isin("int", "ext") & F.col("month").between(MONTH_MIN, MONTH_MAX))
                         .select(F.col("cust").cast("string").alias("cust"), "scope", F.col("other").cast("string").alias("other"),
                                 mi_col("month").alias("mi"), F.col("amount").cast("double").alias("amount"))
                         .join(F.broadcast(_src), ["scope", "other"], "inner")
                         .groupBy("cust", "mi").agg(F.sum("amount").alias("from_payer"))
                         .join(pop.select("cust", "naics2n").dropDuplicates(["cust"]), "cust", "inner")
                         .toPandas())
                lo_m = max(0, m0 - 3)
                payees = sorted(set(_from[(_from.mi >= lo_m) & (_from.mi < m0) & (_from.from_payer > 0)].cust))
                if not payees:
                    _sc_blocks.append((sc["name"], pd.DataFrame([dict(status="no TM organisation was paid by it before the month",
                                                                     ents=_src.count())])))
                    continue
                _nai = _from[_from.cust.isin(payees)].drop_duplicates("cust").naics2n.value_counts()
                _ctl = (pop.where(F.col("naics2n").isin(list(_nai.index)) & (F.col("mi") == m0 - 1) & (F.col("amt_all_in") > 0))
                        .select("cust").distinct()
                        .join(spark.createDataFrame(pd.DataFrame({"cust": payees})), "cust", "left_anti")
                        .withColumn("_h", F.abs(F.xxhash64("cust"))).orderBy("_h").limit(CONFIG["SCEN_MAX_UNITS"]).toPandas())
                _units = pd.concat([pd.DataFrame({"cust": payees, "g": "S"}), _ctl[["cust"]].assign(g="C")])
                m_hi = min(MAX_MI, m0 + 9)
                ser = _sc_series(_units, lo_m, m_hi)
                tt = np.arange(lo_m - m0, m_hi - m0 + 1)
                ser["t"] = ser["mi"] - m0
                _mm = _m_master.join(spark.createDataFrame(_units[["cust"]]), "cust").toPandas().set_index("cust")["master"]
                eff = _sc_effects(ser, payees, list(_ctl.cust), _mm, tt, [-1], sc["name"])
                eff["scenario"], eff["group"] = sc["name"], "paid by it"
                _sc_eff.append(eff)
                fr = (_from[_from.cust.isin(payees)].groupby("mi").agg(payees_paid=("cust", "nunique"), dollars=("from_payer", "sum"))
                      .reindex(range(lo_m, m_hi + 1)).fillna(0).reset_index())
                fr.insert(0, "month", fr.pop("mi").map(mi2m))
                _sc_blocks.append((f"{sc['name']}: payees before {sc['month']} ({len(payees):,}), dollars from it by month "
                                   f"({sc.get('note', '')})", fr))
                _pv = eff.pivot_table(index="outcome", columns="t", values="effect")
                _pv.columns = [mi2m(m0 + int(t)) for t in _pv.columns]
                _sc_blocks.append((f"{sc['name']}: payees minus same-NAICS2 customers, change from {mi2m(m0 - 1)} "
                                   "(% for dollars, counts and balance)", _pv.reset_index()))

        elif sc["kind"] == "sector":
            with step(f"scenario: {sc['name']}"):
                _nl = sc["naics2"]

                def _sector_run(mc, tag):
                    """Group and control draws for a shock month mc: {group: (S, C, main draws, pre draws, tt)}."""
                    lo_m, m_hi = max(0, mc - BM), min(MAX_MI, mc + 5)
                    if mc - BM < 0:
                        return {}
                    _at = (pop.where((F.col("mi") == mc) & (F.col("base_in") >= CONFIG["BASE_IN_MIN"]))
                           .select("cust", "master", "naics2n", "base_in"))
                    _pin = (spark.table(CONFIG["PAIR_TABLE"])
                            .where((F.col("dir") == "in") & (F.col("scope") == "int") & F.col("month").between(mi2m(lo_m), mi2m(mc - 1)))
                            .select(F.col("cust").cast("string").alias("cust"), F.col("other").cast("string").alias("other"),
                                    F.col("amount").cast("double").alias("amount"))
                            .join(pop.select(F.col("cust").alias("other"), F.col("naics2n").alias("o_naics")).dropDuplicates(["other"]), "other")
                            .groupBy("cust").agg(F.sum(F.when(F.col("o_naics").isin(_nl), F.col("amount")).otherwise(0.0)).alias("from_sector")))
                    U = (_at.join(_pin, "cust", "left").na.fill(0.0, ["from_sector"])
                         .withColumn("dep_share", F.col("from_sector") / (F.col("base_in") * float(mc - lo_m)))
                         .withColumn("dec", F.ntile(10).over(W.orderBy("base_in"))).toPandas())
                    U["grp"] = np.where(U.naics2n.isin(_nl), "transport",
                                        np.where(U.dep_share >= sc["exposed_share"], "dependent",
                                                 np.where(~U.naics2n.isin(sc["control_exclude"]) & (U.dep_share < 0.05), "control", "")))
                    U["_h"] = U.cust.map(lambda v: int(hashlib.md5(f"{v}|{tag}".encode()).hexdigest()[:8], 16))
                    U = U.sort_values("_h")
                    grp = {}
                    for gname in ["transport", "dependent"]:
                        S_ = U[U.grp == gname].head(CONFIG["SCEN_MAX_UNITS"])
                        if len(S_) < 20:
                            continue
                        need = S_.dec.value_counts() * CONFIG["N_CONTROLS"]
                        C_ = pd.concat([U[(U.grp == "control") & (U.dec == d_)].head(int(k)) for d_, k in need.items()])
                        grp[gname] = (S_, C_)
                    if not grp:
                        return {}
                    units_ = pd.concat([pd.concat([a[["cust"]], b[["cust"]]]) for a, b in grp.values()]).drop_duplicates()
                    ser = _sc_series(units_, lo_m, m_hi)
                    ser["t"] = ser["mi"] - mc
                    tt = np.arange(-BM, 6)
                    mm = U.set_index("cust")["master"]
                    out = {}
                    for gname, (S_, C_) in grp.items():
                        dm, dp = {}, {}
                        _sc_effects(ser, list(S_.cust), list(C_.cust), mm, tt, list(range(-BM, 0)), (sc["name"], gname, tag), dm)
                        _sc_effects(ser, list(S_.cust), list(C_.cust), mm, tt, list(range(-BM, -3)), (sc["name"], gname, tag, "pre"), dp)
                        out[gname] = (S_, C_, dm, dp, tt)
                    return out

                cur = _sector_run(m0, "shock")
                ref = _sector_run(m0 - 12, "year before") if sc.get("seasonal") else {}
                if not cur:
                    _sc_blocks.append((sc["name"], pd.DataFrame([dict(status="fewer than 20 units in the sector groups")])))
                    continue
                for gname, (S_, C_, dm, dp, tt) in cur.items():
                    rf = ref.get(gname)
                    rows_ = []
                    for col, (d0, b0, lab, kind) in dm.items():
                        f = (lambda z: 100 * (np.exp(z) - 1)) if kind == "logpct" else (lambda z: z)
                        for which, draws_cur, draws_ref in [("main", dm, rf[2] if rf else None), ("pre", dp, rf[3] if rf else None)]:
                            dc, bc, _, _ = draws_cur[col]
                            if draws_ref is not None and col in draws_ref:
                                dr, br, _, _ = draws_ref[col]
                                dd, bb = dc - dr, bc - br
                            else:
                                dd, bb = dc, bc
                            with warnings.catch_warnings():
                                warnings.simplefilter("ignore")
                                lo_, hi_ = np.nanpercentile(bb, 2.5, axis=0), np.nanpercentile(bb, 97.5, axis=0)
                            for j, t in enumerate(tt):
                                rows_.append(dict(which=which, outcome=lab, col=col, kind=kind, t=int(t), effect=f(dd[j]),
                                                  lo=f(lo_[j]), hi=f(hi_[j]), raw_2026=f(dc[j]),
                                                  year_before=f(draws_ref[col][0][j]) if draws_ref is not None and col in draws_ref else np.nan))
                    E_ = pd.DataFrame(rows_)
                    eff = E_[E_.which == "main"].drop(columns="which").copy()
                    pre = E_[E_.which == "pre"].drop(columns="which")
                    eff["scenario"], eff["group"] = sc["name"], gname
                    _sc_eff.append(eff)
                    for col in eff.col.unique():
                        x = pd.concat([pre[(pre.col == col) & pre.t.between(-3, -1)], eff[(eff.col == col) & (eff.t >= 0)]]).sort_values("t")
                        r = onset_read(x.t, x.effect, x.lo, x.hi)
                        e1 = eff[(eff.col == col) & (eff.t == 1)]
                        _sc_first.append(dict(scenario=sc["name"], group=gname, outcome=x.outcome.iloc[0],
                                              adjusted="minus a year earlier" if rf else "not adjusted",
                                              units=len(S_), controls=len(C_),
                                              reading=("moves before the event" if r["clear"] and r["onset"] < 0 else
                                                       ("moves" if r["clear"] else "no clear move")),
                                              onset=r["onset"], half=r["half"], peak_month=r["peak_t"], peak=r["peak"],
                                              effect_at_1=float(e1.effect.iloc[0]) if len(e1) else np.nan,
                                              unadjusted_at_1=float(e1.raw_2026.iloc[0]) if len(e1) else np.nan,
                                              year_before_at_1=float(e1.year_before.iloc[0]) if len(e1) else np.nan))
    sc_effects = pd.concat(_sc_eff, ignore_index=True) if _sc_eff else pd.DataFrame()
    sc_first = pd.DataFrame(_sc_first)
    if len(sc_effects):
        sc_effects.to_csv(os.path.join(OUT_DIR, "p1_scenario_effects.csv"), index=False)
    if len(sc_first):
        sc_first["_r"] = sc_first.reading.map({"moves before the event": 0, "moves": 1, "no clear move": 2})
        sc_first = sc_first.sort_values(["scenario", "group", "_r", "onset", "half"]).drop(columns="_r")

    # f20 / f21: event minus controls for the key outcomes, one line per group, shaded interval
    _keyc = ["amt_all_in", "amt_all_out", "ntxn_all_out", "dep_bal_mean", "net_flow_all", "pd_pct_now"]
    _gcol = {"paid by it": C_DIR["loss"], "transport": C_DIR["loss"], "dependent": C_DIR["gain"]}
    for k, (scn, fn) in enumerate([(sc["name"], f"f{20 + i}_scenario_{i + 1}") for i, sc in enumerate(CONFIG["SCENARIOS"])]):
        e = sc_effects[sc_effects.scenario == scn] if len(sc_effects) else pd.DataFrame()
        cols = [c for c in _keyc if len(e) and c in set(e.col)]
        if not cols:
            continue
        fig, axes = plt.subplots(2, 3, figsize=(14, 7), squeeze=False)
        for ax, c in zip(axes.ravel(), cols):
            for g in e.group.unique():
                x = e[(e.col == c) & (e.group == g)].sort_values("t")
                ax.fill_between(x.t, x.lo, x.hi, color=_gcol.get(g, C_EVENT), alpha=0.16, linewidth=0)
                ax.plot(x.t, x.effect, color=_gcol.get(g, C_EVENT), marker="o", markersize=4, label=g)
            ax.axhline(0, color=INK2, linewidth=1); ax.axvline(0, color=INK2, linewidth=1, linestyle=":")
            ax.set_title(x.outcome.iloc[0] + (" (%)" if x.kind.iloc[0] == "logpct" else ""), fontsize=9.5)
            ax.set_xlabel("months from the shock month")
        for ax in axes.ravel()[len(cols):]:
            ax.axis("off")
        axes[0][0].legend(loc="lower left", bbox_to_anchor=(0, 1.08), ncol=3, fontsize=9)
        fig.suptitle(f"{scn}: event group minus controls (95% interval)", x=0.01, ha="left", fontsize=12, fontweight="bold")
        fig.tight_layout(rect=(0, 0, 1, 0.93))
        savefig(fig, fn, "7b", f"Real scenario: {scn}, event group minus controls")

    _sfs = (sc_first.assign(onset=lambda x: x.onset.map(lambda v: "" if not np.isfinite(v) else f"{int(v):+d}"),
                            half=lambda x: x.half.map(lambda v: "" if not np.isfinite(v) else f"{int(v):+d}"),
                            peak_month=lambda x: x.peak_month.map(lambda v: "" if not np.isfinite(v) else f"{int(v):+d}"))
            if len(sc_first) else sc_first)
    CELL.show(*_sc_blocks, ("Sector scenarios: order in which outcomes move (onset, half of peak, peak)", _sfs))
    for g, x in (sc_first.groupby(["scenario", "group"]) if len(sc_first) else []):
        mv = x[x.reading != "no clear move"]
        ans(f"{g[0]}, {g[1]}: outcomes that move, in order",
            " > ".join(f"{r.outcome} ({int(r.onset):+d})" for r in mv.itertuples()) or "none clear of zero")

## §7c · Exposure score and watchlist (prototype)

Turns the study into a monthly list. For every eligible TM organisation, as of the end of a month m (only data
through m is used):

- **Payer under stress at m:** it broke at least 25% of its standing PNC payees in month m−1 (3+ payees), or, for a
  PNC-customer payer, one of its own warning signals switched on in the 6 months before (inflow shock, standing payees
  unpaid, negative-balance days, past due, default). §6b found payers cutting many payees had their own shock ~6
  months earlier.
- **Exposure** = monthly dollars expected from the customer's standing payers that are under stress ÷ usual monthly
  inflow (payers still paying it; related companies and own accounts elsewhere left out).
- **Own warning signals:** switch-ons in months m−2..m of fewer outgoing payments, standing payees unpaid,
  negative-balance days, past due (the signals that moved first before real defaults).
- **Tier:** A exposure 20%+; B exposure 10–20% or 2+ own signals; C exposure 5–10% or 1 own signal.
- **Absorption and stakes** (latest month only, from the §5e loss models and each customer's traits): predicted change
  in own payments, payments to regular PNC payees and average balance if the exposed income is lost, plus DHE and
  trailing TM revenue for context. Shown side by side, not added up.

**Backtest:** months 2025-01 .. three months before the end. For each customer-month: was there a loss event of 20%+
in the next 3 months, did standing payees go unpaid (25%+) in the next 3 months, did past due or default switch on in
the next 6 months (customers with credit data). Rates by exposure band, own-signal count and tier, against customers
with neither.

The lists name customers and stay in the notebook and the csv files; the report shows only the method and backtest.

In [ ]:
if RUN_FULL and CONFIG["RUN_SCORE"]:
    CELL.start("§7c · Exposure score and watchlist")
    _ta, _tb, _tc = sorted(CONFIG["TIER_EDGES_SCORE"], reverse=True)
    _ten = (F.col("cnt12") >= CONFIG["TENURE_MIN"]) if CONFIG["TENURE_MIN"] else F.lit(True)
    _have_sigs = "sigs" in globals() and sigs is not None

    with step("standing payers as of each month end"):
        _st = (win.where((F.col("mi").between(EV_START - 1, MAX_MI)) & (F.col("cnt6") >= SM) & _ten)
               .withColumn("med6", F.expr("element_at(array_sort(last6), cast(ceil(size(last6) / 2.0) as int))"))
               .where(F.expr("aggregate(last3, cast(0 as double), (a, x) -> a + x) / size(last3)")
                      <= CONFIG["STEADY_MAX"] * F.col("med6"))
               .select("cust", "scope", "other", F.col("mi").alias("m"),
                       F.expr("element_at(array_sort(last3), cast(ceil(size(last3) / 2.0) as int))").alias("expected")))
        if EXCL_PAIRS is not None:
            _st = _st.join(EXCL_PAIRS.distinct(), ["cust", "scope", "other"], "left_anti")
        standing = cache_parquet(_st, "score_standing")

    with step("payers under stress each month"):
        _ps = (payer_month.where((F.col("n_risk") >= CONFIG["STRESS_MIN_PAYEES"])
                                 & (F.col("n_broken") >= CONFIG["STRESS_BREAK_SHARE"] * F.col("n_risk")))
               .select("scope", "other", (F.col("e") + 1).alias("m"), F.lit(1).alias("st_breaks"),
                       (F.col("n_broken") / F.col("n_risk")).alias("broke_share"), F.col("n_risk").alias("payer_payees")))
        if _have_sigs:
            _own_cols = [c for c in ["on_s_brk", "on_s_pdue", "on_s_def", "on_s_neg"] if c in sigs.columns]
            _o = sigs.select("cust", "mi", F.greatest(*[F.coalesce(F.col(c), F.lit(0)) for c in _own_cols]).alias("on"),
                             F.coalesce(F.col("on_s_shock"), F.lit(0)).alias("shock") if "on_s_shock" in sigs.columns else F.lit(0).alias("shock"))
            _L = CONFIG["STRESS_OWN_MONTHS"]
            _own = (_o.where((F.col("on") == 1) | (F.col("shock") == 1))
                    .withColumn("m", F.explode(F.when(F.col("on") == 1, F.sequence(F.col("mi"), F.col("mi") + _L - 1))
                                               .otherwise(F.sequence(F.col("mi") + 1, F.col("mi") + _L))))
                    .select(F.lit("int").alias("scope"), F.col("cust").alias("other"), "m").distinct()
                    .withColumn("st_own", F.lit(1)))
            _ps = _ps.join(_own, ["scope", "other", "m"], "full").na.fill(0, ["st_breaks", "st_own"])
        else:
            _ps = _ps.withColumn("st_own", F.lit(0))
        stress = cache_parquet(_ps.where(F.col("m").between(EV_START, MAX_MI)), "score_stress")

    with step("exposure, own signals, tiers for every customer-month"):
        _ex = (standing.join(stress.select("scope", "other", "m", "st_breaks", "st_own"), ["scope", "other", "m"], "inner")
               .withColumn("rn", F.row_number().over(W.partitionBy("cust", "m").orderBy(F.desc("expected"), "other")))
               .groupBy("cust", "m").agg(F.sum("expected").alias("at_risk"), F.count(F.lit(1)).alias("n_stressed"),
                                         F.max(F.when(F.col("rn") == 1, F.concat(F.col("scope"), F.lit(":"), F.col("other")))).alias("top_payer"),
                                         F.max(F.when(F.col("rn") == 1, F.col("expected"))).alias("top_exp")))
        _base = elig.where(F.col("mi").between(EV_START, MAX_MI)).select(
            "cust", F.col("mi").alias("m"), "customer_pwr_id", "master", "customer_name", "naics2n", "segment", "size_band", "base_in")
        _sc = (_base.join(_ex, ["cust", "m"], "left").na.fill(0.0, ["at_risk", "n_stressed"])
               .withColumn("exposure", F.least(F.col("at_risk") / F.col("base_in"), F.lit(5.0))))
        _sigk = [k for k in CONFIG["SCORE_OWN_SIGNALS"] if _have_sigs and f"on_{k}" in sigs.columns]
        if _sigk:
            _w3 = W.partitionBy("cust").orderBy("mi").rangeBetween(-2, 0)
            _os = sigs.select("cust", F.col("mi").alias("m"),
                              *[F.max(F.coalesce(F.col(f"on_{k}"), F.lit(0))).over(_w3).alias(f"own_{k}") for k in _sigk])
            _sc = (_sc.join(_os, ["cust", "m"], "left").na.fill(0, [f"own_{k}" for k in _sigk])
                   .withColumn("n_own", sum((F.col(f"own_{k}") for k in _sigk), F.lit(0))))
        else:
            _sc = _sc.withColumn("n_own", F.lit(0))
        _sc = _sc.withColumn("tier", F.when(F.col("exposure") >= _ta, "A")
                             .when((F.col("exposure") >= _tb) | (F.col("n_own") >= 2), "B")
                             .when((F.col("exposure") >= _tc) | (F.col("n_own") >= 1), "C").otherwise("-"))
        score_all = cache_parquet(_sc, "score_all")

    with step("backtest: what happened in the next months"):
        _hl = CONFIG["HEADLINE_MIN"]
        _ev = (shocks.where((F.col("direction") == "loss") & (F.col("size") >= _hl)).select("cust", F.col("e").alias("e_l")))
        _bt = score_all.where((F.col("m") <= MAX_MI - 3) & (F.col("segment") == "operating"))
        _y1 = (_bt.select("cust", "m").join(_ev, "cust").where(F.col("e_l").between(F.col("m") + 1, F.col("m") + 3))
               .select("cust", "m").distinct().withColumn("y_loss", F.lit(1)))
        _bt = _bt.join(_y1, ["cust", "m"], "left").na.fill(0, ["y_loss"])
        _ylabs = {"y_loss": "loss event 20%+ in next 3 months"}
        if _have_sigs and "on_s_brk" in sigs.columns:
            _sb = sigs.where(F.col("on_s_brk") == 1).select("cust", F.col("mi").alias("e_b"))
            _y2 = (_bt.select("cust", "m").join(_sb, "cust").where(F.col("e_b").between(F.col("m") + 1, F.col("m") + 3))
                   .select("cust", "m").distinct().withColumn("y_brk", F.lit(1)))
            _bt = _bt.join(_y2, ["cust", "m"], "left").na.fill(0, ["y_brk"])
            _ylabs["y_brk"] = "standing payees unpaid 25%+ in next 3 months"
        _cc = [c for c in ["on_s_pdue", "on_s_def"] if _have_sigs and c in sigs.columns]
        if _cc:
            _scr = sigs.select("cust", F.col("mi").alias("e_c"), F.greatest(*[F.coalesce(F.col(c), F.lit(0)) for c in _cc]).alias("on"),
                               F.col("s_pdue").isNotNull().alias("has") if "s_pdue" in sigs.columns else F.lit(True).alias("has"))
            _hc = _scr.where("has").select("cust", F.col("e_c").alias("m")).distinct().withColumn("has_credit", F.lit(1))
            _y3 = (_bt.select("cust", "m").join(_scr.where(F.col("on") == 1), "cust")
                   .where(F.col("e_c").between(F.col("m") + 1, F.col("m") + 6)).select("cust", "m").distinct().withColumn("y_credit", F.lit(1)))
            _bt = (_bt.join(_hc, ["cust", "m"], "left").join(_y3, ["cust", "m"], "left").na.fill(0, ["y_credit", "has_credit"])
                   .withColumn("y_credit", F.when(F.col("has_credit") == 1, F.col("y_credit"))))
            _ylabs["y_credit"] = "past due or default switches on in next 6 months (customers with credit data)"
        _band = (F.when(F.col("exposure") <= 0, "0").when(F.col("exposure") < _tc, f"0-{_tc:.0%}").when(F.col("exposure") < _tb, f"{_tc:.0%}-{_tb:.0%}")
                 .when(F.col("exposure") < _ta, f"{_tb:.0%}-{_ta:.0%}").otherwise(f"{_ta:.0%}+"))
        _bt = _bt.withColumn("band", _band).withColumn("own", F.when(F.col("n_own") >= 2, "2+").otherwise(F.col("n_own").cast("string")))
        _agg = [F.count(F.lit(1)).alias("customer_months")] + [F.avg(k).alias(k) for k in _ylabs]
        bt_band = _bt.groupBy("band", "own").agg(*_agg).toPandas()
        bt_tier = _bt.groupBy("tier").agg(*_agg).toPandas()
        bt_month = _bt.groupBy("m").agg(F.avg((F.col("tier") != "-").cast("double")).alias("share_flagged"),
                                       *[F.avg(F.when(F.col("tier").isin("A", "B"), F.col(k))).alias(f"{k}_AB") for k in _ylabs],
                                       *[F.avg(F.when(F.col("tier") == "-", F.col(k))).alias(f"{k}_none") for k in _ylabs]).toPandas()
    _ref = bt_band[(bt_band.band == "0") & (bt_band.own == "0")]
    for k in _ylabs:
        r0 = float(_ref[k].iloc[0]) if len(_ref) else np.nan
        bt_band[f"{k}_lift"] = bt_band[k] / r0
        bt_tier[f"{k}_lift"] = bt_tier[k] / r0
    _bo = {"0": 0, f"0-{_tc:.0%}": 1, f"{_tc:.0%}-{_tb:.0%}": 2, f"{_tb:.0%}-{_ta:.0%}": 3, f"{_ta:.0%}+": 4}
    bt_band = bt_band.assign(_o=bt_band.band.map(_bo)).sort_values(["own", "_o"]).drop(columns="_o")
    bt_tier = bt_tier.sort_values("tier", key=lambda s_: s_.map({"A": 0, "B": 1, "C": 2, "-": 3}))
    bt_month["month"] = bt_month.m.map(mi2m); bt_month = bt_month.drop(columns="m").sort_values("month")
    for _n, _d in [("p1_score_backtest_band", bt_band), ("p1_score_backtest_tier", bt_tier), ("p1_score_backtest_month", bt_month)]:
        _d.to_csv(os.path.join(OUT_DIR, f"{_n}.csv"), index=False)

    # --- latest month: traits, predictions, lists -------------------------------------------------------
    SM_ = MAX_MI
    with step(f"latest month {mi2m(SM_)}: traits and predicted impact"):
        cur = score_all.where(F.col("m") == SM_).toPandas()
        cur["cust"] = cur.cust.astype(str)
        _ids = spark.createDataFrame(cur[["cust"]])
        _w6 = lambda col_: F.avg(col_)
        _mcx = set(spark.table(CONFIG["MET_TABLE"]).columns)
        _bc = [c for c in ["amt_all_in", "amt_all_out", "ncpty_all_in", "ext_share_out"] if c in _mcx]
        _bl = (spark.table(CONFIG["MET_TABLE"]).where(F.col("month").between(mi2m(SM_ - BM + 1), mi2m(SM_)))
               .select(F.col("mdm_id").cast("string").alias("cust"), mi_col("month").alias("mi"), *[F.col(c).cast("double").alias(c) for c in _bc])
               .join(_ids, "cust").groupBy("cust").agg(*[F.avg(c).alias(f"b_{c}") for c in _bc]).toPandas())
        cur = cur.merge(_bl, on="cust", how="left")
        _dep, _dk = fam_table("deposits", ["dep_bal_mean"])
        if _dep is not None:
            _db = (_dep.where(F.col("mi").between(SM_ - BM + 1, SM_)).groupBy(_dk).agg(F.avg("dep_bal_mean").alias("b_bal")).toPandas())
            cur = cur.merge(_db.rename(columns={_dk: "cust" if _dk == "cust" else _dk}), on=_dk if _dk != "cust" else "cust", how="left")
        _cr, _ck = fam_table("credit", ["dhe"])
        if _cr is not None:
            cur = cur.merge(_cr.where(F.col("mi") == SM_).select(_ck, F.col("dhe")).toPandas(), on=_ck, how="left")
        _rv, _rk = fam_table("revenue", ["tm_rev_12m"])
        if _rv is not None:
            cur = cur.merge(_rv.where(F.col("mi") == SM_).select(_rk, F.col("tm_rev_12m")).toPandas(), on=_rk, how="left")
        if selfpay is not None:
            _sp = (selfpay.where(F.col("mi").between(SM_ - BM + 1, SM_)).join(_ids, "cust")
                   .groupBy("cust").agg(F.avg("selfpay_amt_in").alias("b_self_in"), F.avg("selfpay_amt_out").alias("b_self_out")).toPandas())
            cur = cur.merge(_sp, on="cust", how="left")
        _reg = (standing.where(F.col("m") == SM_).groupBy("cust").agg(F.sum("expected").alias("reg_in")).toPandas())
        _down = (standing.where((F.col("m") == SM_) & (F.col("scope") == "int"))
                 .groupBy(F.col("other").alias("cust")).agg(F.sum("expected").alias("reg_int_out"),
                                                            F.countDistinct("cust").alias("pnc_payees")).toPandas())
        cur = cur.merge(_reg, on="cust", how="left").merge(_down, on="cust", how="left")
        cur[["reg_int_out", "pnc_payees"]] = cur[["reg_int_out", "pnc_payees"]].fillna(0)
        _pw = (payer_month.where((F.col("e") == SM_) & (F.col("pw_loss") == 1))
               .select(F.concat(F.col("scope"), F.lit(":"), F.col("other")).alias("top_payer")).toPandas())
        cur["top_payer_wide"] = cur.top_payer.isin(set(_pw.top_payer)).astype(float)

        # traits on the same scale as §5e (rank against the loss events the model was fitted on)
        def _rank(k, v):
            m_ = REG_MODELS.get(("loss", "amt_all_out"))
            ref = m_["ref"].get(k) if m_ else None
            if ref is None or len(ref) == 0:
                return pd.Series(0.5, index=v.index)
            r = np.searchsorted(ref, v.to_numpy(float), side="right") / len(ref)
            return pd.Series(np.where(np.isfinite(v.to_numpy(float)), r, 0.5), index=v.index)
        _bi = cur.b_amt_all_in if "b_amt_all_in" in cur else cur.base_in
        TR = pd.DataFrame(index=cur.index)
        TR["size"] = cur.exposure.clip(upper=0.95)
        TR["main_share"] = (cur.top_exp / cur.base_in).clip(upper=2)
        TR["gone_share"] = np.nan
        TR["reg_share"] = (cur.reg_in / cur.base_in).clip(upper=1.5)
        TR["n_payers"] = np.log1p(cur.b_ncpty_all_in) if "b_ncpty_all_in" in cur else np.nan
        TR["buffer"] = (cur.b_bal / cur.b_amt_all_out.where(cur.b_amt_all_out > 0)) if "b_bal" in cur else np.nan
        TR["self_in"] = (cur.b_self_in / _bi.where(_bi > 0)) if "b_self_in" in cur else np.nan
        TR["self_out"] = (cur.b_self_out / cur.b_amt_all_out.where(cur.b_amt_all_out > 0)) if "b_self_in" in cur else np.nan
        TR["payer_wide"] = cur.top_payer_wide
        TR["pnc_payer"] = cur.top_payer.fillna("").str.startswith("int:").astype(float)

        def _predict(col):
            m_ = REG_MODELS.get(("loss", col))
            if m_ is None:
                return pd.Series(np.nan, index=cur.index)
            slope = np.full(len(cur), m_["beta"][0])
            for i, k in enumerate(m_["ks"]):
                z = TR[k] if k in ("payer_wide", "pnc_payer") else _rank(k, TR[k])
                slope = slope + m_["beta"][1 + i] * (z.to_numpy(float) - m_["zm"][k])
            return pd.Series(slope, index=cur.index)
        _shock = np.log(1 - cur.exposure.clip(upper=0.95))
        _s_out, _s_reg, _s_bal = _predict("amt_all_out"), _predict("amt_reg_out"), _predict("dep_bal_mean")
        cur["pass_on_per_10pct"] = 100 * (np.exp(_s_out * math.log(0.9)) - 1)
        cur["income_at_risk"] = cur.at_risk
        # predicted monthly change if the exposed income is lost (negative = drop)
        cur["own_payments_change"] = (cur.b_amt_all_out if "b_amt_all_out" in cur else np.nan) * (np.exp(_s_out * _shock) - 1)
        cur["pnc_payees_change"] = cur.reg_int_out * (np.exp(_s_reg * _shock) - 1)
        cur["deposits_change"] = (cur.b_bal * (np.exp(_s_bal * _shock) - 1)) if "b_bal" in cur else np.nan
        cur.loc[cur.exposure <= 0, ["own_payments_change", "pnc_payees_change", "deposits_change"]] = 0.0

        # payer names (internal from the population, external from the counterparty tables)
        _pn = {}
        _tp = cur.top_payer.dropna().unique().tolist()
        _ints = [p_[4:] for p_ in _tp if p_.startswith("int:")]
        if _ints:
            _pn.update({f"int:{r.cust}": r.customer_name for r in pop.where(F.col("cust").isin(_ints))
                        .select("cust", "customer_name").dropDuplicates(["cust"]).toPandas().itertuples()})
        _exts = [p_[4:] for p_ in _tp if p_.startswith("ext:")]
        if _exts:
            try:
                _nm = (spark.table(CONFIG["CPTY_MAP_TABLE"]).select(F.col("cpty_id").cast("string").alias("cpty_id"),
                                                                    F.col("ent_l2").cast("string").alias("ent"))
                       .where(F.col("ent").isin(_exts))
                       .join(spark.table(CONFIG["CPTY_DIM_TABLE"]).select(F.col("cpty_id").cast("string").alias("cpty_id"), "cpty_name"), "cpty_id")
                       .groupBy("ent").agg(F.max("cpty_name").alias("nm")).toPandas())
                _pn.update({f"ext:{r.ent}": r.nm for r in _nm.itertuples()})
            except Exception:
                pass
        cur["top_stressed_payer"] = cur.top_payer.map(lambda p_: _pn.get(p_, p_) if isinstance(p_, str) else "")
        _osig = {"own_s_n": "fewer payments", "own_s_brk": "payees unpaid", "own_s_neg": "negative balance", "own_s_pdue": "past due"}
        cur["own_signals"] = cur.apply(lambda r: ", ".join(v for k, v in _osig.items() if k in cur and r.get(k, 0) == 1), axis=1)

    _keep = ["tier", "customer_name", "customer_pwr_id", "naics2n", "segment", "size_band", "base_in", "exposure", "n_stressed",
             "top_stressed_payer", "own_signals", "pass_on_per_10pct", "income_at_risk", "own_payments_change", "pnc_payees_change",
             "pnc_payees", "deposits_change", "b_bal", "dhe", "tm_rev_12m", "b_self_in", "b_self_out", "cust"]
    score_cur = cur[[c for c in _keep if c in cur]].copy()
    score_cur.to_parquet(os.path.join(OUT_DIR, f"p1_score_{mi2m(SM_)}.parquet"), index=False)
    watch = (score_cur[(score_cur.tier != "-") & (score_cur.segment == "operating")]
             .sort_values(["tier", "income_at_risk"], ascending=[True, False]))
    watch.to_csv(os.path.join(OUT_DIR, f"p1_watchlist_{mi2m(SM_)}.csv"), index=False)

    with step("payers under stress now and growth opportunities"):
        _sp_now = (stress.where(F.col("m") == SM_)
                   .join(standing.where(F.col("m") == SM_).select("scope", "other", "cust", "expected"), ["scope", "other"], "inner")
                   .groupBy("scope", "other").agg(F.countDistinct("cust").alias("pnc_payees_still_paid"),
                                                  F.sum("expected").alias("monthly_dollars_to_them"),
                                                  F.max("st_breaks").alias("breaking_payees"), F.max("st_own").alias("own_signals"),
                                                  F.max("broke_share").alias("broke_share_last_month"))
                   .orderBy(F.desc("monthly_dollars_to_them")).limit(200).toPandas())
        _sp_now["payer"] = (_sp_now.scope + ":" + _sp_now.other)
        _miss = [p_ for p_ in _sp_now.payer if p_ not in _pn]
        if _miss:
            _ii = [p_[4:] for p_ in _miss if p_.startswith("int:")]
            if _ii:
                _pn.update({f"int:{r.cust}": r.customer_name for r in pop.where(F.col("cust").isin(_ii))
                            .select("cust", "customer_name").dropDuplicates(["cust"]).toPandas().itertuples()})
            _ee = [p_[4:] for p_ in _miss if p_.startswith("ext:")]
            if _ee:
                try:
                    _nm = (spark.table(CONFIG["CPTY_MAP_TABLE"]).select(F.col("cpty_id").cast("string").alias("cpty_id"),
                                                                        F.col("ent_l2").cast("string").alias("ent"))
                           .where(F.col("ent").isin(_ee))
                           .join(spark.table(CONFIG["CPTY_DIM_TABLE"]).select(F.col("cpty_id").cast("string").alias("cpty_id"), "cpty_name"), "cpty_id")
                           .groupBy("ent").agg(F.max("cpty_name").alias("nm")).toPandas())
                    _pn.update({f"ext:{r.ent}": r.nm for r in _nm.itertuples()})
                except Exception:
                    pass
        _sp_now["name"] = _sp_now.payer.map(lambda p_: _pn.get(p_, ""))
        _sp_now["kind"] = np.where(_sp_now.scope == "int", "PNC customer", "outside PNC")
        _sp_now = _sp_now[["name", "kind", "pnc_payees_still_paid", "monthly_dollars_to_them", "breaking_payees", "own_signals",
                           "broke_share_last_month", "payer"]]
        _sp_now.to_csv(os.path.join(OUT_DIR, f"p1_stressed_payers_{mi2m(SM_)}.csv"), index=False)

        _g_out = REG_MODELS.get(("gain", "amt_all_out"))
        _gr = (shocks.where((F.col("direction") == "gain") & (F.col("e") >= SM_ - 2) & (F.col("segment") == "operating")
                            & (F.col("hub") == "non-hub payer") & (F.col("size") >= CONFIG["HEADLINE_MIN"]))
               .select("cust", "e", "customer_name", "customer_pwr_id", "naics2n", "size_band", "base_in", "size", "type").toPandas())
        if len(_gr):
            _gx = (spark.table(CONFIG["MET_TABLE"]).where(F.col("month").between(mi2m(SM_ - BM), mi2m(SM_)))
                   .select(F.col("mdm_id").cast("string").alias("cust"),
                           *[F.col(c).cast("double").alias(c) for c in ["amt_all_out", "ext_share_out"] if c in _mcx])
                   .join(spark.createDataFrame(_gr[["cust"]].astype(str)), "cust").groupBy("cust")
                   .agg(*[F.avg(c).alias(c) for c in ["amt_all_out", "ext_share_out"] if c in _mcx]).toPandas())
            _gr = _gr.merge(_gx, on="cust", how="left")
            _sl = _g_out["beta"][0] if _g_out is not None else np.nan
            _gr["predicted_extra_payments"] = _gr.get("amt_all_out", np.nan) * (np.exp(_sl * np.log1p(_gr["size"])) - 1)
            _gr["of_which_outside_pnc"] = _gr["predicted_extra_payments"] * _gr.get("ext_share_out", np.nan)
            if selfpay is not None:
                _gs = (selfpay.where(F.col("mi").between(SM_ - 2, SM_)).join(spark.createDataFrame(_gr[["cust"]].astype(str)), "cust")
                       .groupBy("cust").agg(F.sum("selfpay_amt_in").alias("money_in_from_own_accounts_3m")).toPandas())
                _gr = _gr.merge(_gs, on="cust", how="left")
            _gr["month"] = _gr.e.map(mi2m)
            _gr = (_gr.sort_values("e").drop_duplicates("cust", keep="last").drop(columns=["e"])
                   .sort_values("of_which_outside_pnc", ascending=False))
            _gr.to_csv(os.path.join(OUT_DIR, f"p1_growth_{mi2m(SM_)}.csv"), index=False)
        growth = _gr

    _pct = lambda x, cols: x.assign(**{c: x[c].map(lambda v: "" if pd.isna(v) else f"{v:.1%}") for c in cols if c in x})
    _ycols = list(_ylabs)
    _tc_now = (score_cur[score_cur.segment == "operating"].groupby("tier")
               .agg(customers=("cust", "count"), income_at_risk=("income_at_risk", "sum"), deposits_change=("deposits_change", "sum"),
                    pnc_payees_change=("pnc_payees_change", "sum"), balances=("b_bal", "sum") if "b_bal" in score_cur else ("cust", "count"),
                    dhe=("dhe", "sum") if "dhe" in score_cur else ("cust", "count"))
               .reset_index().round(0))
    def _tidy(x):
        x = x.copy()
        for c in x.columns:
            if c in ("exposure", "size", "ext_share_out"):
                x[c] = x[c].map(lambda v: "" if pd.isna(v) else f"{v:.0%}")
            elif c == "pass_on_per_10pct":
                x[c] = x[c].map(lambda v: "" if pd.isna(v) else f"{v:+.1f}%")
            elif x[c].dtype.kind == "f":
                x[c] = x[c].round(0)
        return x
    _wv = _tidy(watch.head(CONFIG["WATCH_TOP"]).drop(columns=["cust"]))
    CELL.show(("What the outcomes mean", pd.DataFrame([dict(column=k, meaning=v) for k, v in _ylabs.items()])),
              ("Backtest by exposure band and own-signal count (rates, and lift against band 0 with no signal)",
               _pct(bt_band, _ycols)),
              ("Backtest by tier", _pct(bt_tier, _ycols)),
              ("Backtest by month: tiers A and B against unflagged customers", _pct(bt_month, [c for c in bt_month if c != "month"])),
              (f"Tiers in {mi2m(SM_)} (operating customers; monthly dollars, balances and DHE summed)", _tc_now),
              (f"Watchlist {mi2m(SM_)}: top {CONFIG['WATCH_TOP']} (full list in p1_watchlist_{mi2m(SM_)}.csv)", _wv),
              (f"Payers under stress in {mi2m(SM_)} with the most PNC money depending on them",
               _tidy(_sp_now.head(CONFIG["WATCH_TOP"]).assign(broke_share_last_month=lambda x: x.broke_share_last_month.map(
                   lambda v: "" if pd.isna(v) else f"{v:.0%}")))),
              (f"Growth opportunities: latest gain event {mi2m(SM_ - 2)} .. {mi2m(SM_)}, one row per customer",
               _tidy(growth.head(CONFIG["WATCH_TOP"])) if len(growth) else growth))
    r = bt_tier.set_index("tier")
    for k, lab in _ylabs.items():
        if "A" in r.index and "-" in r.index:
            ans(f"Score backtest, {lab}: tier A / B / C / none",
                " / ".join(f"{r.loc[t, k]:.1%}" if t in r.index else "n/a" for t in ["A", "B", "C", "-"]))
    ans(f"Watchlist {mi2m(SM_)}: operating customers in tiers A / B / C",
        " / ".join(str(int((watch.tier == t).sum())) for t in ["A", "B", "C"]))

## §8 · Answer sheet, gates and files

One output to screenshot at the end of a run.

In [ ]:
CELL.start("§8 · Answer sheet, gates and files")
pd.DataFrame(FIG_INDEX).to_csv(os.path.join(FIG_DIR, "figures_index.csv"), index=False)
gates_frame().to_csv(os.path.join(OUT_DIR, "p1_gates.csv"), index=False)
pd.DataFrame(ANSWERS).to_csv(os.path.join(OUT_DIR, "p1_answers.csv"), index=False)
CELL.show(("Answers", pd.DataFrame(ANSWERS)), ("Gates", gates_frame()),
          (f"Figures saved in {FIG_DIR}", pd.DataFrame(FIG_INDEX, columns=["file", "section", "caption"])),
          ("Timings (minutes)", pd.DataFrame(TIMINGS, columns=["step", "minutes"])))

## §9 · HTML report

A standalone file (`p1_report.html` in the output folder, figures embedded) built from this run's tables and
figures, to review and share. Numbers in the text are filled in from the run, not typed. Case panels name customers,
so they are left out unless `REPORT_CASES = True`. Each part is built on its own: if one fails, the report says so
and keeps the rest.

In [ ]:
CELL.start("§9 · HTML report")
import base64, html as _h


def _g(name):
    v = globals().get(name)
    return v if isinstance(v, pd.DataFrame) and len(v) else None


def _eff(d, g, col, t=None):
    E_ = _g("effects")
    if E_ is None:
        return None
    x = E_[(E_.direction == d) & (E_.group == g) & (E_.outcome == col) & (E_.t == (ET if t is None else t))]
    return x.iloc[0] if len(x) else None


def _fv(v, kind, lab=""):
    if v is None or not np.isfinite(v):
        return "n/a"
    if kind == "logpct":
        return f"{v:+.0f}%"
    if "(pp" in lab or "pp of" in lab:
        return f"{v:+.1f} pp"
    return f"{v:+.4f}" if abs(v) < 0.01 else (f"{v:+.3f}" if abs(v) < 0.1 else f"{v:+.2f}")


def _fe(d, g, col, t=None, ci=True):
    r = _eff(d, g, col, t)
    if r is None:
        return "n/a"
    s_ = _fv(r.effect, r.kind, r.label)
    if ci:
        s_ += f" ({_fv(r.lo, r.kind, r.label).lstrip('+')} to {_fv(r.hi, r.kind, r.label).lstrip('+')})".replace("% to", " to")
    return s_


def _img(name, caption=""):
    pth = os.path.join(FIG_DIR, name)
    if not os.path.exists(pth):
        return ""
    b64 = base64.b64encode(open(pth, "rb").read()).decode()
    return f'<figure><img src="data:image/png;base64,{b64}"/><figcaption>{_h.escape(caption)}</figcaption></figure>'


def _tbl(df, n=40):
    if df is None or not len(df):
        return "<p class='note'>Not available in this run.</p>"
    d = df.head(n).copy()
    for c in d.columns:
        if d[c].dtype.kind == "f":
            d[c] = d[c].map(lambda v: "" if pd.isna(v) else (f"{v:,.0f}" if abs(v) >= 1000 else f"{v:,.2f}"))
    return d.to_html(index=False, escape=True, classes="t", border=0)


def _ul(items):
    return "<ul>" + "".join(f"<li>{x}</li>" for x in items if x) + "</ul>"


_parts, _fail = [], []


def _section(title, fn):
    try:
        body = fn()
        if body:
            _parts.append(f"<section><h2>{_h.escape(title)}</h2>{body}</section>")
    except Exception as ex:
        _fail.append(dict(section=title, error=f"{type(ex).__name__}: {str(ex)[:160]}"))
        _parts.append(f"<section><h2>{_h.escape(title)}</h2><p class='note'>This part could not be built in this run.</p></section>")


_A = pd.DataFrame(ANSWERS)
_ans = lambda q: (_A[_A.question == q].answer.iloc[-1] if len(_A) and (_A.question == q).any() else "n/a")
_nl = int(events[(events.direction == "loss") & events.headline].shape[0]) if "events" in globals() else 0
_ng = int(events[(events.direction == "gain") & events.headline].shape[0]) if "events" in globals() else 0
_ptx = _g("passthru")
_pt = lambda d: (f"{100 * _ptx[(_ptx.direction == d) & (_ptx.group == 'headline')].pass_through.iloc[0]:.0f}%"
                 if _ptx is not None and ((_ptx.direction == d) & (_ptx.group == "headline")).any() else "n/a")


def _order_text(d, grp):
    """first-move months in words, from §5b (moves before / at / after the event)"""
    fm = _g("fm_all")
    if fm is None:
        return "n/a."
    import re as _re
    x = fm[(fm.direction == d) & (fm.group == grp) & fm.reading.isin(["moves before the event", "moves"])]
    if x.empty:
        return "no outcome moves clearly."
    x = x.assign(o=x.outcome.map(lambda v: (lambda w: w if w[:2].isupper() else w[:1].lower() + w[1:])(
        _re.sub(r"\s*\([^)]*\)\s*$", "", v))))
    x = x[x.onset <= 2]
    out = []
    for t, g in x.groupby("onset"):
        if len(g) > 5:
            g = g.head(5).assign(o=list(g.o.head(4)) + [f"{g.o.iloc[4]} and {len(g) - 5} more"])
        when = (f"{int(-t)} month{'s' if t < -1 else ''} before" if t < 0 else ("in the event month" if t == 0
                else f"{int(t)} month{'s' if t > 1 else ''} after"))
        out.append(f"{when}: {', '.join(g.o)}")
    return "; ".join(out) + "."


def _s_summary():
    rb = _g("reg_base")
    _r10 = lambda d, c: (f"{rb[(rb.direction == d) & (rb.col == c)].per_10pct.iloc[0]:+.1f}%"
                         if rb is not None and ((rb.direction == d) & (rb.col == c)).any() else "n/a")
    b = [f"<b>After a loss of regular income.</b> When regular payers cut 20%+ of a business customer's income ({_nl:,} cases), "
         f"three months later, against matched customers: income {_fe('loss', 'headline', 'amt_all_in', ci=False)}, own payments "
         f"{_fe('loss', 'headline', 'amt_all_out', ci=False)}, average balance {_fe('loss', 'headline', 'dep_bal_mean', ci=False)}. "
         f"Share of the income change passed on as payments: {_pt('loss')}. Per 10% of income lost, own payments change "
         f"{_r10('loss', 'amt_all_out')}.",
         f"<b>Where the change lands.</b> After a loss: payments to regular payees {_fe('loss', 'headline', 'amt_reg_out', ci=False)}, "
         f"to other payees {_fe('loss', 'headline', 'amt_irr_out', ci=False)}. After a gain ({_ng:,} cases): regular payees "
         f"{_fe('gain', 'headline', 'amt_reg_out', ci=False)}, other payees {_fe('gain', 'headline', 'amt_irr_out', ci=False)}.",
         f"<b>One step further.</b> Extra share of the shocked customer's own regular payments to other PNC customers that break "
         f"in the next three months: {_ans('Second hop, loss: extra break rate of own streams to PNC customers, +1..+3')}.",
         f"<b>Credit.</b> PD change at +3: {_fe('loss', 'headline', 'pd_pct_now', ci=False)}; days past due "
         f"{_fe('loss', 'headline', 'past_due_days', ci=False)}. Strongest signals in the year before real defaults: "
         f"{_ans('Early warning, default: strongest signals before it')}.",
         f"<b>Absorbing.</b> Pass-through by balance buffer (low / middle / high third): "
         f"{_ans('loss: pass-through by balance buffer (low / middle / high third)')}. Traits that clearly change the response "
         f"to a 10% loss (change in own payments): {_ans('loss: traits that clearly change outflow response (per 10% shock, low → high end)')}.",
         f"<b>Timing (losses).</b> {_order_text('loss', 'headline')} When the payer cut many PNC customers at once "
         f"(outside the customer's control): {_order_text('loss', 'payer-wide, non-hub')}",
         f"<b>Self-payment.</b> TM organisations with money in (out) from own accounts at other banks: "
         f"{_ans('TM organisations with any self-payment in (out), last 6 months')}; net money in from own accounts after a "
         f"loss: {_ans('loss +3: net money in from own accounts elsewhere (pp of monthly inflow)')} pp of monthly income."]
    return "<p>Numbers are filled in from this run. Read the sections below for intervals and caveats.</p>" + _ul(b)


def _s_method():
    return _ul([
        "Population: Treasury Management business customers (organisations) with at least one regular incoming payer, "
        f"data {MONTH_MIN} to {MONTH_MAX}, events {mi2m(EV_START)} to {mi2m(EV_END)}.",
        "Regular payer: paid in at least 4 of the 6 and 8 of the 12 months before, steadily. A loss is a payer falling at "
        "least half short of its usual amount (counted over three months, so late payments do not count); a gain is a payer "
        "paying at least 50% more, or a new payer paying three months running. Losses and gains in the same month are netted, "
        "related companies and payers moving money between companies of one group are left out.",
        "An event is a net change of at least 10% of the customer's usual monthly income; the headline uses 20%+, operating "
        "businesses, and payers that are not processors or PNC entities.",
        "Each shocked customer is compared with 3 matched customers (same month, sector, income size, another group) over the "
        "same months; intervals come from a bootstrap that resamples groups of companies.",
        "Cross-checks: each payer counted once, intervals by payer, leaving out each month, and a second event definition on "
        "total income."])


def _s_effects():
    rows = []
    for col in [c for c in ["amt_all_in", "amt_all_out", "amt_reg_out", "amt_irr_out", "ntxn_all_out", "rec_broken_share_out",
                            "dep_bal_mean", "selfpay_net", "dhe", "pd_pct_now", "past_due_days", "tm_rev_12m"] if c in ALL_OUT]:
        rows.append({"outcome": ALL_OUT[col]["label"], "after a loss": _fe("loss", "headline", col),
                     "after a gain": _fe("gain", "headline", col)})
    return ("<p>Effect three months after the event, against matched customers (95% interval).</p>" + _tbl(pd.DataFrame(rows))
            + _img("f03_inflow_loss.png", "Income around a loss, shocked vs matched customers")
            + _img("f04_outflow_loss.png", "Own payments around a loss")
            + _img("f06_deposits_loss.png", "Deposits around a loss"))


def _s_order():
    fm = _g("fm_all")
    t = None
    if fm is not None:
        f_ = lambda v: "" if not np.isfinite(v) else f"{int(v):+d}"
        x = fm[(fm.direction == "loss") & fm.reading.isin(["moves before the event", "moves"])]
        t = (x.assign(cell=x.onset.map(f_)).pivot_table(index="outcome", columns="group", values="cell", aggfunc="first")
             .reindex(columns=[g for g in TIMING_GROUPS if g in set(x.group)]).reset_index())
        t = t.reindex(t[t.columns[1]].map(lambda v: int(v) if v not in ("", None) and pd.notna(v) else 99).sort_values().index)
    return ("<p>First month the effect is clearly different from zero (months from the event; negative = before). "
            "Headline, payer-wide stops from non-processors (outside the customer's control), single-relationship losses, "
            "and losses defined on total income.</p>" + _tbl(t)
            + _img("f18_what_moves_first.png", "Share of each outcome's peak reached, month by month"))


def _s_absorb():
    sc = _g("susc")
    rb = _g("reg_base"); rt = _g("reg_traits")
    parts = []
    if rt is not None:
        x = rt[(rt.direction == "loss") & (rt.col == "amt_all_out")]
        x = x.assign(**{"low end / no": x.low.map(lambda v: f"{v:+.1f}%"), "high end / yes": x.high.map(lambda v: f"{v:+.1f}%"),
                        "difference": [f"{r.diff:+.1f} ({r.diff_lo:+.1f} to {r.diff_hi:+.1f})" for r in x.itertuples()]})
        b = rb[(rb.direction == "loss") & (rb.col == "amt_all_out")] if rb is not None else None
        lead = (f"<p>A 10% loss of income cuts the customer's own payments by {b.per_10pct.iloc[0]:+.1f}% at +3 for an average "
                f"customer. How that changes with each trait (others at average):</p>") if b is not None and len(b) else ""
        parts.append(lead + _tbl(x[["trait", "low end / no", "high end / yes", "difference", "clear"]]))
        parts.append(_img("f24_traits_regression.png", "Outflow response to a 10% income shock at the low and high end of each trait"))
    if sc is not None:
        y = sc[(sc.direction == "loss")][["cut", "group", "events"] + [c for c in ["inflow_pct", "pass_through", "balance_pct",
                                                                                 "payees_unpaid_pp"] if c in sc]].copy()
        if "pass_through" in y:
            y["pass_through"] = y.pass_through.map(lambda v: "" if pd.isna(v) else f"{100 * v:.0f}%")
        parts.append("<p>Headline losses by sector and trait (thirds), effect at +3.</p>" + _tbl(y, 80))
    return "".join(parts)


def _s_hop():
    return (_tbl(_g("_h2")) + "<p>By event group (extra pp, +1..+3 against -3..-1, event minus controls):</p>" + _tbl(_g("_h2g"))
            + _img("f14_second_hop.png", "Share of the shocked customer's own regular payments to other PNC "
                                                         "customers that break (losses) or grow (gains)"))


def _s_warning():
    e = _g("early"); ff = _g("first_fire")
    t = None
    if e is not None:
        t = (e[~e.defines_anchor & (e.lo_pp > 0)].sort_values(["anchor", "excess_pp"], ascending=[True, False]).groupby("anchor").head(4)
             [["anchor", "signal", "anchored_before", "controls_before", "excess_pp", "median_lead_months"]])
        t["anchored_before"] = t.anchored_before.map(lambda v: f"{v:.0%}"); t["controls_before"] = t.controls_before.map(lambda v: f"{v:.0%}")
    return ("<p>Real events (default, PD doubling, deposit collapse, a PNC customer cutting many of its payees) and the signals "
            "that switched on in the 12 months before, against matched customers without such an event.</p>" + _tbl(t)
            + "<p>Signal usually first to switch on:</p>"
            + _tbl(ff.sort_values("share_first", ascending=False).groupby("anchor").head(2) if ff is not None else None)
            + _img("f12_early_warning.png", "Extra switch-on rate of each signal by month before the event"))


def _s_self():
    pv = _g("_pvs"); sw = _g("_swt"); se = _g("_se_tab")
    if pv is None and sw is None:
        return ""
    return ("<p>Payments between the customer's own accounts at PNC and at other banks (matched by name).</p>" + _tbl(pv)
            + "<p>Around events (% for dollars, pp of monthly income for the net):</p>" + _tbl(se)
            + "<p>Headline effects at +3 with vs without self-payment in the 6 months before:</p>" + _tbl(sw))


def _s_scen():
    s_ = _g("sc_first")
    t = s_[["scenario", "group", "outcome", "adjusted", "units", "reading", "onset", "peak_month", "peak", "effect_at_1"]] if s_ is not None and "adjusted" in s_ else s_
    return (_tbl(t) + _img("f21_scenario_2.png", "2026 oil price shock: transport and transport-dependent customers vs other sectors")
            + _img("f20_scenario_1.png", "Change Healthcare: payees vs same-sector customers (descriptive)"))


def _s_robust():
    return ("<p>Pass-through with every payer-month counted once and intervals by payer:</p>"
            + _tbl(_g("_ptr").pivot_table(index=["direction", "group"], columns="variant", values="pass_through", aggfunc="first").reset_index()
                   if _g("_ptr") is not None else None)
            + "<p>Headline effect at +3 when each event month is left out:</p>" + _tbl(_g("_loo"))
            + "<p>How concentrated events are on a few payers:</p>" + _tbl(_g("_conc"))
            + _img("f23_effect_by_event_month.png", "Headline effects at +3 by event month"))


def _s_score():
    bb, bt = _g("bt_band"), _g("bt_tier")
    if bt is None:
        return ""
    yl = globals().get("_ylabs", {})
    pc = lambda x: x.assign(**{c: x[c].map(lambda v: "" if pd.isna(v) else (f"{v:.1%}" if not c.endswith("_lift") else f"{v:.1f}x"))
                               for c in x.columns if c.startswith("y_")})
    tn = _g("_tc_now")
    return (_ul(["For every operating TM organisation each month: <b>exposure</b> = share of its usual monthly income expected "
                 "from regular payers that are under stress now (a payer that broke at least 25% of its regular PNC payees last "
                 "month, or a PNC-customer payer whose own warning signals switched on in the last 6 months).",
                 "<b>Own signals</b>: fewer outgoing payments, regular payees unpaid, negative-balance days, past due, switched on "
                 "in the last 3 months.",
                 "<b>Tier</b> A: exposure 20%+; B: 10-20% or 2+ own signals; C: 5-10% or 1 own signal.",
                 "For flagged customers the §5e models give the predicted monthly change if the exposed income is lost: own "
                 "payments, payments to regular PNC payees, deposits; DHE and TM revenue for context. Customer lists stay inside the bank's "
                 "environment and are not part of this report."])
            + "<p>Backtest (" + "; ".join(f"{k}: {v}" for k, v in yl.items()) + "):</p>" + _tbl(pc(bt))
            + "<p>By exposure band and own-signal count:</p>" + _tbl(pc(bb), 60)
            + "<p>Latest month, operating customers by tier (monthly dollars and balances summed):</p>" + _tbl(tn))


def _s_caveats():
    gf = gates_frame()
    return (_ul(["Matched comparisons, not experiments: customers who lose a payer can differ in ways the matching misses. "
                 "Some outcomes move before the event, partly because customers see a relationship ending coming.",
                 "Payments cover what the payment table carries (about 86% of deposit-account dollars; debit card and part of "
                 "ACH, checks and RTP are missing).",
                 "Credit history in the TM tables is monthly; six months is short for defaults.",
                 "Self-payment is matched on names and misses accounts under other names."])
            + "<p>Checks run on this data:</p>" + _tbl(gf))


for _t, _fn in [("Summary", _s_summary), ("How it was measured", _s_method), ("What a payment shock does", _s_effects),
                ("What moves first", _s_order), ("Who absorbs and who passes it on", _s_absorb),
                ("Does it spread to the next customer?", _s_hop), ("Early warning before real events", _s_warning),
                ("Self-payment", _s_self), ("Real scenarios", _s_scen),
                ("Exposure score and watchlist (prototype)", _s_score), ("Robustness", _s_robust),
                ("Caveats and checks", _s_caveats)]:
    _section(_t, _fn)
if CONFIG["REPORT_CASES"]:
    _section("Case panels", lambda: "".join(_img(f["file"], f["caption"]) for f in FIG_INDEX if f["file"].startswith("f16_")))

_css = """
body{font-family:'IBM Plex Sans','Segoe UI',Arial,sans-serif;color:#0b0b0b;background:#fcfcfb;margin:0}
main{max-width:1080px;margin:0 auto;padding:32px 28px 64px}
h1{font-size:28px;margin:0 0 6px} .sub{color:#52514e;margin-bottom:24px}
h2{font-size:20px;border-top:1px solid #e6e5e0;padding-top:22px;margin-top:30px}
li{margin:6px 0;line-height:1.5} p{line-height:1.5}
table.t{border-collapse:collapse;font-size:12.5px;margin:10px 0 16px;width:100%}
table.t th{background:#f0efec;text-align:left;padding:6px 8px;font-weight:600}
table.t td{border-top:1px solid #e6e5e0;padding:5px 8px;font-variant-numeric:tabular-nums}
figure{margin:16px 0} figure img{max-width:100%;border:1px solid #e6e5e0}
figcaption{font-size:12px;color:#52514e;margin-top:4px} .note{color:#8a8985;font-style:italic}
"""
_html_doc = (f"<!DOCTYPE html><html><head><meta charset='utf-8'><title>PKG contagion: payment shocks</title><style>{_css}</style>"
             f"</head><body><main><h1>Payment shocks and how they spread</h1>"
             f"<div class='sub'>DSI · Payment Knowledge Graph (PKG) · data {MONTH_MIN} to {MONTH_MAX} · "
             f"built {time.strftime('%Y-%m-%d')} · configuration {SIG}-{SIG2}</div>" + "".join(_parts) + "</main></body></html>")
REPORT_PATH = os.path.join(OUT_DIR, CONFIG["REPORT_FILE"])
with open(REPORT_PATH, "w", encoding="utf-8") as _f:
    _f.write(_html_doc)
CELL.show(("Report", pd.DataFrame([dict(file=REPORT_PATH, size_kb=round(len(_html_doc) / 1024), sections=len(_parts),
                                        sections_failed=len(_fail))])),
          ("Sections that could not be built", pd.DataFrame(_fail) if _fail else pd.DataFrame({"section": ["none"]})))